# Colab experiment
Run from the top in Google Colab. Choose the required runtime first. If dependency installation asks for a runtime restart, restart and run from the top again. Set RESUME_RUN_ID only for an explicitly verified compatible checkpoint. Use RECOVER_RUN_ID to deliver staged results without rerunning compute. For a shared folder, add its shortcut in My Drive or set PROJECT_PATH_OVERRIDE to the actual mounted folder path. Put credentials in Colab Secrets; never paste them into this notebook.


In [ ]:
import json, sys, subprocess, importlib.metadata, time, uuid
from pathlib import Path
CONFIG = json.loads('{"budget":{"burn_rate_cu_per_hour":5.4,"estimated_hours":1.8,"max_hours":6.0},"confirmed":true,"context":{"confidence_floor":0.001,"dataset_id":"rdd2020_datasprint_2_0","dataset_revision":"2026-11-23","evaluation_config_hash":"coco101_iou50_95_bestf1_t4speed_skycrop22","evaluator_id":"official_score_py","evaluator_revision":"starter_kit_v1","image_resolution":640,"iou_threshold":0.5,"max_dets_per_image":100,"n_samples":2150,"split_hash":"train14691_val2150_test4200","split_id":"official_train_val_test"},"dependency_check":"required","drive_folder_id":"1n8ddoVvmlpff1ScGLi_WPw-pPBe_AyC1","drive_relative_path":"DataSprint_2.0_Road_Damage","experiment":{"accum_a100":2,"accum_l4":2,"accum_t4":3,"batch_a100":16,"batch_l4":12,"batch_t4":8,"bright_clahe_luma_thresh":155.0,"clahe_train_p":0.35,"conf_floor":0.001,"ema_decay":0.9998,"epochs":24,"epochs_t4":18,"eval_every_epochs":3,"fallback_repo":"ustc-community/dfine-small-coco","imgsz":640,"lowlight_luma_thresh":60.0,"lowlight_target_luma":115.0,"lr_backbone":2e-05,"lr_head":0.0002,"max_det":100,"max_grad_norm":0.1,"model_repo":"ustc-community/dfine-medium-coco","robustness_aug_p":0.3,"seed":42,"sky_crop_frac":0.22,"sky_crop_jitter_max":0.24,"sky_crop_jitter_min":0.18,"smoke_only":false,"test_flip_tta":true,"warmup_epochs":1.5,"wbf_iou_thr":0.55,"weight_decay":0.0001},"experiment_name":"datasprint2_dfine_m_skycrop22_clahe_oneof30_v1","gpu_required":true,"metric_context":{"ab_winner_delta_final_pts":{"higher_is_better":true,"split":"val_ab_test","unit":"score_delta_pts"},"accuracy_score_100":{"higher_is_better":true,"split":"val_clean","unit":"score_0_100"},"ap50_D00":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"ap50_D10":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"ap50_D20":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"ap50_D40":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"effective_batch_size":{"higher_is_better":true,"split":"train","unit":"count"},"estimated_compute_units":{"higher_is_better":false,"split":"runtime","unit":"CU"},"final_score_proxy_100":{"higher_is_better":true,"split":"val_combined_proxy","unit":"score_0_100"},"mAP50":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"mAP50_95":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"median_ms_t4":{"higher_is_better":false,"split":"val_clean","unit":"ms"},"peak_vram_gib":{"higher_is_better":false,"split":"runtime","unit":"GiB"},"precision":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"recall":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"robustness_mAP50_synth":{"higher_is_better":true,"split":"val_degraded_synth","unit":"ratio_0_1"},"speed_score_t4":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"train_images_count":{"higher_is_better":true,"split":"train","unit":"count"},"val_images_count":{"higher_is_better":true,"split":"val","unit":"count"}},"min_free_disk_gib":20,"min_gpu_memory_gib":14,"packages":["transformers==5.24.0","albumentations==2.0.8","safetensors==0.8.0","accelerate==1.16.0"],"project_id":"datasprint-2-0-road-damage","schema_version":"colab-builder-config/1","source_fingerprints":{"builder_sha256":"40efd2603d219b5ffce5bca7f5149ed20644fb0e49db68cc529b6b4f86a998a0","runtime_sha256":"e30c67b27342488b5304b3b938070e2fbe06c8a8cdcc3064cc645e732c2f906c","task_sha256":"1afe12fa4147e92cdae12fb951a00c6167f9d40b16dc0e8efd090b9ac32cd9aa"},"workload":"training"}')
RESUME_RUN_ID = ''  # Set an existing run ID only when resuming.
RECOVER_RUN_ID = ''  # Deliver saved results only; does not rerun the task.
PROJECT_PATH_OVERRIDE = ''  # Optional path inside /content/drive.
START_NEW_RUN = False  # Set True for a fresh independent attempt.
if not RECOVER_RUN_ID and CONFIG.get('python_min') and sys.version_info[:2] < tuple(map(int, CONFIG['python_min'].split('.'))):
    raise RuntimeError('Python version below project requirement; choose a compatible Colab runtime')
if not RECOVER_RUN_ID and CONFIG.get('python_max_exclusive') and sys.version_info[:2] >= tuple(map(int, CONFIG['python_max_exclusive'].split('.'))):
    raise RuntimeError('Python version outside project compatibility range')
if RESUME_RUN_ID and RECOVER_RUN_ID:
    raise ValueError('Select recovery or checkpoint resumption, not both')
if RESUME_RUN_ID or RECOVER_RUN_ID:
    RUN_ID = RESUME_RUN_ID or RECOVER_RUN_ID
elif START_NEW_RUN or 'RUN_ID' not in globals() or globals().get('RUN_PROJECT_ID') != CONFIG['project_id']:
    RUN_ID = str(uuid.uuid4())
RUN_PROJECT_ID = CONFIG['project_id']
print('Run ID:', RUN_ID)
import importlib.util
RUNTIME_SOURCE = '"""Standard-library logging/recovery. No network calls or ML dependencies.\n\nAuditing third-party requirement markers uses the installed packaging parser.\nStandard-library-only EDA and event recovery do not require that package.\n\nLocal locks/CAS do not provide distributed locking on a mounted Google Drive.\nSerialize remote writers and reconcile destination markers before retrying writes.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport tempfile\nimport uuid\nimport importlib.metadata\nimport subprocess\nimport sys\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nSCHEMA_VERSION = 2\n_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9_-]{0,95}\\Z")\n_SECRET = re.compile(r"(?:^|_)(?:api_key|access_token|refresh_token|hf_token|password|secret|authorization|cookie|credentials|private_key|secret_access_key|access_key_id)$")\nDEFAULT_CONTEXT_KEYS = (\n    "dataset_id", "dataset_revision", "split_id", "split_hash",\n    "evaluator_id", "evaluator_revision", "evaluation_config_hash", "n_samples",\n)\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat(timespec="microseconds")\n\n\ndef check_json(value, path="root"):\n    """Reject non-JSON objects, non-finite values, and credential-bearing keys."""\n    if value is None or isinstance(value, (str, bool, int)):\n        return\n    if isinstance(value, float):\n        if not math.isfinite(value):\n            raise ValueError(f"Non-finite number at {path}")\n        return\n    if isinstance(value, list):\n        for i, child in enumerate(value):\n            check_json(child, f"{path}[{i}]")\n        return\n    if isinstance(value, dict):\n        for key, child in value.items():\n            if not isinstance(key, str):\n                raise TypeError("JSON object keys must be strings")\n            snake = re.sub(r"([a-z0-9])([A-Z])", r"\\1_\\2", key)\n            normalized = re.sub(r"[^a-z0-9]+", "_", snake.lower()).strip("_")\n            if _SECRET.search(normalized) or normalized in {"token", "apikey"}:\n                raise ValueError("Credential-bearing field rejected; use Colab Secrets")\n            check_json(child, f"{path}.{key}")\n        return\n    raise TypeError(f"Unsupported JSON value at {path}; convert explicitly")\n\n\ndef canonical(value):\n    check_json(value)\n    return json.dumps(value, sort_keys=True, ensure_ascii=False,\n                      separators=(",", ":"), allow_nan=False).encode("utf-8")\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef file_digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef strict_loads(text):\n    # Reject NaN/Infinity rather than silently accepting JavaScript extensions.\n    def reject(_):\n        raise ValueError("Invalid JSON numeric constant")\n    def unique_pairs(pairs):\n        result = {}\n        for key, value in pairs:\n            if key in result:\n                raise ValueError("Duplicate JSON object key")\n            result[key] = value\n        return result\n    def finite_float(raw):\n        number = float(raw)\n        if not math.isfinite(number):\n            raise ValueError("Invalid JSON numeric constant")\n        return number\n    return json.loads(text, parse_constant=reject, object_pairs_hook=unique_pairs, parse_float=finite_float)\n\n\ndef read_json(path):\n    return strict_loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef component(value):\n    if not isinstance(value, str) or not _ID.fullmatch(value):\n        raise ValueError("Use a nonempty, safe ID (letters, digits, hyphen, underscore)")\n    return value\n\n\ndef within(root, relative):\n    root = Path(root).resolve()\n    rel = Path(relative)\n    if rel.is_absolute() or ".." in rel.parts:\n        raise ValueError("Expected a project-relative path")\n    target = (root / rel).resolve()\n    if target == root or root not in target.parents:\n        raise ValueError("Path escapes project or names its root")\n    return target\n\n\ndef resolve_project_root(mount_root, relative_path, project_id, *, override=\'\', folder_id=None):\n    """Resolve an explicit mounted path and verify identity before ANY write.\n\n    Shared-folder visibility is not inferred from connector access. An override\n    must remain inside the mount and point at the verified project registry.\n    """\n    mount = Path(mount_root).resolve()\n    if override:\n        target = Path(override)\n        if not target.is_absolute():\n            target = mount / target\n        target = target.resolve()\n        if mount not in target.parents:\n            raise ValueError(\'Project override must be inside the mounted Drive\')\n    else:\n        target = within(mount / \'MyDrive\', relative_path)\n    state_path = target / \'05_reports/autopilot/project_state.json\'\n    if not state_path.is_file():\n        raise FileNotFoundError(\'Project is not visible in mounted Drive. Add its shortcut to My Drive, \'\n                                \'or set PROJECT_PATH_OVERRIDE to its mounted path. Mount an account \'\n                                \'that can access the folder. No replacement project was created.\')\n    state = read_json(state_path)\n    if state.get(\'schema_version\') != SCHEMA_VERSION or state.get(\'project_id\') != project_id:\n        raise ValueError(\'Mounted project identity/schema mismatch\')\n    if folder_id and state.get(\'registry\', {}).get(\'root_folder\', {}).get(\'id\') != folder_id:\n        raise ValueError(\'Mounted Drive folder ID mismatch\')\n    return target\n\n\ndef dependency_report(pins, *, policy=\'required\', run=subprocess.run,\n                      requires=importlib.metadata.requires):\n    """Inspect pip check, blocking workload dependency conflicts.\n\n    Unrelated preinstalled conflicts are surfaced separately; unknown checker\n    failures still stop execution. No environment-wide repair or uninstall.\n    """\n    if policy not in {\'required\', \'all\'}:\n        raise ValueError(\'Unknown dependency policy\')\n    normalize = lambda name: re.sub(r\'[-_.]+\', \'-\', name.lower())\n    roots = {normalize(pin.split(\'==\', 1)[0]) for pin in pins}\n    required = set(roots)\n    requested_extras = {name: set() for name in roots}\n    processed = {}\n    queue = list(required)\n    while queue:\n        name = queue.pop()\n        active_extras = requested_extras[name]\n        if name in processed and processed[name] == active_extras:\n            continue\n        processed[name] = set(active_extras)\n        try:\n            dependencies = requires(name) or []\n        except importlib.metadata.PackageNotFoundError:\n            if name in roots:\n                raise RuntimeError(\'A required package is missing\') from None\n            continue  # Optional/conditional requirement; pip check judges runtime applicability.\n        for requirement in dependencies:\n            try:\n                from packaging.requirements import Requirement\n            except ImportError:\n                raise RuntimeError(\'Requirement metadata needs the packaging parser; include a compatible packaging pin\') from None\n            try:\n                parsed = Requirement(requirement)\n                if parsed.marker and not any(parsed.marker.evaluate({\'extra\': extra})\n                                             for extra in {\'\'} | active_extras):\n                    continue\n            except (ValueError, KeyError):\n                raise RuntimeError(\'Cannot interpret a required dependency marker\') from None\n            child = normalize(parsed.name)\n            extras = {normalize(extra) for extra in parsed.extras}\n            if child not in requested_extras or not extras <= requested_extras[child]:\n                requested_extras.setdefault(child, set()).update(extras)\n                required.add(child)\n                queue.append(child)\n    checked = run([sys.executable, \'-m\', \'pip\', \'check\'], capture_output=True, text=True)\n    if checked.returncode == 0:\n        return {\'status\': \'clean\', \'unrelated_conflicts\': 0, \'required_packages\': sorted(required)}\n    conflicts = []\n    for line in checked.stdout.splitlines():\n        match = re.match(r\'^([A-Za-z0-9_.-]+)\\s+\\S+\\s+(?:requires|has requirement)\\s+([A-Za-z0-9_.-]+)\', line)\n        if not match:\n            raise RuntimeError(\'Dependency checker failed or returned an unrecognized conflict\')\n        conflicts.append((normalize(match[1]), normalize(match[2])))\n    if not conflicts or checked.stderr.strip():\n        raise RuntimeError(\'Dependency checker did not produce a valid report\')\n    # Only requirements OF the workload closure constrain this run. An unrelated\n    # reverse dependent (for example an unused app constraining numpy) does not.\n    if policy == \'all\' or any(a in required for a, b in conflicts):\n        raise RuntimeError(\'Required dependency conflict; diagnose before running this workload\')\n    return {\'status\': \'unrelated_conflicts\', \'unrelated_conflicts\': len(conflicts),\n            \'required_packages\': sorted(required)}\n\n\n@contextmanager\ndef local_lock(path):\n    """Fail closed on contention. Never auto-delete another writer\'s lock."""\n    lock = Path(str(path) + ".lock")\n    lock.parent.mkdir(parents=True, exist_ok=True)\n    fd = os.open(str(lock), os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600)\n    try:\n        os.write(fd, str(os.getpid()).encode("ascii"))\n        os.close(fd)\n        fd = None\n        yield\n    finally:\n        if fd is not None:\n            os.close(fd)\n        lock.unlink()\n\n\ndef atomic_json(path, value, *, expected=None, absent=False):\n    """Atomic same-directory replace plus a local cooperative lock and SHA CAS."""\n    path = Path(path)\n    payload = canonical(value)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with local_lock(path):\n        if absent and path.exists():\n            raise FileExistsError(str(path))\n        if expected is not None and (not path.exists() or file_digest(path) != expected):\n            raise RuntimeError("State changed; reload and reconcile")\n        fd, temporary = tempfile.mkstemp(prefix=".pending-", dir=path.parent)\n        try:\n            with os.fdopen(fd, "wb") as out:\n                out.write(payload)\n                out.flush()\n                os.fsync(out.fileno())\n            os.replace(temporary, path)\n        finally:\n            if os.path.exists(temporary):\n                os.unlink(temporary)\n    # Readback checks local bytes only; it cannot prove Drive server durability.\n    if path.read_bytes() != payload:\n        raise IOError("JSON write readback mismatch")\n    return path\n\n\ndef _numeric(value):\n    return (not isinstance(value, bool) and\n            (isinstance(value, int) or isinstance(value, float) and math.isfinite(value)))\n\n\ndef validate_metrics(metrics):\n    if not isinstance(metrics, dict):\n        raise TypeError("metrics must be a JSON object")\n    for name, value in metrics.items():\n        if not isinstance(name, str) or not name or not _numeric(value):\n            raise ValueError("Metric names must be nonempty strings and values finite numbers")\n\n\nclass LogDeliveryError(OSError):\n    """The log is staged locally but its project destination was not verified."""\n    def __init__(self, spool_path, error_category):\n        self.spool_path = Path(spool_path)\n        self.error_category = error_category\n        super().__init__(f"Log delivery pending ({error_category}); recover from {self.spool_path}")\n\n\ndef _write_event(path, record):\n    try:\n        atomic_json(path, record, absent=True)\n    except FileExistsError:\n        if not path.is_file():\n            raise\n        prior = read_json(path)\n        body = {k: v for k, v in prior.items() if k not in {"timestamp_utc", "payload_digest"}}\n        expected = record["payload_digest"]\n        if prior.get("payload_digest") != expected or digest(body) != expected:\n            raise FileExistsError("Event ID already contains different data; choose a revision ID") from None\n    return path\n\n\ndef validate_context(context, metric_context):\n    if not isinstance(context,dict) or not isinstance(metric_context,dict):\n        raise ValueError(\'Evaluation context and metric_context must be objects\')\n    if any(not isinstance(key,str) or not key.strip() or not isinstance(value,dict)\n           for key,value in metric_context.items()):\n        raise ValueError(\'Each metric context must be a named object\')\n    check_json(context)\n    check_json(metric_context)\n\n\ndef validate_event(record, *, run_id=None, project_id=None):\n    """Validate a saved event before sync or recovery; never execute its source."""\n    check_json(record)\n    if not isinstance(record, dict) or record.get(\'schema_version\') != SCHEMA_VERSION:\n        raise ValueError(\'Unsupported event schema\')\n    component(record.get(\'run_id\'))\n    component(record.get(\'event_id\'))\n    if run_id and record[\'run_id\'] != run_id:\n        raise ValueError(\'Event run identity mismatch\')\n    if not isinstance(record.get(\'config\'), dict):\n        raise ValueError(\'Event configuration is missing\')\n    if project_id and record[\'config\'].get(\'project_id\') != project_id:\n        raise ValueError(\'Event project identity mismatch\')\n    if record.get(\'status\') not in {\'completed\', \'partial\', \'failed\', \'interrupted\'}:\n        raise ValueError(\'Invalid event status\')\n    validate_context(record.get(\'context\', {}), record.get(\'metric_context\', {}))\n    validate_metrics(record.get(\'metrics\'))\n    if record[\'status\'] == \'completed\' and not record[\'metrics\']:\n        raise ValueError(\'Completed event lacks metrics\')\n    body = {k: v for k, v in record.items() if k not in {\'timestamp_utc\', \'payload_digest\'}}\n    if digest(body) != record.get(\'payload_digest\'):\n        raise ValueError(\'Event payload checksum mismatch\')\n    return record\n\n\ndef copy_immutable_file(source, destination, expected_sha256, *, max_bytes=16 * 1024 * 1024):\n    """Recover bounded task artifacts; refuse oversized files and replacements."""\n    source, destination = Path(source), Path(destination)\n    if source.stat().st_size > max_bytes:\n        raise ValueError(\'Recovery artifact is too large or has a mismatched checksum\')\n    with source.open(\'rb\') as stream:\n        payload = stream.read(max_bytes + 1)\n    if len(payload) > max_bytes or hashlib.sha256(payload).hexdigest() != expected_sha256:\n        raise ValueError(\'Recovery artifact is too large or has a mismatched checksum\')\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with local_lock(destination):\n        if destination.exists():\n            if file_digest(destination) != expected_sha256:\n                raise FileExistsError(\'Recovery destination contains different content\')\n            return destination\n        fd, temporary = tempfile.mkstemp(prefix=\'.recovery-\', dir=destination.parent)\n        try:\n            with os.fdopen(fd, \'wb\') as stream:\n                stream.write(payload)\n                stream.flush()\n                os.fsync(stream.fileno())\n            os.replace(temporary, destination)\n        finally:\n            if os.path.exists(temporary):\n                os.unlink(temporary)\n    if file_digest(destination) != expected_sha256:\n        raise IOError(\'Recovery artifact readback mismatch\')\n    return destination\n\n\ndef recover_spooled_events(project_root, spool_root, run_id, *, project_id):\n    """Deliver original staged events and bounded artifacts WITHOUT recompute."""\n    run_id = component(run_id)\n    state = read_json(within(project_root, \'05_reports/autopilot/project_state.json\'))\n    if state.get(\'schema_version\') != SCHEMA_VERSION or state.get(\'project_id\') != project_id:\n        raise ValueError(\'Recovery project mismatch\')\n    events = within(spool_root, f\'03_results/runs/{run_id}/events\')\n    delivered = []\n    for source in sorted(events.glob(\'*.json\')):\n        source = within(spool_root, f\'03_results/runs/{run_id}/events/{source.name}\')\n        record = validate_event(read_json(source), run_id=run_id, project_id=project_id)\n        if source.stem != record[\'event_id\']:\n            raise ValueError(\'Event path identity mismatch\')\n        artifacts = record.get(\'artifacts\', {}).get(\'files\', {})\n        if not isinstance(artifacts, dict):\n            raise ValueError(\'Invalid artifact manifest\')\n        for relative, sha256 in artifacts.items():\n            copy_immutable_file(within(spool_root, relative), within(project_root, relative), sha256)\n        destination = within(project_root, f\'03_results/runs/{run_id}/events/{source.name}\')\n        _write_event(destination, record)\n        observed = validate_event(read_json(destination), run_id=run_id, project_id=project_id)\n        if observed[\'payload_digest\'] != record[\'payload_digest\']:\n            raise IOError(\'Recovered event readback mismatch\')\n        delivered.append({\'event_id\': record[\'event_id\'], \'status\': record[\'status\']})\n    return delivered\n\n\ndef log_experiment(project_root, exp_id, metrics, *, run_id=None, event_id="final",\n                   status="completed", model_name=None, config=None, context=None,\n                   metric_context=None, artifacts=None, notes="", errors=None,\n                   spool_root=None):\n    """Write an immutable run event. Retry with the SAME run_id/event_id.\n\n    A different payload at an existing ID raises; log corrections under a new\n    event_id. Returned files contain JSON primitives only. No secret values print.\n    """\n    run_id = component(run_id or str(uuid.uuid4()))\n    event_id = component(event_id)\n    if status not in {"completed", "partial", "failed", "interrupted"}:\n        raise ValueError("Unknown run status")\n    if config is not None and not isinstance(config,dict):\n        raise ValueError("Experiment config must be an object")\n    validate_context({} if context is None else context, {} if metric_context is None else metric_context)\n    validate_metrics(metrics)\n    if status == "completed" and not metrics:\n        raise ValueError("Completed runs need observed metrics")\n    if artifacts is not None and not isinstance(artifacts, dict):\n        raise ValueError(\'Artifacts must be an object\')\n    files = (artifacts or {}).get(\'files\', {})\n    if not isinstance(files, dict) or any(not isinstance(path, str) or not isinstance(sha, str)\n            or not re.fullmatch(\'[0-9a-f]{64}\', sha) for path, sha in files.items()):\n        raise ValueError(\'Artifact files need relative paths and SHA-256 strings\')\n    # Invalid local exports are task/export failures, not completed results to\n    # recover. Validate before staging an immutable terminal event. Delivery\n    # still copies/checks again to detect a subsequent source change.\n    if spool_root is not None:\n        for relative, expected_hash in files.items():\n            source = within(spool_root, relative)\n            if not source.is_file() or source.stat().st_size > 16 * 1024 * 1024 or file_digest(source) != expected_hash:\n                raise ValueError(\'Staged task artifact is missing, oversized, or has a mismatched checksum\')\n    body = {\n        "schema_version": SCHEMA_VERSION, "run_id": run_id, "event_id": event_id,\n        "exp_id": exp_id, "status": status, "model_name": model_name,\n        "metrics": metrics, "config": config or {}, "context": context or {},\n        "metric_context": metric_context or {}, "artifacts": artifacts or {},\n        "notes": notes, "errors": errors or [],\n    }\n    payload_digest = digest(body)\n    path = within(project_root, f"03_results/runs/{run_id}/events/{event_id}.json")\n    record = {**body, "timestamp_utc": utc_now(), "payload_digest": payload_digest}\n    # Reuse the immutable destination timestamp when the same event is restaged\n    # into a new spool. A mount/read failure must not prevent local staging.\n    try:\n        previous = validate_event(read_json(path), run_id=run_id)\n    except (OSError, ValueError, TypeError):\n        previous = None\n    if previous is not None and previous[\'payload_digest\'] == payload_digest:\n        record = previous\n    spool_path = None\n    if spool_root is not None:\n        if Path(spool_root).resolve() == Path(project_root).resolve():\n            raise ValueError("Use separate local spool storage")\n        spool_path = within(spool_root, f"03_results/runs/{run_id}/events/{event_id}.json")\n        _write_event(spool_path, record)\n        # Deliver the original staged bytes/UTC time after an uncertain retry.\n        record = read_json(spool_path)\n    try:\n        for relative, expected_hash in record[\'artifacts\'].get(\'files\', {}).items():\n            destination = within(project_root, relative)\n            if spool_root is not None:\n                copy_immutable_file(within(spool_root, relative), destination, expected_hash)\n            elif not destination.is_file() or file_digest(destination) != expected_hash:\n                raise ValueError(\'Task artifact could not be verified\')\n        return _write_event(path, record)\n    except (OSError, ValueError) as error:\n        if spool_path is not None:\n            raise LogDeliveryError(spool_path, type(error).__name__) from None\n        raise\n\n\ndef save_checkpoint_manifest(project_root, run_id, checkpoint_id, config, files):\n    """Call only AFTER model/optimizer/RNG artifacts have been fully written."""\n    run_id, checkpoint_id = component(run_id), component(checkpoint_id)\n    if not files or not isinstance(files, list):\n        raise ValueError("List at least one finalized checkpoint artifact")\n    hashes = {}\n    for relative in files:\n        target = within(project_root, relative)\n        if not target.is_file():\n            raise FileNotFoundError(str(target))\n        hashes[Path(relative).as_posix()] = file_digest(target)\n    payload = {"schema_version": SCHEMA_VERSION, "run_id": run_id,\n               "checkpoint_id": checkpoint_id, "config_digest": digest(config),\n               "files": hashes, "complete": True}\n    path = within(project_root, f"03_results/runs/{run_id}/checkpoints/{checkpoint_id}.json")\n    try:\n        return atomic_json(path, payload, absent=True)\n    except FileExistsError:\n        if read_json(path) != payload:\n            raise FileExistsError("Checkpoint manifest is immutable") from None\n        return path\n\n\ndef verify_checkpoint(project_root, manifest_path, config, *, run_id):\n    manifest = read_json(manifest_path)\n    if (manifest.get("schema_version") != SCHEMA_VERSION or manifest.get("complete") is not True\n            or manifest.get("run_id") != component(run_id)\n            or manifest.get("config_digest") != digest(config) or not manifest.get("files")):\n        raise ValueError("Checkpoint context/run mismatch or incomplete manifest")\n    for relative, expected in manifest["files"].items():\n        if file_digest(within(project_root, relative)) != expected:\n            raise ValueError("Checkpoint artifact checksum mismatch")\n    return manifest\n\n\ndef seed_everything(seed=42, *, deterministic=True):\n    """Lock Python and any already-loaded NumPy/PyTorch RNGs; return JSON receipt."""\n    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:\n        raise ValueError("seed must be a non-negative integer")\n    import random\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    seeded = ["python_random"]\n    np_mod = sys.modules.get("numpy")\n    if np_mod is not None and hasattr(np_mod, "random"):\n        np_mod.random.seed(seed)\n        seeded.append("numpy")\n    torch_mod = sys.modules.get("torch")\n    if torch_mod is not None and hasattr(torch_mod, "manual_seed"):\n        torch_mod.manual_seed(seed)\n        seeded.append("torch_cpu")\n        if hasattr(torch_mod, "cuda") and torch_mod.cuda.is_available():\n            torch_mod.cuda.manual_seed_all(seed)\n            seeded.append("torch_cuda")\n        if hasattr(torch_mod, "backends") and hasattr(torch_mod.backends, "cudnn"):\n            torch_mod.backends.cudnn.deterministic = bool(deterministic)\n            torch_mod.backends.cudnn.benchmark = not bool(deterministic)\n    return {"seed": seed, "deterministic": bool(deterministic), "seeded_modules": seeded}\n\n\ndef stage_to_local_cache(project_root, relative_path, cache_root="/content/autopilot_cache", *, expected_sha256=None):\n    """Stage a Drive file to fast local Colab disk with SHA-256 verification."""\n    source = within(project_root, relative_path)\n    if not source.is_file():\n        raise FileNotFoundError(str(source))\n    target = within(cache_root, relative_path)\n    source_size = source.stat().st_size\n    if target.is_file() and target.stat().st_size == source_size:\n        cached_sha = file_digest(target)\n        if expected_sha256 is None or cached_sha == expected_sha256:\n            if expected_sha256 is not None or cached_sha == file_digest(source):\n                return {"local_path": str(target), "sha256": cached_sha,\n                        "size_bytes": source_size, "cache_hit": True}\n    source_sha = file_digest(source)\n    if expected_sha256 is not None and source_sha != expected_sha256:\n        raise ValueError("Source file SHA-256 mismatch before local caching")\n    target.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=".cache-", dir=target.parent)\n    try:\n        with os.fdopen(fd, "wb") as out, source.open("rb") as inp:\n            for block in iter(lambda: inp.read(1024 * 1024), b""):\n                out.write(block)\n            out.flush()\n            os.fsync(out.fileno())\n        os.replace(temporary, target)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n    if file_digest(target) != source_sha:\n        raise IOError("Local cache readback checksum mismatch")\n    return {"local_path": str(target), "sha256": source_sha,\n            "size_bytes": source_size, "cache_hit": False}\n\n\ndef collect_colab_telemetry(elapsed_seconds, *, burn_rate_cu_per_hour=None):\n    """Collect JSON-safe Colab compute telemetry (elapsed, peak VRAM, estimated CUs)."""\n    if isinstance(elapsed_seconds, bool) or not _numeric(elapsed_seconds) or elapsed_seconds < 0:\n        raise ValueError("elapsed_seconds must be a non-negative finite number")\n    telemetry = {"elapsed_seconds": float(elapsed_seconds)}\n    if burn_rate_cu_per_hour is not None:\n        if isinstance(burn_rate_cu_per_hour, bool) or not _numeric(burn_rate_cu_per_hour) or burn_rate_cu_per_hour < 0:\n            raise ValueError("burn_rate_cu_per_hour must be a non-negative finite number")\n        telemetry["burn_rate_cu_per_hour"] = float(burn_rate_cu_per_hour)\n        telemetry["estimated_compute_units"] = round((float(elapsed_seconds) / 3600.0) * float(burn_rate_cu_per_hour), 4)\n    torch_mod = sys.modules.get("torch")\n    if torch_mod is not None and hasattr(torch_mod, "cuda") and torch_mod.cuda.is_available():\n        try:\n            telemetry["peak_vram_gib"] = round(float(torch_mod.cuda.max_memory_allocated()) / (1024 ** 3), 4)\n        except Exception:\n            pass\n    return telemetry\n\n\nPROJECT_MEMORY_SCHEMA = "ml-project-memory/1"\nPROJECT_MEMORY_RELPATH = "05_reports/autopilot/project_memory.json"\n_MEMORY_SECTIONS = (\n    "setup_context", "destinations", "active_lineage",\n    "experiment_memory", "publication_memory", "next_experiments_queue",\n)\n\nDOMAIN_CATALOG = {\n    "classical": {\n        "display_name": "Classical Tabular ML",\n        "terminology": {"model_unit": "Estimator / Pipeline", "data_unit": "Feature matrix (rows/cols)", "split_strategy": "Stratified / Group K-Fold", "tuning_knobs": "n_estimators, max_depth, learning_rate, regularization, class_weight, decision_threshold"},\n        "tracked_parameters": ["estimator", "n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda", "class_weight", "decision_threshold", "cv_folds"],\n        "candidate_metrics": ["roc_auc", "pr_auc", "f1_macro", "log_loss", "rmse", "mae", "r2", "brier_score"],\n        "extra_context_keys": ["cv_folds", "Positive_class_or_averaging", "decision_threshold"],\n        "leakage_and_split_controls": ["Fit imputers, scalers, and target encoders strictly on training folds", "Group split by entity_id when multiple rows share an entity"],\n        "sheets_views": ["Ablations & Hyperparameters", "EDA Findings"],\n        "docs_focus": "Feature engineering pipeline, out-of-fold validation vs held-out test, threshold calibration, and feature importance / SHAP ablations.",\n    },\n    "unsupervised": {\n        "display_name": "Unsupervised & Representation Learning",\n        "terminology": {"model_unit": "Clustering / Projection / Anomaly Detector", "data_unit": "Unlabeled feature vectors", "split_strategy": "Stability / Cohort Bootstrap", "tuning_knobs": "n_clusters, eps, min_samples, n_components, contamination, distance_metric"},\n        "tracked_parameters": ["algorithm", "n_clusters_or_components", "distance_metric", "contamination", "scaling_method"],\n        "candidate_metrics": ["silhouette_score", "davies_bouldin_index", "calinski_harabasz", "reconstruction_error", "cluster_stability_ari"],\n        "extra_context_keys": ["distance_metric", "feature_scaling"],\n        "leakage_and_split_controls": ["Do not fabricate supervised accuracy without ground-truth labels", "Evaluate stability across bootstrap resamples or temporal cohorts"],\n        "sheets_views": ["EDA Findings", "Ablations & Hyperparameters"],\n        "docs_focus": "Cluster/anomaly stability, representation quality, distance metric sensitivity, and domain sanity checks.",\n    },\n    "deep_learning": {\n        "display_name": "Deep Learning & Neural Architectures",\n        "terminology": {"model_unit": "Neural Backbone / Head", "data_unit": "Mini-batch tensors", "split_strategy": "Train / Val / Held-out Test", "tuning_knobs": "architecture, optimizer, lr_schedule, warmup_steps, batch_size, weight_decay, dropout, mixed_precision, grad_clip"},\n        "tracked_parameters": ["architecture", "optimizer", "learning_rate", "lr_scheduler", "warmup_ratio", "batch_size", "epochs_or_steps", "weight_decay", "dropout", "precision_dtype", "grad_accum_steps"],\n        "candidate_metrics": ["val_loss", "train_loss", "top1_accuracy", "f1_score", "throughput_samples_per_sec", "peak_vram_gib"],\n        "extra_context_keys": ["checkpoint_selection_metric", "precision_dtype"],\n        "leakage_and_split_controls": ["Select best checkpoint on validation split only, never held-out test", "Verify complete optimizer/scheduler/scaler/RNG state before claiming training resume"],\n        "sheets_views": ["Training Curves", "Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Convergence curves, train-val generalization gap, architecture & regularization ablations, and checkpoint resume verification.",\n    },\n    "computer_vision": {\n        "display_name": "Computer Vision (Classification, Detection, Segmentation)",\n        "terminology": {"model_unit": "Vision Backbone / Detector / Segmenter", "data_unit": "Images / Video frames / Bounding boxes / Masks", "split_strategy": "Scene / Patient / Camera Group Split", "tuning_knobs": "backbone, image_resolution, augmentation_policy, iou_threshold, conf_threshold, anchor_or_query_config, focal_loss_gamma"},\n        "tracked_parameters": ["backbone", "pretrained_weights", "image_size", "augmentation_policy", "iou_threshold", "conf_threshold", "nms_type", "batch_size", "learning_rate", "precision_dtype"],\n        "candidate_metrics": ["mAP_50", "mAP_50_95", "mean_iou", "dice_score", "top1_accuracy", "fps_throughput", "latency_p95_ms"],\n        "extra_context_keys": ["image_resolution", "iou_threshold", "eval_augmentation_tta"],\n        "leakage_and_split_controls": ["Group split by video/patient/scene/camera to prevent near-duplicate frame leakage", "Apply random augmentations (MixUp, CutMix, flips) to training split only", "Stage image archives to /content/autopilot_cache to avoid Google Drive FUSE I/O bottlenecks"],\n        "sheets_views": ["Ablations & Hyperparameters", "Training Curves", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Resolution & augmentation ablations, mAP/IoU across object sizes/classes, qualitative failure modes (occlusion, small objects), and FPS/latency tradeoff.",\n    },\n    "time_series": {\n        "display_name": "Time Series Forecasting & Sequential Modeling",\n        "terminology": {"model_unit": "Forecaster / Sequential Architecture", "data_unit": "Temporal series (entity x timestamp)", "split_strategy": "Rolling / Expanding Walk-Forward Window with Embargo", "tuning_knobs": "forecast_horizon, lookback_window, seasonality_periods, lag_features, exogenous_vars, scaling_per_series"},\n        "tracked_parameters": ["model_type", "forecast_horizon", "lookback_window", "frequency", "lag_features", "rolling_aggregations", "exogenous_features", "target_transform", "cv_windows"],\n        "candidate_metrics": ["mase", "smape", "wape", "rmse", "mae", "quantile_pinball_loss", "directional_accuracy"],\n        "extra_context_keys": ["forecast_horizon", "lookback_window", "temporal_cutoff_dates", "target_scale_space"],\n        "leakage_and_split_controls": ["Strict causal ordering: never use random K-Fold on time series", "Enforce embargo/gap between train end and validation forecast origin", "Compute rolling stats and scalers strictly using past window data (no future lookahead)", "Always compare against a Seasonal Naive / Last-Value baseline"],\n        "sheets_views": ["Ablations & Hyperparameters", "Dataset Splits", "EDA Findings"],\n        "docs_focus": "Walk-forward validation windows, Seasonal Naive baseline comparison, horizon decay analysis, lag/exogenous feature ablations, and regime-shift errors.",\n    },\n    "nlp": {\n        "display_name": "Natural Language Processing (Classification, NER, Sequence Tasks)",\n        "terminology": {"model_unit": "Language Encoder / Sequence Tagger", "data_unit": "Documents / Sentences / Token spans", "split_strategy": "Document / Author / Source-Deduplicated Split", "tuning_knobs": "pretrained_checkpoint, tokenizer, max_seq_length, pooling_strategy, learning_rate, warmup_ratio, label_smoothing"},\n        "tracked_parameters": ["pretrained_model", "tokenizer_revision", "max_seq_length", "learning_rate", "batch_size", "warmup_ratio", "weight_decay", "aggregation_level"],\n        "candidate_metrics": ["f1_macro", "f1_micro", "exact_match", "span_f1", "matthews_corrcoef", "roc_auc"],\n        "extra_context_keys": ["tokenizer_revision", "max_seq_length", "span_or_token_granularity"],\n        "leakage_and_split_controls": ["Deduplicate near-identical texts before splitting", "Keep all chunks/sentences from the same source document in the same split"],\n        "sheets_views": ["Ablations & Hyperparameters", "EDA Findings"],\n        "docs_focus": "Tokenizer & sequence-length sensitivity, entity/class slice performance, out-of-vocabulary / rare-class errors, and latency vs accuracy.",\n    },\n    "llm_genai": {\n        "display_name": "LLMs, Prompt Engineering & Generative AI",\n        "terminology": {"model_unit": "Foundation Model / Prompt Template / Decoding Policy", "data_unit": "Prompts / Completions / Multimodal generations", "split_strategy": "Decontaminated Eval Benchmark / Frozen Prompt Set", "tuning_knobs": "model_id, prompt_template_version, system_prompt_hash, temperature, top_p, max_new_tokens, few_shot_count, structured_schema"},\n        "tracked_parameters": ["model_id", "prompt_template_version", "few_shot_examples", "temperature", "top_p", "max_new_tokens", "judge_model_and_rubric", "response_format"],\n        "candidate_metrics": ["exact_match", "pass_at_k", "rubric_judge_score", "bleu_or_rouge_l", "bertscore", "format_validity_rate", "tokens_per_sec", "cost_per_1k_calls"],\n        "extra_context_keys": ["prompt_template_hash", "decoding_temperature", "judge_model_revision", "judge_rubric_hash"],\n        "leakage_and_split_controls": ["Never tune few-shot examples on the final test prompts", "Separate deterministic task accuracy from LLM-as-a-Judge scores and lock judge model/temperature=0", "Check format/JSON schema compliance separately from semantic quality"],\n        "sheets_views": ["Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Prompt template & few-shot ablations, deterministic vs judge-based metrics, hallucination/refusal taxonomy, and latency/token cost pareto frontier.",\n    },\n    "finetuning": {\n        "display_name": "LLM / Foundation Model Fine-Tuning (SFT, PEFT, LoRA/QLoRA, DPO)",\n        "terminology": {"model_unit": "Base Checkpoint + PEFT Adapter / Merged Weights", "data_unit": "Instruction-Response pairs / Preference triples", "split_strategy": "Prompt-Family Deduplicated Train / Val / Eval", "tuning_knobs": "base_model, peft_method, lora_r, lora_alpha, lora_dropout, target_modules, quant_bits, lr, effective_batch_size, max_seq_len, chat_template, dpo_beta"},\n        "tracked_parameters": ["base_model_revision", "finetuning_method", "lora_r", "lora_alpha", "lora_dropout", "target_modules", "quantization_bits", "learning_rate", "effective_batch_size", "max_seq_length", "chat_template", "epochs_or_steps", "dpo_beta"],\n        "candidate_metrics": ["val_perplexity_or_loss", "downstream_task_score", "instruction_following_rate", "win_rate_vs_base", "catastrophic_forgetting_delta", "peak_vram_gib", "training_tokens_per_sec"],\n        "extra_context_keys": ["base_model_revision", "chat_template_hash", "quantization_bits", "eval_decoding_config"],\n        "leakage_and_split_controls": ["Decontaminate instruction tuning data against downstream evaluation benchmarks (n-gram / embedding overlap check)", "Verify exact chat template & EOS token parity between training and inference", "Track base model zero-shot baseline AND general capability retention (catastrophic forgetting check)"],\n        "sheets_views": ["Training Curves", "Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Base vs LoRA/QLoRA adapter comparison, rank (r/alpha) & target-module ablations, training loss vs downstream eval curves, VRAM/Colab CU efficiency, and forgetting diagnostics.",\n    },\n    "rag": {\n        "display_name": "Retrieval-Augmented Generation (RAG, Vector/Hybrid Search, GraphRAG)",\n        "terminology": {"model_unit": "Ingestion + Retriever + Reranker + Generator Pipeline", "data_unit": "Corpus Documents -> Chunks -> Queries -> Grounded Answers", "split_strategy": "Frozen Corpus Snapshot + Disjoint Query/Document Eval Set", "tuning_knobs": "chunk_size, chunk_overlap, embedding_model, index_type, retrieval_mode, top_k_retrieve, reranker_model, top_k_rerank, generator_model, citation_prompt"},\n        "tracked_parameters": ["corpus_revision", "chunk_strategy_and_size", "chunk_overlap", "embedding_model", "hybrid_bm25_alpha", "top_k_retrieve", "reranker_model", "top_k_rerank", "generator_model", "prompt_template"],\n        "candidate_metrics": ["recall_at_k", "ndcg_at_k", "mrr_at_k", "context_precision", "faithfulness_groundedness", "answer_correctness", "citation_f1", "end_to_end_latency_ms"],\n        "extra_context_keys": ["corpus_revision", "index_revision", "top_k_cutoff", "stage_retrieval_vs_generation"],\n        "leakage_and_split_controls": ["Separate Retrieval metrics (Recall@k, MRR, nDCG) from Generation metrics (Faithfulness, Answer Correctness, Citation F1)", "Lock corpus snapshot & index revision across retriever comparisons", "Ensure evaluation queries were not synthesized directly from the top-1 chunk without multi-hop / paraphrase verification"],\n        "sheets_views": ["Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Two-stage evaluation tables (Retrieval stage vs Answer/Faithfulness stage), chunking & embedding ablations, reranker impact vs latency, and unanswerable/hallucination error slices.",\n    },\n    "agentic": {\n        "display_name": "Agentic AI, Multi-Agent Workflows & Tool-Use Systems",\n        "terminology": {"model_unit": "Agent Policy / Orchestration Graph / Toolset", "data_unit": "Task Episodes -> Multi-Step Trajectories -> Tool Calls", "split_strategy": "Sandboxed Task Suite with Deterministic Environment Reset", "tuning_knobs": "planner_model, worker_models, architecture_pattern, tool_schema_version, max_steps_budget, retry_policy, memory_strategy, guardrail_policy"},\n        "tracked_parameters": ["agent_architecture", "backbone_models", "tool_schema_version", "max_steps_budget", "max_retries", "memory_or_reflection_mode", "sandbox_environment_version", "temperature"],\n        "candidate_metrics": ["episode_success_rate", "pass_at_1", "tool_call_validity_rate", "avg_steps_to_completion", "policy_violation_count", "loop_or_timeout_rate", "cost_or_tokens_per_episode", "p95_episode_latency_sec"],\n        "extra_context_keys": ["sandbox_env_revision", "tool_schema_hash", "max_steps_budget", "granularity_episode_vs_step"],\n        "leakage_and_split_controls": ["Reset sandbox/environment state deterministically before every episode", "Never conflate step-level tool call syntax success with end-to-end episode task completion", "Redact credentials and sensitive payloads from saved trajectory traces"],\n        "sheets_views": ["Agent Traces", "Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "End-to-end episode success vs step-level tool reliability, planner/reflection ablations, trajectory failure taxonomy (infinite loops, wrong tool args, premature stop), and token/latency cost per solved task.",\n    },\n    "multimodal": {\n        "display_name": "Multimodal AI (Vision-Language, Audio-Text, Cross-Modal)",\n        "terminology": {"model_unit": "Multimodal Encoder-Decoder / VLM", "data_unit": "Aligned Image/Audio/Video + Text tuples", "split_strategy": "Entity / Media-Asset Disjoint Split", "tuning_knobs": "vision_encoder, language_backbone, projector_type, frame_sample_rate, image_tiles, fusion_strategy"},\n        "tracked_parameters": ["vision_or_audio_encoder", "language_model", "connector_or_projector", "input_resolution_or_fps", "max_tokens", "finetuning_recipe"],\n        "candidate_metrics": ["multimodal_task_accuracy", "vqa_score", "clip_score", "cross_modal_recall_at_k", "hallucination_rate", "latency_p95_ms"],\n        "extra_context_keys": ["modality_inputs", "media_resolution_or_sampling"],\n        "leakage_and_split_controls": ["Group split by underlying media asset so multiple captions/questions on the same image/video never cross train and test", "Verify modality alignment and missing-modality fallback behavior"],\n        "sheets_views": ["Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Modality contribution ablations (unimodal vs multimodal baseline), resolution/frame-sampling tradeoffs, and cross-modal hallucination analysis.",\n    },\n    "audio_speech": {\n        "display_name": "Audio & Speech Processing (ASR, TTS, Diarization, Audio Events)",\n        "terminology": {"model_unit": "Acoustic / Speech Model", "data_unit": "Waveforms / Log-Mel Spectrograms / Utterances", "split_strategy": "Speaker / Recording-Session Disjoint Split", "tuning_knobs": "sample_rate, n_mels, window_stride, backbone, language_model_decoder, spec_augment"},\n        "tracked_parameters": ["backbone", "sample_rate_hz", "feature_frontend", "spec_augment_config", "decoder_type", "chunk_duration_sec"],\n        "candidate_metrics": ["wer", "cer", "der_diarization", "pesq_or_mos", "f1_macro", "real_time_factor_rtf"],\n        "extra_context_keys": ["sample_rate_hz", "text_normalizer_version", "snr_noise_condition"],\n        "leakage_and_split_controls": ["Split strictly by speaker_id and recording session to prevent voice/channel leakage", "Apply identical text normalization before computing WER/CER"],\n        "sheets_views": ["Ablations & Hyperparameters", "Training Curves"],\n        "docs_focus": "Speaker-disjoint WER/CER results, noise/accent robustness slices, SpecAugment ablations, and Real-Time Factor (RTF) efficiency.",\n    },\n    "recommender_ranking": {\n        "display_name": "Recommender Systems & Learning-to-Rank",\n        "terminology": {"model_unit": "Candidate Retriever + Ranking Model", "data_unit": "User-Item Interactions / Query-Candidate Slates", "split_strategy": "Temporal Global Cutoff or Leave-Last-k-Out per User", "tuning_knobs": "embedding_dim, negative_sampling_ratio, loss_function, top_k_cutoff, cold_start_fallback"},\n        "tracked_parameters": ["model_architecture", "embedding_dim", "negative_sampling", "ranking_loss", "top_k_cutoff", "feature_crosses"],\n        "candidate_metrics": ["ndcg_at_k", "recall_at_k", "map_at_k", "hit_rate_at_k", "catalog_coverage", "inference_latency_ms"],\n        "extra_context_keys": ["top_k_cutoff", "negative_sampling_eval_pool", "cold_vs_warm_slice"],\n        "leakage_and_split_controls": ["Enforce global temporal cutoff so future interactions never leak into user/item history features", "Evaluate on full or consistent negative candidate pools, never biased random mini-samples without disclosure"],\n        "sheets_views": ["Ablations & Hyperparameters", "EDA Findings"],\n        "docs_focus": "Retrieval vs re-ranking stage metrics (@k), warm vs cold-start user/item slices,popularity-bias analysis, and serving latency.",\n    },\n    "custom": {\n        "display_name": "Custom / Hybrid / Scientific / Domain-Specific ML",\n        "terminology": {"model_unit": "Domain Model / Pipeline", "data_unit": "Domain observations", "split_strategy": "Domain-validated independent split", "tuning_knobs": "Task-defined experimental factors"},\n        "tracked_parameters": ["model_or_method", "experimental_factor", "key_hyperparameters", "preprocessing_version"],\n        "candidate_metrics": ["primary_objective_metric", "secondary_constraint_metric", "runtime_seconds"],\n        "extra_context_keys": ["evaluation_protocol_version"],\n        "leakage_and_split_controls": ["Establish domain unit of independence before splitting", "Separate exploratory diagnostics from ranked evaluation claims"],\n        "sheets_views": ["Ablations & Hyperparameters"],\n        "docs_focus": "Domain problem formulation, custom evaluation contract, baseline vs experimental variants, and reproducibility.",\n    },\n}\n\n\ndef resolve_domain_profile(families, *, custom_overrides=None):\n    """Synthesize a polymorphic domain profile by inheriting across 1+ ML domains.\n\n    Supports any single domain, multi-domain hybrid (e.g. [\'computer_vision\', \'finetuning\']\n    or [\'rag\', \'agentic\']), and project-specific custom_overrides for terminology,\n    tracked parameters, metrics, context keys, leakage rules, and Docs/Sheets style.\n    """\n    if not isinstance(families, list) or not families:\n        families = ["custom"]\n    cleaned = []\n    for fam in families:\n        if not isinstance(fam, str) or fam not in DOMAIN_CATALOG:\n            raise ValueError(f"Unsupported domain family: {fam!r}. Supported: {sorted(DOMAIN_CATALOG)}")\n        if fam not in cleaned:\n            cleaned.append(fam)\n\n    merged_terminology = {}\n    tracked_parameters = []\n    candidate_metrics = []\n    extra_context_keys = []\n    leakage_controls = []\n    sheets_views = []\n    docs_focus_parts = []\n    display_names = []\n\n    for fam in cleaned:\n        spec = DOMAIN_CATALOG[fam]\n        display_names.append(spec["display_name"])\n        for k, v in spec["terminology"].items():\n            merged_terminology[k] = f"{merged_terminology[k]} + {v}" if k in merged_terminology and v not in merged_terminology[k] else v\n        for item in spec["tracked_parameters"]:\n            if item not in tracked_parameters:\n                tracked_parameters.append(item)\n        for item in spec["candidate_metrics"]:\n            if item not in candidate_metrics:\n                candidate_metrics.append(item)\n        for item in spec["extra_context_keys"]:\n            if item not in extra_context_keys:\n                extra_context_keys.append(item)\n        for item in spec["leakage_and_split_controls"]:\n            if item not in leakage_controls:\n                leakage_controls.append(item)\n        for item in spec["sheets_views"]:\n            if item not in sheets_views:\n                sheets_views.append(item)\n        docs_focus_parts.append(f"[{spec[\'display_name\']}] {spec[\'docs_focus\']}")\n\n    profile = {\n        "inherited_families": cleaned,\n        "is_hybrid": len(cleaned) > 1,\n        "display_title": " × ".join(display_names),\n        "terminology": merged_terminology,\n        "tracked_parameters": tracked_parameters,\n        "candidate_metrics": candidate_metrics,\n        "extra_context_keys": extra_context_keys,\n        "leakage_and_split_controls": leakage_controls,\n        "sheets_views": sheets_views,\n        "docs_narrative_focus": " | ".join(docs_focus_parts),\n    }\n\n    if custom_overrides is not None:\n        if not isinstance(custom_overrides, dict):\n            raise ValueError("custom_overrides must be a JSON object")\n        check_json(custom_overrides)\n        if isinstance(custom_overrides.get("terminology"), dict):\n            profile["terminology"] = {**profile["terminology"], **custom_overrides["terminology"]}\n        for list_key in ("tracked_parameters", "candidate_metrics", "extra_context_keys",\n                         "leakage_and_split_controls", "sheets_views"):\n            if isinstance(custom_overrides.get(list_key), list):\n                for item in custom_overrides[list_key]:\n                    if isinstance(item, str) and item.strip() and item not in profile[list_key]:\n                        profile[list_key].append(item.strip())\n        if isinstance(custom_overrides.get("docs_narrative_focus"), str) and custom_overrides["docs_narrative_focus"].strip():\n            profile["docs_narrative_focus"] = custom_overrides["docs_narrative_focus"].strip()\n\n    check_json(profile)\n    return profile\n\n\n_DOMAIN_DETECTION_RULES = (\n    ("finetuning", {\n        "files": (r"lora", r"qlora", r"peft", r"sft", r"dpo", r"rlhf", r"adapter_config\\.json", r"instruction"),\n        "columns": ("instruction", "chosen", "rejected", "system_prompt", "conversations", "messages", "prompt_response"),\n        "imports": ("peft", "trl", "unsloth", "bitsandbytes", "axolotl"),\n        "keywords": (r"\\blora\\b", r"\\bqlora\\b", r"\\bpeft\\b", r"\\bfine[- ]?tun(?:e|ing)\\b", r"\\bsft\\b", r"\\bdpo\\b", r"\\brlhf\\b"),\n    }),\n    ("rag", {\n        "files": (r"corpus", r"chunks?", r"faiss", r"chroma", r"qdrant", r"bm25", r"retriever", r"rerank", r"knowledge_base"),\n        "columns": ("retrieved_chunks", "context_chunks", "ground_truth_answer", "chunk_id", "doc_id", "query_text"),\n        "imports": ("faiss", "chromadb", "qdrant_client", "llama_index", "ragas", "rank_bm25", "sentence_transformers"),\n        "keywords": (r"\\brag\\b", r"retrieval[- ]augmented", r"\\brecall@\\d+", r"\\bndcg@\\d+", r"\\bfaithfulness\\b", r"\\brerank(?:er|ing)\\b", r"\\bchunk(?:ing|_size)\\b"),\n    }),\n    ("agentic", {\n        "files": (r"agent", r"trajector(?:y|ies)", r"episodes?", r"tool_schema", r"sandbox", r"mcp"),\n        "columns": ("episode_id", "trajectory", "tool_calls", "observation", "step_budget", "agent_action"),\n        "imports": ("langgraph", "autogen", "crewai", "smolagents", "gymnasium"),\n        "keywords": (r"\\bagentic\\b", r"\\bmulti[- ]agent\\b", r"\\btool[- ]call(?:ing|s)?\\b", r"\\bepisode_success\\b", r"\\btrajectory\\b", r"\\breact\\s+(?:agent|loop|prompt)\\b"),\n    }),\n    ("computer_vision", {\n        "files": (r"\\.(?:jpg|jpeg|png|bmp|tif|tiff|webp|mp4|avi)\\Z", r"coco", r"yolo", r"bboxes", r"segmentation", r"masks?", r"images\\.zip"),\n        "columns": ("image_path", "image_id", "bbox", "xmin", "ymin", "xmax", "ymax", "rle_mask", "mask_path", "frame_id"),\n        "imports": ("cv2", "timm", "ultralytics", "torchvision", "albumentations", "kornia", "mmdet"),\n        "keywords": (r"\\bcomputer vision\\b", r"\\bobject detection\\b", r"\\bsegmentation\\b", r"\\bmap@\\d+|\\bmean average precision\\b", r"\\biou\\b", r"\\bdice score\\b", r"\\byolo\\b"),\n    }),\n    ("time_series", {\n        "files": (r"timeseries", r"forecast", r"horizon", r"ohlcv", r"seasonal", r"telemetry"),\n        "columns": ("timestamp", "datetime", "date", "ds", "forecast_horizon", "series_id", "lag_1", "lookback"),\n        "imports": ("statsforecast", "neuralforecast", "prophet", "pmdarima", "tsfresh", "sktime", "darts"),\n        "keywords": (r"\\btime[- ]series\\b", r"\\bforecast(?:ing)?\\b", r"\\bmase\\b", r"\\bsmape\\b", r"\\bwalk[- ]forward\\b", r"\\bhorizon\\b", r"\\bseasonality\\b"),\n    }),\n    ("audio_speech", {\n        "files": (r"\\.(?:wav|mp3|flac|ogg|m4a)\\Z", r"diarization", r"utterances?", r"spectrogram"),\n        "columns": ("audio_path", "speaker_id", "sample_rate", "duration_sec", "utterance_id", "waveform"),\n        "imports": ("librosa", "torchaudio", "soundfile", "whisper", "pyannote", "speechbrain"),\n        "keywords": (r"\\basr\\b", r"\\bspeech[- ]to[- ]text\\b", r"\\bdiarization\\b", r"\\bwer\\b", r"\\bcer\\b", r"\\blog[- ]mel\\b", r"\\bspecaugment\\b"),\n    }),\n    ("multimodal", {\n        "files": (r"multimodal", r"vqa", r"vlm", r"image_text", r"video_caption", r"cross_modal"),\n        "columns": ("image_caption", "visual_question", "audio_transcript", "modality"),\n        "imports": ("open_clip", "llava", "qwen_vl_utils"),\n        "keywords": (r"\\bmultimodal\\b", r"\\bvlm\\b", r"\\bvision[- ]language\\b", r"\\bvqa\\b", r"\\bclip[- ]?score\\b", r"\\bcross[- ]modal\\b"),\n    }),\n    ("recommender_ranking", {\n        "files": (r"interactions", r"user_item", r"clickstream", r"impressions", r"movielens", r"recsys"),\n        "columns": ("user_id", "item_id", "rating", "impression_id", "candidate_id", "slate_id"),\n        "imports": ("lightfm", "implicit", "recbole", "surprise"),\n        "keywords": (r"\\brecommender\\b", r"\\brecsys\\b", r"\\blearning[- ]to[- ]rank\\b", r"\\bcollaborative filtering\\b", r"\\bcold[- ]start\\b", r"\\bhit_rate@\\d+"),\n    }),\n    ("llm_genai", {\n        "files": (r"prompts?", r"few_shot", r"rubric", r"completions", r"generations"),\n        "columns": ("prompt", "completion", "response", "rubric_score", "judge_rationale"),\n        "imports": ("openai", "anthropic", "google.genai", "vllm", "litellm"),\n        "keywords": (r"\\bprompt engineering\\b", r"\\bfew[- ]shot\\b", r"\\bllm[- ]as[- ]a[- ]judge\\b", r"\\bpass@\\d+", r"\\bgenerative ai\\b"),\n    }),\n    ("nlp", {\n        "files": (r"conll", r"ner", r"sentiment", r"token_classification", r"text_classification"),\n        "columns": ("text", "sentence", "tokens", "ner_tags", "span_start", "span_end", "document_text"),\n        "imports": ("spacy", "nltk", "seqeval"),\n        "keywords": (r"\\bner\\b", r"\\bnamed entity\\b", r"\\btext classification\\b", r"\\bsentiment\\b", r"\\bspan[- ]f1\\b", r"\\btokenizer\\b"),\n    }),\n    ("deep_learning", {\n        "files": (r"\\.(?:pt|pth|ckpt|safetensors|onnx|h5)\\Z", r"backbone", r"checkpoints?"),\n        "columns": (),\n        "imports": ("torch", "pytorch_lightning", "lightning", "tensorflow", "keras", "jax", "flax"),\n        "keywords": (r"\\bdeep learning\\b", r"\\bneural network\\b", r"\\bbackprop\\b", r"\\badamw\\b", r"\\bcosine annealing\\b"),\n    }),\n    ("unsupervised", {\n        "files": (r"clustering", r"anomaly", r"outliers?", r"unsupervised", r"embeddings"),\n        "columns": ("cluster_id", "anomaly_score", "is_outlier"),\n        "imports": ("umap", "hdbscan"),\n        "keywords": (r"\\bunsupervised\\b", r"\\bclustering\\b", r"\\banomaly detection\\b", r"\\bsilhouette\\b", r"\\bdimensionality reduction\\b"),\n    }),\n    ("classical", {\n        "files": (r"\\.(?:csv|tsv|parquet|feather)\\Z", r"train\\.csv", r"test\\.csv", r"sample_submission\\.csv"),\n        "columns": ("target", "label", "class", "y", "prediction"),\n        "imports": ("sklearn", "scikit-learn", "xgboost", "lightgbm", "catboost", "optuna"),\n        "keywords": (r"\\btabular\\b", r"\\bxgboost\\b", r"\\blightgbm\\b", r"\\bcatboost\\b", r"\\bgradient boosting\\b", r"\\brandom forest\\b", r"\\broc[- ]auc\\b", r"\\bpr[- ]auc\\b"),\n    }),\n)\n\n\ndef auto_detect_domain(*, inventory=None, csv_headers=None, rules_text="", code_imports=None, custom_overrides=None):\n    """Auto-detect project ML domain(s), data schema signals, and inherited domain profile.\n\n    Inspects Drive file inventory names, CSV/table headers, rules/guidelines text,\n    and code imports without executing any untrusted files.\n    """\n    inventory = inventory or []\n    csv_headers = csv_headers or []\n    code_imports = code_imports or []\n    rules_text = rules_text or ""\n    if (not isinstance(inventory, list) or not isinstance(csv_headers, list)\n            or not isinstance(code_imports, list) or not isinstance(rules_text, str)):\n        raise ValueError("auto_detect_domain expects list inventory/csv_headers/code_imports and string rules_text")\n\n    file_names = []\n    for item in inventory:\n        if isinstance(item, dict) and isinstance(item.get("name"), str):\n            file_names.append(item["name"].strip().lower())\n        elif isinstance(item, str):\n            file_names.append(item.strip().lower())\n\n    norm_headers = [h.strip().lower() for h in csv_headers if isinstance(h, str) and h.strip()]\n    norm_imports = [imp.strip().lower() for imp in code_imports if isinstance(imp, str) and imp.strip()]\n    lower_rules = rules_text.lower()\n\n    evidence_by_family = {}\n    scores = {}\n\n    for family, matchers in _DOMAIN_DETECTION_RULES:\n        reasons = []\n        weight = 0\n        for pat in matchers["files"]:\n            matched_file = next((fn for fn in file_names if re.search(pat, fn, re.I)), None)\n            if matched_file:\n                reasons.append(f"file:{matched_file}")\n                weight += 2 if family != "classical" else 1\n        for col in matchers["columns"]:\n            if col in norm_headers:\n                reasons.append(f"column:{col}")\n                weight += 3\n        for imp in matchers["imports"]:\n            if any(imp == x or x.startswith(imp + ".") for x in norm_imports):\n                reasons.append(f"import:{imp}")\n                weight += 3\n        for kw in matchers["keywords"]:\n            if re.search(kw, lower_rules, re.I):\n                clean_kw = kw.replace(r"\\b", "").replace("\\\\", "")\n                reasons.append(f"rule:{clean_kw}")\n                weight += 3\n        if reasons:\n            evidence_by_family[family] = reasons\n            scores[family] = weight\n\n    # Schema signal extraction for leakage & split controls\n    temporal_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:timestamp|datetime|date|time|ds|year|month|week|day)(?:$|_)", h)]\n    group_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:user_id|patient_id|speaker_id|entity_id|group_id|session_id|customer_id|account_id|scene_id|video_id|doc_id)(?:$|_)", h)]\n    media_cols = [h for h in norm_headers if re.search(r"(?:image|audio|video|frame|mask|bbox|waveform)_(?:path|url|id)", h)]\n    text_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:text|prompt|instruction|query|question|sentence|document|transcript|completion|response)(?:$|_)", h)]\n    target_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:target|label|class|y|is_fraud|outcome|score|churn)(?:$|_)", h)]\n\n    # If user_id + item_id coexist in headers, strongly boost recommender_ranking\n    if "user_id" in norm_headers and any(c in norm_headers for c in ("item_id", "product_id", "movie_id", "candidate_id")):\n        evidence_by_family.setdefault("recommender_ranking", []).append("schema:user_item_pair")\n        scores["recommender_ranking"] = scores.get("recommender_ranking", 0) + 5\n\n    # If temporal columns + forecasting keywords or horizon exist, boost time_series\n    if temporal_cols and ("time_series" in scores or any("lag" in h or "horizon" in h for h in norm_headers)):\n        evidence_by_family.setdefault("time_series", []).append(f"schema:temporal({temporal_cols[0]})")\n        scores["time_series"] = scores.get("time_series", 0) + 3\n\n    ranked = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))\n    if not ranked:\n        detected = ["custom"]\n        confidence = "none"\n    else:\n        top_score = ranked[0][1]\n        # Keep specialized families within strong evidence threshold; don\'t let generic CSV alone override a specific domain\n        selected = [fam for fam, sc in ranked if sc >= max(3, top_score - 3)]\n        if not selected:\n            selected = [ranked[0][0]]\n        if len(selected) > 1 and "classical" in selected and any(\n            f in selected for f in ("computer_vision", "time_series", "finetuning", "rag", "agentic", "audio_speech", "multimodal", "recommender_ranking")\n        ):\n            if scores.get("classical", 0) <= 2:\n                selected.remove("classical")\n        detected = selected[:3]\n        confidence = "high" if top_score >= 5 else "medium" if top_score >= 3 else "low"\n\n    profile = resolve_domain_profile(detected, custom_overrides=custom_overrides)\n    return {\n        "detected_families": detected,\n        "confidence": confidence,\n        "is_hybrid": len(detected) > 1,\n        "evidence_by_family": {k: evidence_by_family[k] for k in detected if k in evidence_by_family},\n        "all_candidate_scores": dict(ranked),\n        "schema_signals": {\n            "temporal_columns": temporal_cols,\n            "entity_or_group_columns": group_cols,\n            "media_columns": media_cols,\n            "text_or_prompt_columns": text_cols,\n            "candidate_target_columns": target_cols,\n        },\n        "domain_profile": profile,\n    }\n\n\n\ndef default_project_memory(project_id):\n    """Create an initial combined project-specific memory record (never global)."""\n    project_id = component(project_id)\n    return {\n        "schema_version": PROJECT_MEMORY_SCHEMA,\n        "memory_scope": "project_specific_only",\n        "global_memory_prohibited": True,\n        "project_id": project_id,\n        "updated_at_utc": utc_now(),\n        "last_writer_skill": "ml-project-setup",\n        "setup_context": {\n            "root_folder_id": None,\n            "task_families": [],\n            "domain_profile": None,\n            "objective": None,\n            "primary_metric_id": None,\n            "split_and_leakage_policy": None,\n            "folder_slots": {},\n            "hf_selected_assets": [],\n            "library_docs_verifications": [],\n        },\n        "destinations": {\n            "tracker_sheet_id": None,\n            "report_doc_id": None,\n            "presentation_slides_id": None,\n            "active_notebook_id": None,\n        },\n        "active_lineage": {\n            "baseline_run_id": None,\n            "baseline_revision_id": None,\n            "best_run_id": None,\n            "best_revision_id": None,\n            "best_metric_value": None,\n            "completed_run_count": 0,\n        },\n        "experiment_memory": {\n            "hyperparameter_lessons": [],\n            "ablation_findings": [],\n            "failed_runs_and_pitfalls": [],\n            "compute_units_total": 0.0,\n        },\n        "publication_memory": {\n            "sheets_last_operation_id": None,\n            "sheets_preserved_regions": [],\n            "docs_last_operation_id": None,\n            "docs_preserved_sections": [],\n            "slides_last_operation_id": None,\n            "slides_preserved_slide_ids": [],\n            "submission_validation_status": "not_run",\n            "validated_submission_artifacts": [],\n            "last_quality_gate_status": "not_run",\n        },\n        "next_experiments_queue": [],\n    }\n\n\ndef validate_project_memory(memory, *, project_id=None):\n    """Validate project-specific memory isolation, schema, and JSON safety."""\n    check_json(memory)\n    if not isinstance(memory, dict) or memory.get("schema_version") != PROJECT_MEMORY_SCHEMA:\n        raise ValueError("Unsupported project memory schema")\n    if memory.get("memory_scope") != "project_specific_only" or memory.get("global_memory_prohibited") is not True:\n        raise ValueError("Project memory must remain strictly project-specific (never global)")\n    pid = component(memory.get("project_id"))\n    if project_id is not None and pid != component(project_id):\n        raise ValueError("Cross-project memory contamination rejected")\n    for section in ("setup_context", "destinations", "active_lineage", "experiment_memory", "publication_memory"):\n        if not isinstance(memory.get(section), dict):\n            raise ValueError(f"Project memory section {section} must be an object")\n    if not isinstance(memory.get("next_experiments_queue"), list):\n        raise ValueError("next_experiments_queue must be a list")\n    return memory\n\n\ndef load_project_memory(project_root, *, project_id=None):\n    """Load combined project-specific memory from 05_reports/autopilot/project_memory.json."""\n    path = within(project_root, PROJECT_MEMORY_RELPATH)\n    if not path.is_file():\n        if project_id is None:\n            raise FileNotFoundError(str(path))\n        return default_project_memory(project_id)\n    return validate_project_memory(read_json(path), project_id=project_id)\n\n\ndef update_project_memory(project_root, project_id, *, skill_name, patch):\n    """Atomically update combined project-specific memory inside the project root."""\n    project_id = component(project_id)\n    skill_name = component(skill_name)\n    if not isinstance(patch, dict) or not patch or any(k not in _MEMORY_SECTIONS for k in patch):\n        raise ValueError("Memory patch must target valid project memory sections")\n    check_json(patch)\n    path = within(project_root, PROJECT_MEMORY_RELPATH)\n    expected = file_digest(path) if path.is_file() else None\n    current = load_project_memory(project_root, project_id=project_id)\n    for section, updates in patch.items():\n        if section == "next_experiments_queue":\n            if not isinstance(updates, list):\n                raise ValueError("next_experiments_queue patch must be a list")\n            current[section] = list(updates)\n        else:\n            if not isinstance(updates, dict):\n                raise ValueError(f"Section {section} patch must be an object")\n            current[section] = {**current[section], **updates}\n    current["updated_at_utc"] = utc_now()\n    current["last_writer_skill"] = skill_name\n    validate_project_memory(current, project_id=project_id)\n    atomic_json(path, current, expected=expected)\n    return current\n\n\n\n'
helper = Path('/content') / ('autopilot_runtime_' + CONFIG['project_id'] + '.py')
helper.write_text(RUNTIME_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('autopilot_runtime', helper)
autopilot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(autopilot)
autopilot.component(RUN_ID)
missing = []
for pin in ([] if RECOVER_RUN_ID else CONFIG.get('packages', [])):
    name, version = pin.split('==', 1)
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(pin)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade-strategy', 'only-if-needed', *missing], check=True)
    raise RuntimeError('Dependencies installed. Restart the Colab runtime, then run from the top.')
DEPENDENCIES = ({'status': 'recovery_only'} if RECOVER_RUN_ID else autopilot.dependency_report(CONFIG.get('packages', []), policy=CONFIG.get('dependency_check', 'required')))
if DEPENDENCIES['status'] == 'unrelated_conflicts':
    print('Environment note:', DEPENDENCIES['unrelated_conflicts'], 'preinstalled conflicts outside this workload; no packages changed.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = autopilot.resolve_project_root('/content/drive', CONFIG['drive_relative_path'], CONFIG['project_id'], override=PROJECT_PATH_OVERRIDE, folder_id=CONFIG.get('drive_folder_id'))
print('Project:', PROJECT_ROOT)


In [ ]:
import shutil, platform, hashlib, importlib.util
SPOOL_ROOT = Path('/content/autopilot_spool') / CONFIG['project_id']
if RECOVER_RUN_ID:
    recovered = autopilot.recover_spooled_events(PROJECT_ROOT, SPOOL_ROOT, RUN_ID, project_id=CONFIG['project_id'])
    print('Recovered', len(recovered), 'staged events. No experiment was rerun.')
    raise RuntimeError('Recovery-only mode finished. Request a project sync; do not run the task cell.')
free_gib = shutil.disk_usage('/content').free / 2**30
if free_gib < CONFIG.get('min_free_disk_gib', 0):
    raise RuntimeError('Insufficient local disk for this task')
gpu_query = shutil.which('nvidia-smi')
gpu_description = None
if gpu_query and (CONFIG.get('gpu_required') or CONFIG.get('min_gpu_memory_gib', 0) > 0):
    gpu_description = subprocess.check_output([gpu_query, '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip()
if CONFIG.get('gpu_required') or CONFIG.get('min_gpu_memory_gib', 0) > 0:
    if not gpu_description:
        raise RuntimeError('Select a compatible GPU runtime in Colab')
    memories = [float(line.rsplit(',', 1)[1].strip()) / 1024 for line in gpu_description.splitlines()]
    if max(memories) < CONFIG.get('min_gpu_memory_gib', 0):
        raise RuntimeError('GPU memory below configured minimum')
ENVIRONMENT = {'python': platform.python_version(), 'platform': platform.platform(), 'gpu': gpu_description, 'packages': {pin.split('==')[0]: importlib.metadata.version(pin.split('==')[0]) for pin in CONFIG.get('packages', [])}}
CHECKPOINT_CONFIG = {'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'evaluation': CONFIG.get('context', {}), 'metric_context': CONFIG.get('metric_context', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}
if RESUME_RUN_ID:
    manifest = CONFIG.get('resume_manifest')
    if not manifest:
        raise RuntimeError('Set resume_manifest and restore task-specific optimizer/RNG/model state explicitly')
    autopilot.verify_checkpoint(PROJECT_ROOT, autopilot.within(PROJECT_ROOT, manifest), CHECKPOINT_CONFIG, run_id=RUN_ID)
print('Preflight passed. Local free GiB:', round(free_gib, 1))


## Task and logging
The task must assign `metrics` from actual observations and call `check_budget()` at safe loop boundaries. Write model, optimizer, scheduler, step, and RNG state to finalized checkpoint artifacts, then call `autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, checkpoint_id, CHECKPOINT_CONFIG, files)`. Restore that state explicitly on resume. The helper verifies artifacts; it does not serialize models. Budget checks cannot interrupt a blocking GPU operation. Estimated cost is user-supplied and does not cap Colab billing. After a completed run, start a fresh run ID for another experiment.


In [ ]:
if globals().get('RECOVER_RUN_ID'):
    raise RuntimeError('Recovery-only mode cannot execute the experiment task.')
EVENT_DIRS = [root / '03_results' / 'runs' / RUN_ID / 'events' for root in (PROJECT_ROOT, SPOOL_ROOT)]
if any((directory / 'final.json').exists() for directory in EVENT_DIRS):
    raise RuntimeError('This run already completed or has a staged final result. Use RECOVER_RUN_ID for delivery, or START_NEW_RUN for an independent attempt.')
if not RESUME_RUN_ID and any(directory.exists() and any(directory.glob('start-*.json')) for directory in EVENT_DIRS):
    raise RuntimeError('An attempt already exists for this run. Use a fresh RUN_ID or explicitly resume a verified checkpoint.')
STARTED = time.monotonic()
metrics = {}
TASK_ARTIFACTS = {}
def check_budget():
    limit = CONFIG.get('budget', {}).get('max_hours')
    if limit is not None and time.monotonic() - STARTED > limit * 3600:
        raise RuntimeError('Runtime budget reached; save a checkpoint and stop')
autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), {}, run_id=RUN_ID, event_id='start-' + str(uuid.uuid4()), status='partial', spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})})
_STAGE_STATE = {'next': 0, 'failed': False, 'finalized': False}
def _execute_stage(index, source):
    if _STAGE_STATE['failed'] or _STAGE_STATE['finalized'] or index != _STAGE_STATE['next']:
        raise RuntimeError('Run task sections once in order; recover or create a new run after failure')
    try:
        check_budget()
        exec(compile(source, '<experiment-stage>', 'exec'), globals())
        check_budget()
    except autopilot.LogDeliveryError:
        _STAGE_STATE['failed'] = True
        print('Delivery pending; preserve spool and recover without repeating compute:', RUN_ID)
        raise
    except BaseException as error:
        _STAGE_STATE['failed'] = True
        status = 'interrupted' if isinstance(error, KeyboardInterrupt) else 'failed'
        try:
            autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), {}, run_id=RUN_ID, event_id='failure-' + str(uuid.uuid4()), status=status, spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}, errors=[{'type': type(error).__name__, 'stage': index}])
        except Exception:
            print('Failure delivery pending; preserve local runtime files.')
        raise
    _STAGE_STATE['next'] += 1


## 1. Preflight, D-FINE-M HF Contract & Precision Dry-Run
Validate ustc-community/dfine-medium-coco (do_normalize=False, [0,1] scale), run a real 2-image (damaged+empty) AMP stability dry-run, detect Colab vs Kaggle runtime, and verify 21,041 unarchived images.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
import csv
import inspect
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from transformers import (
    AutoImageProcessor,
    DFineForObjectDetection,
    get_cosine_schedule_with_warmup,
)

_TF_MAJOR = int(transformers.__version__.split('.')[0])
_TF_DTYPE_KW = {'dtype': torch.float32} if _TF_MAJOR >= 5 else {'torch_dtype': torch.float32}

EXP_CFG = CONFIG['experiment']
SEED = int(EXP_CFG.get('seed', 42))
SEED_RECEIPT = autopilot.seed_everything(SEED, deterministic=False)
CONFIG['experiment']['seed_receipt'] = SEED_RECEIPT

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for D-FINE-M training. Select GPU T4/P100/L4/A100.')

# Ampere/Ada TF32 + cuDNN kernel autotuning
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

def _avail_ram_gib() -> float:
    try:
        return (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_AVPHYS_PAGES')) / (1024 ** 3)
    except Exception:
        for line in Path('/proc/meminfo').read_text(encoding='utf-8').splitlines():
            if line.startswith('MemAvailable:'):
                return float(line.split()[1]) / (1024 ** 2)
    return 12.0

IS_KAGGLE_ENV = Path('/kaggle/input').is_dir() and not Path('/content/drive/MyDrive').is_dir()
WORK_BASE = Path('/kaggle/working') if IS_KAGGLE_ENV else Path('/content')
CACHE_ROOT = WORK_BASE / 'autopilot_cache'
HF_CACHE_DIR = CACHE_ROOT / 'hf_cache'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)

GPU_COUNT = torch.cuda.device_count()
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
FREE_RAM_GIB = _avail_ram_gib()
FREE_DISK_GIB = shutil.disk_usage(str(WORK_BASE)).free / (1024 ** 3)
CPU_CORES = os.cpu_count() or 4

if GPU_VRAM_GIB >= 36.0:
    GPU_TIER = 'A100'
    BATCH_SIZE = int(EXP_CFG.get('batch_a100', 16))
    GRAD_ACCUM = int(EXP_CFG.get('accum_a100', 2))
    EVAL_BATCH = 16
    DATALOADER_WORKERS = min(8, CPU_CORES)
    TARGET_EPOCHS = int(EXP_CFG.get('epochs', 24))
elif GPU_VRAM_GIB >= 20.0:
    GPU_TIER = 'L4'
    BATCH_SIZE = int(EXP_CFG.get('batch_l4', 12))
    GRAD_ACCUM = int(EXP_CFG.get('accum_l4', 2))
    EVAL_BATCH = 12
    DATALOADER_WORKERS = min(6, CPU_CORES)
    TARGET_EPOCHS = int(EXP_CFG.get('epochs', 24))
else:
    GPU_TIER = 'T4'
    BATCH_SIZE = int(EXP_CFG.get('batch_t4', 8))
    GRAD_ACCUM = int(EXP_CFG.get('accum_t4', 3))
    EVAL_BATCH = 8
    DATALOADER_WORKERS = min(4, CPU_CORES)
    TARGET_EPOCHS = int(EXP_CFG.get('epochs_t4', 18))

if EXP_CFG.get('smoke_only'):
    TARGET_EPOCHS = 1

print(f'Env: {"Kaggle" if IS_KAGGLE_ENV else "Colab"} | PyTorch: {torch.__version__} | GPUs: {GPU_COUNT}x {GPU_NAME} ({GPU_VRAM_GIB:.1f} GiB -> {GPU_TIER})')
print(f'RAM free: {FREE_RAM_GIB:.1f} GiB | Disk free: {FREE_DISK_GIB:.1f} GiB | CPUs: {CPU_CORES}')
print(f'Auto config -> bs={BATCH_SIZE}, grad_accum={GRAD_ACCUM} (eff={BATCH_SIZE * GRAD_ACCUM}), epochs={TARGET_EPOCHS}, workers={DATALOADER_WORKERS}')

CLASS_MAP = {
    0: 'D00_longitudinal_crack',
    1: 'D10_transverse_crack',
    2: 'D20_alligator_crack',
    3: 'D40_pothole',
}
LABEL2ID = {v: k for k, v in CLASS_MAP.items()}
MODEL_REPO = str(EXP_CFG.get('model_repo', 'ustc-community/dfine-medium-coco'))
IMGSZ = int(EXP_CFG.get('imgsz', 640))
SKY_CROP_FRAC = float(EXP_CFG.get('sky_crop_frac', 0.22))

# Verify HF D-FINE preprocessor contract & run real 2-image dry-run (damaged + empty 0-box)
assert 'ignore_mismatched_sizes' in inspect.signature(DFineForObjectDetection.from_pretrained).parameters
IMAGE_PROCESSOR = AutoImageProcessor.from_pretrained(MODEL_REPO, cache_dir=str(HF_CACHE_DIR))
print(f'HF ImageProcessor: {type(IMAGE_PROCESSOR).__name__} | size={IMAGE_PROCESSOR.size} | do_rescale={getattr(IMAGE_PROCESSOR, "do_rescale", None)} | do_normalize={getattr(IMAGE_PROCESSOR, "do_normalize", None)}')
assert getattr(IMAGE_PROCESSOR, 'do_normalize', False) is False, 'D-FINE preprocessor expects [0,1] rescale without ImageNet mean/std normalization'
assert getattr(IMAGE_PROCESSOR, 'do_rescale', True) is True

# Select safe AMP precision and validate via real forward + backward on [damaged, empty] batch
DEVICE = torch.device('cuda:0')
_candidate_amp = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
_dry_model = DFineForObjectDetection.from_pretrained(
    MODEL_REPO,
    id2label=CLASS_MAP,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    cache_dir=str(HF_CACHE_DIR),
    **_TF_DTYPE_KW,
).to(DEVICE)
_dry_model.train()

_dry_px = torch.rand((2, 3, IMGSZ, IMGSZ), device=DEVICE, dtype=torch.float32)
_dry_labels = [
    {'class_labels': torch.tensor([0, 2], device=DEVICE, dtype=torch.long), 'boxes': torch.tensor([[0.5, 0.6, 0.2, 0.3], [0.3, 0.7, 0.25, 0.2]], device=DEVICE, dtype=torch.float32)},
    {'class_labels': torch.zeros((0,), device=DEVICE, dtype=torch.long), 'boxes': torch.zeros((0, 4), device=DEVICE, dtype=torch.float32)},
]

# Cross-check annotation encoding vs IMAGE_PROCESSOR reference
_ref_ann = {'image_id': 0, 'annotations': [{'id': 0, 'image_id': 0, 'category_id': 0, 'bbox': [256.0, 288.0, 128.0, 192.0], 'area': 24576.0, 'iscrowd': 0}]}
_ref_proc = IMAGE_PROCESSOR(images=np.zeros((IMGSZ, IMGSZ, 3), dtype=np.uint8), annotations=_ref_ann, return_tensors='pt')
_ref_box = _ref_proc['labels'][0]['boxes'][0].numpy()
assert np.allclose(_ref_box, np.array([0.5, 0.6, 0.2, 0.3], dtype=np.float32), atol=1e-5), f'Box format mismatch: {_ref_box}'

AMP_DTYPE = _candidate_amp
try:
    with torch.amp.autocast('cuda', dtype=_candidate_amp, enabled=True):
        _dry_out = _dry_model(pixel_values=_dry_px, labels=_dry_labels)
    if not torch.isfinite(_dry_out.loss):
        raise FloatingPointError('Non-finite AMP dry-run loss')
    _dry_out.loss.backward()
    for _p in _dry_model.parameters():
        if _p.grad is not None and not torch.isfinite(_p.grad).all():
            raise FloatingPointError('Non-finite AMP gradient')
    with torch.no_grad():
        _dry_out.logits = _dry_out.logits.detach().float()
        _dry_out.pred_boxes = _dry_out.pred_boxes.detach().float()
        _dry_dec = IMAGE_PROCESSOR.post_process_object_detection(
            _dry_out,
            target_sizes=torch.tensor([[IMGSZ, IMGSZ], [IMGSZ, IMGSZ]], dtype=torch.long),
            threshold=0.001,
        )
        assert _dry_dec[0]['scores'].detach().float().cpu().numpy().dtype == np.float32
        assert _dry_dec[0]['boxes'].detach().float().cpu().numpy().dtype == np.float32
    print(f'D-FINE-M real 2-image dry-run PASSED with AMP dtype={AMP_DTYPE} (loss={float(_dry_out.loss.detach()):.4f}).')
except Exception as _amp_exc:
    print(f'WARNING: AMP {_candidate_amp} unstable on {GPU_NAME} ({_amp_exc}); falling back to FP32 for guaranteed convergence.')
    AMP_DTYPE = None
    _dry_model.zero_grad(set_to_none=True)
    _dry_out_fp32 = _dry_model(pixel_values=_dry_px, labels=_dry_labels)
    assert torch.isfinite(_dry_out_fp32.loss), 'FP32 dry-run loss non-finite!'
    _dry_out_fp32.loss.backward()
    print(f'D-FINE-M FP32 fallback dry-run PASSED (loss={float(_dry_out_fp32.loss.detach()):.4f}).')

del _dry_model, _dry_px, _dry_labels
torch.cuda.empty_cache()

# Auto-resolve competition dataset directory (Colab Drive vs Kaggle /kaggle/input)
if IS_KAGGLE_ENV:
    _cands = list(Path('/kaggle/input').rglob('train_labels.csv'))
    if not _cands:
        raise FileNotFoundError('Could not locate train_labels.csv under /kaggle/input. Attach datasprint-2-road-damage-detection dataset.')
    DRIVE_DATA_DIR = _cands[0].parent
    STARTER_SRC_DIR = DRIVE_DATA_DIR
else:
    DRIVE_DATA_DIR = PROJECT_ROOT / '01_data'
    STARTER_SRC_DIR = PROJECT_ROOT / '02_notebooks' / 'starter_kit'

RUN_DRIVE_DIR = PROJECT_ROOT / '03_results' / 'runs' / RUN_ID
RUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DRIVE_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(RUN_DRIVE_DIR / 'weights').mkdir(parents=True, exist_ok=True)

required_csvs = ['train_labels.csv', 'val_labels.csv', 'val_image_ids.csv', 'test_image_ids.csv']
for csv_name in required_csvs:
    if not (DRIVE_DATA_DIR / csv_name).is_file():
        raise FileNotFoundError(f'Missing {csv_name} in {DRIVE_DATA_DIR}')

def _count_jpgs(folder: Path) -> int:
    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}) if folder.is_dir() else 0

train_count = _count_jpgs(DRIVE_DATA_DIR / 'train_images')
val_count = _count_jpgs(DRIVE_DATA_DIR / 'val_images')
test_count = _count_jpgs(DRIVE_DATA_DIR / 'test_images')
print(f'Source image counts -> train: {train_count}, val: {val_count}, test: {test_count}')
if train_count < 14691 or val_count < 2150 or test_count < 4200:
    raise RuntimeError(f'Expected full dataset (14691/2150/4200), got train={train_count}, val={val_count}, test={test_count}')
```


In [ ]:
_execute_stage(0, 'import csv\nimport inspect\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport albumentations as A\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nimport transformers\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset, Sampler\nfrom transformers import (\n    AutoImageProcessor,\n    DFineForObjectDetection,\n    get_cosine_schedule_with_warmup,\n)\n\n_TF_MAJOR = int(transformers.__version__.split(\'.\')[0])\n_TF_DTYPE_KW = {\'dtype\': torch.float32} if _TF_MAJOR >= 5 else {\'torch_dtype\': torch.float32}\n\nEXP_CFG = CONFIG[\'experiment\']\nSEED = int(EXP_CFG.get(\'seed\', 42))\nSEED_RECEIPT = autopilot.seed_everything(SEED, deterministic=False)\nCONFIG[\'experiment\'][\'seed_receipt\'] = SEED_RECEIPT\n\nif not torch.cuda.is_available():\n    raise RuntimeError(\'CUDA GPU is required for D-FINE-M training. Select GPU T4/P100/L4/A100.\')\n\n# Ampere/Ada TF32 + cuDNN kernel autotuning\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\ntorch.set_float32_matmul_precision(\'high\')\ntorch.backends.cudnn.benchmark = True\n\ndef _avail_ram_gib() -> float:\n    try:\n        return (os.sysconf(\'SC_PAGE_SIZE\') * os.sysconf(\'SC_AVPHYS_PAGES\')) / (1024 ** 3)\n    except Exception:\n        for line in Path(\'/proc/meminfo\').read_text(encoding=\'utf-8\').splitlines():\n            if line.startswith(\'MemAvailable:\'):\n                return float(line.split()[1]) / (1024 ** 2)\n    return 12.0\n\nIS_KAGGLE_ENV = Path(\'/kaggle/input\').is_dir() and not Path(\'/content/drive/MyDrive\').is_dir()\nWORK_BASE = Path(\'/kaggle/working\') if IS_KAGGLE_ENV else Path(\'/content\')\nCACHE_ROOT = WORK_BASE / \'autopilot_cache\'\nHF_CACHE_DIR = CACHE_ROOT / \'hf_cache\'\nHF_CACHE_DIR.mkdir(parents=True, exist_ok=True)\nos.environ[\'HF_HOME\'] = str(HF_CACHE_DIR)\n\nGPU_COUNT = torch.cuda.device_count()\nGPU_NAME = torch.cuda.get_device_name(0)\nGPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\nFREE_RAM_GIB = _avail_ram_gib()\nFREE_DISK_GIB = shutil.disk_usage(str(WORK_BASE)).free / (1024 ** 3)\nCPU_CORES = os.cpu_count() or 4\n\nif GPU_VRAM_GIB >= 36.0:\n    GPU_TIER = \'A100\'\n    BATCH_SIZE = int(EXP_CFG.get(\'batch_a100\', 16))\n    GRAD_ACCUM = int(EXP_CFG.get(\'accum_a100\', 2))\n    EVAL_BATCH = 16\n    DATALOADER_WORKERS = min(8, CPU_CORES)\n    TARGET_EPOCHS = int(EXP_CFG.get(\'epochs\', 24))\nelif GPU_VRAM_GIB >= 20.0:\n    GPU_TIER = \'L4\'\n    BATCH_SIZE = int(EXP_CFG.get(\'batch_l4\', 12))\n    GRAD_ACCUM = int(EXP_CFG.get(\'accum_l4\', 2))\n    EVAL_BATCH = 12\n    DATALOADER_WORKERS = min(6, CPU_CORES)\n    TARGET_EPOCHS = int(EXP_CFG.get(\'epochs\', 24))\nelse:\n    GPU_TIER = \'T4\'\n    BATCH_SIZE = int(EXP_CFG.get(\'batch_t4\', 8))\n    GRAD_ACCUM = int(EXP_CFG.get(\'accum_t4\', 3))\n    EVAL_BATCH = 8\n    DATALOADER_WORKERS = min(4, CPU_CORES)\n    TARGET_EPOCHS = int(EXP_CFG.get(\'epochs_t4\', 18))\n\nif EXP_CFG.get(\'smoke_only\'):\n    TARGET_EPOCHS = 1\n\nprint(f\'Env: {"Kaggle" if IS_KAGGLE_ENV else "Colab"} | PyTorch: {torch.__version__} | GPUs: {GPU_COUNT}x {GPU_NAME} ({GPU_VRAM_GIB:.1f} GiB -> {GPU_TIER})\')\nprint(f\'RAM free: {FREE_RAM_GIB:.1f} GiB | Disk free: {FREE_DISK_GIB:.1f} GiB | CPUs: {CPU_CORES}\')\nprint(f\'Auto config -> bs={BATCH_SIZE}, grad_accum={GRAD_ACCUM} (eff={BATCH_SIZE * GRAD_ACCUM}), epochs={TARGET_EPOCHS}, workers={DATALOADER_WORKERS}\')\n\nCLASS_MAP = {\n    0: \'D00_longitudinal_crack\',\n    1: \'D10_transverse_crack\',\n    2: \'D20_alligator_crack\',\n    3: \'D40_pothole\',\n}\nLABEL2ID = {v: k for k, v in CLASS_MAP.items()}\nMODEL_REPO = str(EXP_CFG.get(\'model_repo\', \'ustc-community/dfine-medium-coco\'))\nIMGSZ = int(EXP_CFG.get(\'imgsz\', 640))\nSKY_CROP_FRAC = float(EXP_CFG.get(\'sky_crop_frac\', 0.22))\n\n# Verify HF D-FINE preprocessor contract & run real 2-image dry-run (damaged + empty 0-box)\nassert \'ignore_mismatched_sizes\' in inspect.signature(DFineForObjectDetection.from_pretrained).parameters\nIMAGE_PROCESSOR = AutoImageProcessor.from_pretrained(MODEL_REPO, cache_dir=str(HF_CACHE_DIR))\nprint(f\'HF ImageProcessor: {type(IMAGE_PROCESSOR).__name__} | size={IMAGE_PROCESSOR.size} | do_rescale={getattr(IMAGE_PROCESSOR, "do_rescale", None)} | do_normalize={getattr(IMAGE_PROCESSOR, "do_normalize", None)}\')\nassert getattr(IMAGE_PROCESSOR, \'do_normalize\', False) is False, \'D-FINE preprocessor expects [0,1] rescale without ImageNet mean/std normalization\'\nassert getattr(IMAGE_PROCESSOR, \'do_rescale\', True) is True\n\n# Select safe AMP precision and validate via real forward + backward on [damaged, empty] batch\nDEVICE = torch.device(\'cuda:0\')\n_candidate_amp = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n_dry_model = DFineForObjectDetection.from_pretrained(\n    MODEL_REPO,\n    id2label=CLASS_MAP,\n    label2id=LABEL2ID,\n    ignore_mismatched_sizes=True,\n    cache_dir=str(HF_CACHE_DIR),\n    **_TF_DTYPE_KW,\n).to(DEVICE)\n_dry_model.train()\n\n_dry_px = torch.rand((2, 3, IMGSZ, IMGSZ), device=DEVICE, dtype=torch.float32)\n_dry_labels = [\n    {\'class_labels\': torch.tensor([0, 2], device=DEVICE, dtype=torch.long), \'boxes\': torch.tensor([[0.5, 0.6, 0.2, 0.3], [0.3, 0.7, 0.25, 0.2]], device=DEVICE, dtype=torch.float32)},\n    {\'class_labels\': torch.zeros((0,), device=DEVICE, dtype=torch.long), \'boxes\': torch.zeros((0, 4), device=DEVICE, dtype=torch.float32)},\n]\n\n# Cross-check annotation encoding vs IMAGE_PROCESSOR reference\n_ref_ann = {\'image_id\': 0, \'annotations\': [{\'id\': 0, \'image_id\': 0, \'category_id\': 0, \'bbox\': [256.0, 288.0, 128.0, 192.0], \'area\': 24576.0, \'iscrowd\': 0}]}\n_ref_proc = IMAGE_PROCESSOR(images=np.zeros((IMGSZ, IMGSZ, 3), dtype=np.uint8), annotations=_ref_ann, return_tensors=\'pt\')\n_ref_box = _ref_proc[\'labels\'][0][\'boxes\'][0].numpy()\nassert np.allclose(_ref_box, np.array([0.5, 0.6, 0.2, 0.3], dtype=np.float32), atol=1e-5), f\'Box format mismatch: {_ref_box}\'\n\nAMP_DTYPE = _candidate_amp\ntry:\n    with torch.amp.autocast(\'cuda\', dtype=_candidate_amp, enabled=True):\n        _dry_out = _dry_model(pixel_values=_dry_px, labels=_dry_labels)\n    if not torch.isfinite(_dry_out.loss):\n        raise FloatingPointError(\'Non-finite AMP dry-run loss\')\n    _dry_out.loss.backward()\n    for _p in _dry_model.parameters():\n        if _p.grad is not None and not torch.isfinite(_p.grad).all():\n            raise FloatingPointError(\'Non-finite AMP gradient\')\n    with torch.no_grad():\n        _dry_out.logits = _dry_out.logits.detach().float()\n        _dry_out.pred_boxes = _dry_out.pred_boxes.detach().float()\n        _dry_dec = IMAGE_PROCESSOR.post_process_object_detection(\n            _dry_out,\n            target_sizes=torch.tensor([[IMGSZ, IMGSZ], [IMGSZ, IMGSZ]], dtype=torch.long),\n            threshold=0.001,\n        )\n        assert _dry_dec[0][\'scores\'].detach().float().cpu().numpy().dtype == np.float32\n        assert _dry_dec[0][\'boxes\'].detach().float().cpu().numpy().dtype == np.float32\n    print(f\'D-FINE-M real 2-image dry-run PASSED with AMP dtype={AMP_DTYPE} (loss={float(_dry_out.loss.detach()):.4f}).\')\nexcept Exception as _amp_exc:\n    print(f\'WARNING: AMP {_candidate_amp} unstable on {GPU_NAME} ({_amp_exc}); falling back to FP32 for guaranteed convergence.\')\n    AMP_DTYPE = None\n    _dry_model.zero_grad(set_to_none=True)\n    _dry_out_fp32 = _dry_model(pixel_values=_dry_px, labels=_dry_labels)\n    assert torch.isfinite(_dry_out_fp32.loss), \'FP32 dry-run loss non-finite!\'\n    _dry_out_fp32.loss.backward()\n    print(f\'D-FINE-M FP32 fallback dry-run PASSED (loss={float(_dry_out_fp32.loss.detach()):.4f}).\')\n\ndel _dry_model, _dry_px, _dry_labels\ntorch.cuda.empty_cache()\n\n# Auto-resolve competition dataset directory (Colab Drive vs Kaggle /kaggle/input)\nif IS_KAGGLE_ENV:\n    _cands = list(Path(\'/kaggle/input\').rglob(\'train_labels.csv\'))\n    if not _cands:\n        raise FileNotFoundError(\'Could not locate train_labels.csv under /kaggle/input. Attach datasprint-2-road-damage-detection dataset.\')\n    DRIVE_DATA_DIR = _cands[0].parent\n    STARTER_SRC_DIR = DRIVE_DATA_DIR\nelse:\n    DRIVE_DATA_DIR = PROJECT_ROOT / \'01_data\'\n    STARTER_SRC_DIR = PROJECT_ROOT / \'02_notebooks\' / \'starter_kit\'\n\nRUN_DRIVE_DIR = PROJECT_ROOT / \'03_results\' / \'runs\' / RUN_ID\nRUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)\n(RUN_DRIVE_DIR / \'checkpoints\').mkdir(parents=True, exist_ok=True)\n(RUN_DRIVE_DIR / \'weights\').mkdir(parents=True, exist_ok=True)\n\nrequired_csvs = [\'train_labels.csv\', \'val_labels.csv\', \'val_image_ids.csv\', \'test_image_ids.csv\']\nfor csv_name in required_csvs:\n    if not (DRIVE_DATA_DIR / csv_name).is_file():\n        raise FileNotFoundError(f\'Missing {csv_name} in {DRIVE_DATA_DIR}\')\n\ndef _count_jpgs(folder: Path) -> int:\n    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {\'.jpg\', \'.jpeg\'}) if folder.is_dir() else 0\n\ntrain_count = _count_jpgs(DRIVE_DATA_DIR / \'train_images\')\nval_count = _count_jpgs(DRIVE_DATA_DIR / \'val_images\')\ntest_count = _count_jpgs(DRIVE_DATA_DIR / \'test_images\')\nprint(f\'Source image counts -> train: {train_count}, val: {val_count}, test: {test_count}\')\nif train_count < 14691 or val_count < 2150 or test_count < 4200:\n    raise RuntimeError(f\'Expected full dataset (14691/2150/4200), got train={train_count}, val={val_count}, test={test_count}\')')


## 2. Parallel Local SSD Staging & Dataset Integrity Summary
Stage all 4 CSVs, 6 starter scripts, and 21,041 images to fast local storage (parallel copy on Colab / instant symlinks on Kaggle) and log eda_summary.json.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
KIT_LOCAL = CACHE_ROOT / 'datasprint_kit'
KIT_LOCAL.mkdir(parents=True, exist_ok=True)

for csv_name in required_csvs:
    src_csv = DRIVE_DATA_DIR / csv_name
    if IS_KAGGLE_ENV:
        shutil.copy2(src_csv, KIT_LOCAL / csv_name)
    else:
        rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'01_data/{csv_name}', cache_root=str(CACHE_ROOT))
        shutil.copy2(rec['local_path'], KIT_LOCAL / csv_name)

for sname in ('benchmark_speed.py', 'check_submission.py', 'inference.py', 'make_kaggle_submission.py', 'score.py', 'train_baseline.py'):
    if IS_KAGGLE_ENV:
        cand_s = STARTER_SRC_DIR / sname
        if not cand_s.is_file():
            found_s = list(Path('/kaggle/input').rglob(sname))
            cand_s = found_s[0] if found_s else cand_s
        shutil.copy2(cand_s, KIT_LOCAL / sname)
    else:
        s_rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'02_notebooks/starter_kit/{sname}', cache_root=str(CACHE_ROOT))
        shutil.copy2(s_rec['local_path'], KIT_LOCAL / sname)

def _stage_split_fast(split_folder: str) -> int:
    src_dir = DRIVE_DATA_DIR / split_folder
    dst_dir = KIT_LOCAL / split_folder
    dst_dir.mkdir(parents=True, exist_ok=True)
    files = [p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}]
    t0 = time.monotonic()
    if IS_KAGGLE_ENV:
        for p in files:
            tgt = dst_dir / p.name
            if not tgt.exists():
                os.symlink(p.resolve(), tgt)
        print(f'  [{split_folder}] linked {len(files)} images in {time.monotonic() - t0:.2f}s')
        return len(files)

    def _copy_one(p: Path):
        tgt = dst_dir / p.name
        if not tgt.is_file() or tgt.stat().st_size != p.stat().st_size:
            shutil.copy2(p, tgt)
        return 1

    done = 0
    with ThreadPoolExecutor(max_workers=24) as pool:
        for _ in pool.map(_copy_one, files, chunksize=64):
            done += 1
            if done % 3000 == 0 or done == len(files):
                print(f'  [{split_folder}] staged {done}/{len(files)} ({done / max(0.01, time.monotonic() - t0):.0f} img/s)')
    return len(files)

staged_train = _stage_split_fast('train_images')
staged_val = _stage_split_fast('val_images')
staged_test = _stage_split_fast('test_images')
check_budget()

train_df = pd.read_csv(KIT_LOCAL / 'train_labels.csv')
val_df = pd.read_csv(KIT_LOCAL / 'val_labels.csv')
val_ids_df = pd.read_csv(KIT_LOCAL / 'val_image_ids.csv')
test_ids_df = pd.read_csv(KIT_LOCAL / 'test_image_ids.csv')

def _vec_split_summary(df: pd.DataFrame, total_imgs: int) -> dict:
    n_dam = int(df['image_id'].nunique())
    vc = df['class_id'].value_counts().sort_index()
    return {
        'total_images': int(total_imgs),
        'damaged_images': n_dam,
        'background_only_images': int(total_imgs - n_dam),
        'total_boxes': int(len(df)),
        'invalid_boxes': int(((df['x_max'] <= df['x_min']) | (df['y_max'] <= df['y_min'])).sum()),
        'class_counts': {CLASS_MAP[int(k)]: int(v) for k, v in vc.items()},
    }

eda_summary = {
    'train': _vec_split_summary(train_df, staged_train),
    'val': _vec_split_summary(val_df, staged_val),
    'val_image_ids_rows': int(len(val_ids_df)),
    'test_image_ids_rows': int(len(test_ids_df)),
    'staged_test_images': int(staged_test),
}
(RUN_DRIVE_DIR / 'eda_summary.json').write_text(json.dumps(eda_summary, indent=2), encoding='utf-8')
TASK_ARTIFACTS['eda_summary'] = f'03_results/runs/{RUN_ID}/eda_summary.json'
print(json.dumps(eda_summary, indent=2))
```


In [ ]:
_execute_stage(1, "KIT_LOCAL = CACHE_ROOT / 'datasprint_kit'\nKIT_LOCAL.mkdir(parents=True, exist_ok=True)\n\nfor csv_name in required_csvs:\n    src_csv = DRIVE_DATA_DIR / csv_name\n    if IS_KAGGLE_ENV:\n        shutil.copy2(src_csv, KIT_LOCAL / csv_name)\n    else:\n        rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'01_data/{csv_name}', cache_root=str(CACHE_ROOT))\n        shutil.copy2(rec['local_path'], KIT_LOCAL / csv_name)\n\nfor sname in ('benchmark_speed.py', 'check_submission.py', 'inference.py', 'make_kaggle_submission.py', 'score.py', 'train_baseline.py'):\n    if IS_KAGGLE_ENV:\n        cand_s = STARTER_SRC_DIR / sname\n        if not cand_s.is_file():\n            found_s = list(Path('/kaggle/input').rglob(sname))\n            cand_s = found_s[0] if found_s else cand_s\n        shutil.copy2(cand_s, KIT_LOCAL / sname)\n    else:\n        s_rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'02_notebooks/starter_kit/{sname}', cache_root=str(CACHE_ROOT))\n        shutil.copy2(s_rec['local_path'], KIT_LOCAL / sname)\n\ndef _stage_split_fast(split_folder: str) -> int:\n    src_dir = DRIVE_DATA_DIR / split_folder\n    dst_dir = KIT_LOCAL / split_folder\n    dst_dir.mkdir(parents=True, exist_ok=True)\n    files = [p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}]\n    t0 = time.monotonic()\n    if IS_KAGGLE_ENV:\n        for p in files:\n            tgt = dst_dir / p.name\n            if not tgt.exists():\n                os.symlink(p.resolve(), tgt)\n        print(f'  [{split_folder}] linked {len(files)} images in {time.monotonic() - t0:.2f}s')\n        return len(files)\n\n    def _copy_one(p: Path):\n        tgt = dst_dir / p.name\n        if not tgt.is_file() or tgt.stat().st_size != p.stat().st_size:\n            shutil.copy2(p, tgt)\n        return 1\n\n    done = 0\n    with ThreadPoolExecutor(max_workers=24) as pool:\n        for _ in pool.map(_copy_one, files, chunksize=64):\n            done += 1\n            if done % 3000 == 0 or done == len(files):\n                print(f'  [{split_folder}] staged {done}/{len(files)} ({done / max(0.01, time.monotonic() - t0):.0f} img/s)')\n    return len(files)\n\nstaged_train = _stage_split_fast('train_images')\nstaged_val = _stage_split_fast('val_images')\nstaged_test = _stage_split_fast('test_images')\ncheck_budget()\n\ntrain_df = pd.read_csv(KIT_LOCAL / 'train_labels.csv')\nval_df = pd.read_csv(KIT_LOCAL / 'val_labels.csv')\nval_ids_df = pd.read_csv(KIT_LOCAL / 'val_image_ids.csv')\ntest_ids_df = pd.read_csv(KIT_LOCAL / 'test_image_ids.csv')\n\ndef _vec_split_summary(df: pd.DataFrame, total_imgs: int) -> dict:\n    n_dam = int(df['image_id'].nunique())\n    vc = df['class_id'].value_counts().sort_index()\n    return {\n        'total_images': int(total_imgs),\n        'damaged_images': n_dam,\n        'background_only_images': int(total_imgs - n_dam),\n        'total_boxes': int(len(df)),\n        'invalid_boxes': int(((df['x_max'] <= df['x_min']) | (df['y_max'] <= df['y_min'])).sum()),\n        'class_counts': {CLASS_MAP[int(k)]: int(v) for k, v in vc.items()},\n    }\n\neda_summary = {\n    'train': _vec_split_summary(train_df, staged_train),\n    'val': _vec_split_summary(val_df, staged_val),\n    'val_image_ids_rows': int(len(val_ids_df)),\n    'test_image_ids_rows': int(len(test_ids_df)),\n    'staged_test_images': int(staged_test),\n}\n(RUN_DRIVE_DIR / 'eda_summary.json').write_text(json.dumps(eda_summary, indent=2), encoding='utf-8')\nTASK_ARTIFACTS['eda_summary'] = f'03_results/runs/{RUN_ID}/eda_summary.json'\nprint(json.dumps(eda_summary, indent=2))")


## 3. Top-22% Sky-Crop Audit, LAB-CLAHE + 70/30 OneOf Dataset & 4-Bucket Stress Val
Synthesizes GRDDC Top-22% sky-crop, Eigen-CAM background cleanup, India 720x720 bright-concrete LAB-CLAHE, 70/30 OneOf robustness augs, DamagedGuaranteedBatchSampler, and deterministic 4-bucket stress validation.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
WORK_ROOT = CACHE_ROOT / 'dfine_work' / RUN_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# 1. Core geometric/photometric helpers (shared across train, eval & standalone inference)
def compute_sky_crop_offset(img_h: int, crop_frac: float) -> int:
    return max(0, min(img_h - 32, int(round(float(img_h) * float(crop_frac)))))

def shift_boxes_for_sky_crop(xyxy_list, labels_list, img_h: int, y_off: int, min_h: float = 4.0, min_area_vis: float = 0.20):
    crop_h = float(img_h - y_off)
    out_boxes, out_labels = [], []
    for box, cls_id in zip(xyxy_list, labels_list):
        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
        old_area = max(1e-6, (x2 - x1) * (y2 - y1))
        ny1 = max(0.0, y1 - float(y_off))
        ny2 = min(crop_h, y2 - float(y_off))
        nw = max(0.0, x2 - x1)
        nh = max(0.0, ny2 - ny1)
        if nh < min_h or nw < 2.0 or (nw * nh) / old_area < min_area_vis:
            continue
        out_boxes.append([x1, ny1, x2, ny2])
        out_labels.append(int(cls_id))
    return out_boxes, out_labels

def restore_boxes_from_sky_crop(xyxy_arr: np.ndarray, orig_h: int, orig_w: int, y_off: int) -> np.ndarray:
    if len(xyxy_arr) == 0:
        return xyxy_arr
    out = xyxy_arr.copy()
    out[:, 0] = np.clip(out[:, 0], 0.0, float(orig_w))
    out[:, 1] = np.clip(out[:, 1] + float(y_off), 0.0, float(orig_h))
    out[:, 2] = np.clip(out[:, 2], 0.0, float(orig_w))
    out[:, 3] = np.clip(out[:, 3] + float(y_off), 0.0, float(orig_h))
    return out

def apply_lab_clahe_rgb(img_rgb: np.ndarray, clip_limit: float = 2.0, grid_size: int = 8) -> np.ndarray:
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l_chan, a_chan, b_chan = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))
    l_eq = clahe.apply(l_chan)
    merged = cv2.merge((l_eq, a_chan, b_chan))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def build_lut_table(luma_thresh: float, target_luma: float) -> dict:
    x = np.arange(256, dtype=np.float32) / 255.0
    tgt = math.log(float(target_luma) / 255.0)
    luts = {}
    for lum in range(int(luma_thresh)):
        gamma = max(0.35, min(0.85, tgt / math.log(max(float(lum), 8.0) / 255.0)))
        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)
    return luts

def wbf_class_aware_two_views(boxes_a, scores_a, labels_a, boxes_b, scores_b, labels_b, iou_thr: float = 0.55, max_det: int = 100):
    """Class-aware Weighted Boxes Fusion across two TTA views (preserves cross-class overlaps)."""
    merged = []
    all_classes = sorted(set([int(c) for c in labels_a] + [int(c) for c in labels_b]))
    for cls_id in all_classes:
        cand = []
        for idx_b, (b, s, c) in enumerate(zip(boxes_a, scores_a, labels_a)):
            if int(c) == cls_id:
                cand.append((float(s), np.asarray(b, dtype=np.float64), 0))
        for idx_b, (b, s, c) in enumerate(zip(boxes_b, scores_b, labels_b)):
            if int(c) == cls_id:
                cand.append((float(s), np.asarray(b, dtype=np.float64), 1))
        cand.sort(key=lambda t: t[0], reverse=True)
        clusters = []
        for sc, box, view_id in cand:
            best_i = -1
            best_iou = iou_thr
            bx1, by1, bx2, by2 = box
            barea = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
            for ci, cl in enumerate(clusters):
                cx1, cy1, cx2, cy2 = cl['fused']
                iw = max(0.0, min(bx2, cx2) - max(bx1, cx1))
                ih = max(0.0, min(by2, cy2) - max(by1, cy1))
                inter = iw * ih
                if inter <= 0.0:
                    continue
                carea = max(0.0, cx2 - cx1) * max(0.0, cy2 - cy1)
                iou = inter / max(1e-9, barea + carea - inter)
                if iou >= best_iou:
                    best_iou = iou
                    best_i = ci
            if best_i >= 0:
                cl = clusters[best_i]
                w_sum = cl['w_sum'] + sc
                cl['fused'] = (cl['fused'] * cl['w_sum'] + box * sc) / max(1e-9, w_sum)
                cl['w_sum'] = w_sum
                cl['scores'].append(sc)
                cl['views'].add(view_id)
            else:
                clusters.append({'fused': box.copy(), 'w_sum': sc, 'scores': [sc], 'views': {view_id}})
        for cl in clusters:
            # Penalize single-view-only boxes slightly by N_views / 2.0 (standard WBF with 2 views)
            fused_sc = float(sum(cl['scores']) / 2.0)
            fb = cl['fused']
            merged.append((cls_id, fused_sc, float(fb[0]), float(fb[1]), float(fb[2]), float(fb[3])))
    merged.sort(key=lambda t: t[1], reverse=True)
    return merged[:max_det]

# 2. Vectorized Sky-Crop Retention Audit on all 20,078 GT boxes
def _build_image_index(split_name: str, lbl_df: pd.DataFrame):
    img_dir = KIT_LOCAL / f'{split_name}_images'
    img_paths = sorted(img_dir.glob('*.jpg'))
    def _read_dim(p: Path):
        with Image.open(p) as im:
            return p.stem, im.size[0], im.size[1], str(p)
    with ThreadPoolExecutor(max_workers=16) as pool:
        rows = list(pool.map(_read_dim, img_paths, chunksize=64))
    dims_df = pd.DataFrame(rows, columns=['image_id', 'w', 'h', 'path'])
    grouped = {}
    for iid, grp in lbl_df.groupby('image_id'):
        grouped[iid] = (
            grp[['x_min', 'y_min', 'x_max', 'y_max']].to_numpy(dtype=np.float32).tolist(),
            grp['class_id'].to_numpy(dtype=np.int64).tolist(),
        )
    records = []
    n_total_boxes, n_kept_boxes, n_clipped_boxes = 0, 0, 0
    for row in rows:
        iid, w, h, p_str = row
        boxes, labels = grouped.get(iid, ([], []))
        y_off = compute_sky_crop_offset(h, SKY_CROP_FRAC)
        kb, _ = shift_boxes_for_sky_crop(boxes, labels, h, y_off)
        n_total_boxes += len(boxes)
        n_kept_boxes += len(kb)
        for bx in boxes:
            if float(bx[1]) < float(y_off) < float(bx[3]):
                n_clipped_boxes += 1
        records.append({'image_id': iid, 'w': int(w), 'h': int(h), 'path': p_str, 'boxes': boxes, 'labels': labels})
    pct_kept = 100.0 * n_kept_boxes / max(1, n_total_boxes)
    print(f'[{split_name}] Top-{int(SKY_CROP_FRAC*100)}% Sky-Crop Audit: {len(records)} imgs | GT kept {n_kept_boxes}/{n_total_boxes} ({pct_kept:.2f}%), upper-clipped {n_clipped_boxes}')
    return records, dims_df

train_records, train_dims_df = _build_image_index('train', train_df)
val_records, val_dims_df = _build_image_index('val', val_df)

# 3. Road-Safe + EDA-Calibrated Dataset & DamagedGuaranteedBatchSampler
ROB_P = float(EXP_CFG.get('robustness_aug_p', 0.30))
CLAHE_TRAIN_P = float(EXP_CFG.get('clahe_train_p', 0.35))
CROP_JITTER_MIN = float(EXP_CFG.get('sky_crop_jitter_min', 0.18))
CROP_JITTER_MAX = float(EXP_CFG.get('sky_crop_jitter_max', 0.24))

class RoadDamageDFineDataset(Dataset):
    def __init__(self, records, imgsz: int, is_train: bool = True, override_dir: Path = None):
        self.records = records
        self.imgsz = int(imgsz)
        self.is_train = bool(is_train)
        self.override_dir = override_dir
        if self.is_train:
            self.geom_color_aug = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.Affine(scale=(0.88, 1.12), translate_percent={'x': (-0.05, 0.05), 'y': (-0.04, 0.04)}, rotate=0.0, border_mode=cv2.BORDER_CONSTANT, p=0.45),
                A.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.25, hue=0.015, p=0.45),
                A.OneOf([
                    A.ImageCompression(compression_type='jpeg', quality_range=(6, 22), p=1.0),
                    A.Compose([
                        A.RandomBrightnessContrast(brightness_limit=(-0.75, -0.45), contrast_limit=(-0.35, 0.0), p=1.0),
                        A.RandomGamma(gamma_limit=(140, 220), p=0.5),
                    ], p=1.0),
                    A.OneOf([
                        A.GaussianBlur(blur_limit=(7, 15), p=1.0),
                        A.MotionBlur(blur_limit=(7, 15), p=1.0),
                    ], p=1.0),
                    A.GaussNoise(std_range=(0.05, 0.15), p=1.0),
                ], p=ROB_P),
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], clip=True, min_area=12.0))
        else:
            self.geom_color_aug = None

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        img_path = (self.override_dir / f"{rec['image_id']}.jpg") if self.override_dir is not None else Path(rec['path'])
        with Image.open(img_path) as im:
            img = np.asarray(im.convert('RGB'))
        orig_h, orig_w = img.shape[0], img.shape[1]

        frac = random.uniform(CROP_JITTER_MIN, CROP_JITTER_MAX) if self.is_train else SKY_CROP_FRAC
        y_off = compute_sky_crop_offset(orig_h, frac)
        crop = img[y_off:, :, :]
        crop_h, crop_w = crop.shape[0], crop.shape[1]

        boxes_px, labels = shift_boxes_for_sky_crop(rec['boxes'], rec['labels'], orig_h, y_off)

        if self.is_train:
            if random.random() < CLAHE_TRAIN_P:
                crop = apply_lab_clahe_rgb(crop, clip_limit=2.0, grid_size=8)
            aug = self.geom_color_aug(image=crop, bboxes=boxes_px, class_labels=labels)
            crop = aug['image']
            boxes_px = [list(b) for b in aug['bboxes']]
            labels = [int(c) for c in aug['class_labels']]

        resized = cv2.resize(crop, (self.imgsz, self.imgsz), interpolation=cv2.INTER_LINEAR)
        px_tensor = torch.from_numpy(np.ascontiguousarray(resized)).permute(2, 0, 1).to(torch.float32).div_(255.0)

        if boxes_px:
            arr = np.asarray(boxes_px, dtype=np.float32)
            # Normalized (cx, cy, w, h) relative to cropped dimensions (crop_w, crop_h)
            cx = np.clip((arr[:, 0] + arr[:, 2]) * 0.5 / float(crop_w), 0.0, 1.0)
            cy = np.clip((arr[:, 1] + arr[:, 3]) * 0.5 / float(crop_h), 0.0, 1.0)
            bw = np.clip((arr[:, 2] - arr[:, 0]) / float(crop_w), 1e-4, 1.0)
            bh = np.clip((arr[:, 3] - arr[:, 1]) / float(crop_h), 1e-4, 1.0)
            norm_boxes = torch.from_numpy(np.stack([cx, cy, bw, bh], axis=1)).to(torch.float32)
            cls_tensor = torch.tensor(labels, dtype=torch.long)
        else:
            norm_boxes = torch.zeros((0, 4), dtype=torch.float32)
            cls_tensor = torch.zeros((0,), dtype=torch.long)

        return {
            'pixel_values': px_tensor,
            'target': {'class_labels': cls_tensor, 'boxes': norm_boxes},
            'image_id': rec['image_id'],
            'orig_size': (orig_h, orig_w),
            'crop_size': (crop_h, crop_w),
            'y_off': y_off,
        }

def dfine_collate_fn(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch], dim=0),
        'labels': [b['target'] for b in batch],
        'image_ids': [b['image_id'] for b in batch],
        'orig_sizes': [b['orig_size'] for b in batch],
        'crop_sizes': [b['crop_size'] for b in batch],
        'y_offs': [b['y_off'] for b in batch],
    }

class DamagedGuaranteedBatchSampler(Sampler):
    """Visits 100% of train images once per epoch while guaranteeing >=1 damaged image in every batch."""
    def __init__(self, damaged_indices: list, clean_indices: list, batch_size: int, seed: int = 42):
        self.damaged = list(damaged_indices)
        self.clean = list(clean_indices)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.epoch = 0
        self.total = len(self.damaged) + len(self.clean)
        self.num_batches = max(1, math.ceil(self.total / self.batch_size))
        if len(self.damaged) < self.num_batches:
            raise ValueError(f'Need damaged >= num_batches ({len(self.damaged)} < {self.num_batches})')

    def set_epoch(self, ep: int):
        self.epoch = int(ep)

    def __len__(self) -> int:
        return self.num_batches

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch * 9973)
        dam = list(self.damaged)
        cln = list(self.clean)
        rng.shuffle(dam)
        rng.shuffle(cln)
        # Anchor 1 guaranteed damaged image into every single batch (including last batch)
        anchors = dam[:self.num_batches]
        rest = dam[self.num_batches:] + cln
        rng.shuffle(rest)
        batches = []
        cursor = 0
        for b_idx in range(self.num_batches):
            cap = self.batch_size - 1
            if b_idx == self.num_batches - 1:
                take = len(rest) - cursor
            else:
                take = min(cap, max(0, len(rest) - cursor))
            batch = [anchors[b_idx]] + rest[cursor:cursor + take]
            cursor += take
            rng.shuffle(batch)
            batches.append(batch)
        return iter(batches)

train_ds = RoadDamageDFineDataset(train_records, IMGSZ, is_train=True)
val_ds = RoadDamageDFineDataset(val_records, IMGSZ, is_train=False)
damaged_idx = [i for i, r in enumerate(train_records) if len(r['boxes']) > 0]
clean_idx = [i for i, r in enumerate(train_records) if len(r['boxes']) == 0]
train_sampler = DamagedGuaranteedBatchSampler(damaged_idx, clean_idx, BATCH_SIZE, seed=SEED)
train_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=DATALOADER_WORKERS, pin_memory=True, persistent_workers=(DATALOADER_WORKERS > 0), collate_fn=dfine_collate_fn)
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=DATALOADER_WORKERS, pin_memory=True, collate_fn=dfine_collate_fn)
print(f'DataLoaders ready -> train batches={len(train_loader)} (damaged={len(damaged_idx)}, clean={len(clean_idx)}), val batches={len(val_loader)}')

# 4. Build exact deterministic 4-bucket degraded val set matching v3
VAL_DEGRADED_DIR = WORK_ROOT / 'val_images_degraded'
VAL_DEGRADED_DIR.mkdir(parents=True, exist_ok=True)
val_jpgs = sorted((KIT_LOCAL / 'val_images').glob('*.jpg'))

_STRESS_BUCKETS = [
    lambda s: A.Compose([A.ImageCompression(compression_type='jpeg', quality_range=(8, 18), p=1.0)], seed=s),
    lambda s: A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.72, -0.48), contrast_limit=(-0.30, 0.0), p=1.0)], seed=s),
    lambda s: A.Compose([A.OneOf([A.GaussianBlur(blur_limit=(9, 13), p=1.0), A.MotionBlur(blur_limit=(9, 13), p=1.0)], p=1.0)], seed=s),
    lambda s: A.Compose([A.GaussNoise(std_range=(0.06, 0.13), p=1.0)], seed=s),
]

def _degrade_one(idx_path):
    idx, p = idx_path
    out_p = VAL_DEGRADED_DIR / p.name
    if out_p.is_file():
        return 1
    pipe = _STRESS_BUCKETS[idx % 4](SEED + idx)
    arr = np.asarray(Image.open(p).convert('RGB'))
    deg = pipe(image=arr)['image']
    Image.fromarray(deg).save(out_p, format='JPEG', quality=88)
    return 1

with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(_degrade_one, enumerate(val_jpgs), chunksize=32))
print(f'Built 4-bucket degraded val set ({len(val_jpgs)} imgs) at {VAL_DEGRADED_DIR}')
check_budget()
```


In [ ]:
_execute_stage(2, 'WORK_ROOT = CACHE_ROOT / \'dfine_work\' / RUN_ID\nWORK_ROOT.mkdir(parents=True, exist_ok=True)\n\n# 1. Core geometric/photometric helpers (shared across train, eval & standalone inference)\ndef compute_sky_crop_offset(img_h: int, crop_frac: float) -> int:\n    return max(0, min(img_h - 32, int(round(float(img_h) * float(crop_frac)))))\n\ndef shift_boxes_for_sky_crop(xyxy_list, labels_list, img_h: int, y_off: int, min_h: float = 4.0, min_area_vis: float = 0.20):\n    crop_h = float(img_h - y_off)\n    out_boxes, out_labels = [], []\n    for box, cls_id in zip(xyxy_list, labels_list):\n        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])\n        old_area = max(1e-6, (x2 - x1) * (y2 - y1))\n        ny1 = max(0.0, y1 - float(y_off))\n        ny2 = min(crop_h, y2 - float(y_off))\n        nw = max(0.0, x2 - x1)\n        nh = max(0.0, ny2 - ny1)\n        if nh < min_h or nw < 2.0 or (nw * nh) / old_area < min_area_vis:\n            continue\n        out_boxes.append([x1, ny1, x2, ny2])\n        out_labels.append(int(cls_id))\n    return out_boxes, out_labels\n\ndef restore_boxes_from_sky_crop(xyxy_arr: np.ndarray, orig_h: int, orig_w: int, y_off: int) -> np.ndarray:\n    if len(xyxy_arr) == 0:\n        return xyxy_arr\n    out = xyxy_arr.copy()\n    out[:, 0] = np.clip(out[:, 0], 0.0, float(orig_w))\n    out[:, 1] = np.clip(out[:, 1] + float(y_off), 0.0, float(orig_h))\n    out[:, 2] = np.clip(out[:, 2], 0.0, float(orig_w))\n    out[:, 3] = np.clip(out[:, 3] + float(y_off), 0.0, float(orig_h))\n    return out\n\ndef apply_lab_clahe_rgb(img_rgb: np.ndarray, clip_limit: float = 2.0, grid_size: int = 8) -> np.ndarray:\n    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)\n    l_chan, a_chan, b_chan = cv2.split(lab)\n    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))\n    l_eq = clahe.apply(l_chan)\n    merged = cv2.merge((l_eq, a_chan, b_chan))\n    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)\n\ndef build_lut_table(luma_thresh: float, target_luma: float) -> dict:\n    x = np.arange(256, dtype=np.float32) / 255.0\n    tgt = math.log(float(target_luma) / 255.0)\n    luts = {}\n    for lum in range(int(luma_thresh)):\n        gamma = max(0.35, min(0.85, tgt / math.log(max(float(lum), 8.0) / 255.0)))\n        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)\n    return luts\n\ndef wbf_class_aware_two_views(boxes_a, scores_a, labels_a, boxes_b, scores_b, labels_b, iou_thr: float = 0.55, max_det: int = 100):\n    """Class-aware Weighted Boxes Fusion across two TTA views (preserves cross-class overlaps)."""\n    merged = []\n    all_classes = sorted(set([int(c) for c in labels_a] + [int(c) for c in labels_b]))\n    for cls_id in all_classes:\n        cand = []\n        for idx_b, (b, s, c) in enumerate(zip(boxes_a, scores_a, labels_a)):\n            if int(c) == cls_id:\n                cand.append((float(s), np.asarray(b, dtype=np.float64), 0))\n        for idx_b, (b, s, c) in enumerate(zip(boxes_b, scores_b, labels_b)):\n            if int(c) == cls_id:\n                cand.append((float(s), np.asarray(b, dtype=np.float64), 1))\n        cand.sort(key=lambda t: t[0], reverse=True)\n        clusters = []\n        for sc, box, view_id in cand:\n            best_i = -1\n            best_iou = iou_thr\n            bx1, by1, bx2, by2 = box\n            barea = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)\n            for ci, cl in enumerate(clusters):\n                cx1, cy1, cx2, cy2 = cl[\'fused\']\n                iw = max(0.0, min(bx2, cx2) - max(bx1, cx1))\n                ih = max(0.0, min(by2, cy2) - max(by1, cy1))\n                inter = iw * ih\n                if inter <= 0.0:\n                    continue\n                carea = max(0.0, cx2 - cx1) * max(0.0, cy2 - cy1)\n                iou = inter / max(1e-9, barea + carea - inter)\n                if iou >= best_iou:\n                    best_iou = iou\n                    best_i = ci\n            if best_i >= 0:\n                cl = clusters[best_i]\n                w_sum = cl[\'w_sum\'] + sc\n                cl[\'fused\'] = (cl[\'fused\'] * cl[\'w_sum\'] + box * sc) / max(1e-9, w_sum)\n                cl[\'w_sum\'] = w_sum\n                cl[\'scores\'].append(sc)\n                cl[\'views\'].add(view_id)\n            else:\n                clusters.append({\'fused\': box.copy(), \'w_sum\': sc, \'scores\': [sc], \'views\': {view_id}})\n        for cl in clusters:\n            # Penalize single-view-only boxes slightly by N_views / 2.0 (standard WBF with 2 views)\n            fused_sc = float(sum(cl[\'scores\']) / 2.0)\n            fb = cl[\'fused\']\n            merged.append((cls_id, fused_sc, float(fb[0]), float(fb[1]), float(fb[2]), float(fb[3])))\n    merged.sort(key=lambda t: t[1], reverse=True)\n    return merged[:max_det]\n\n# 2. Vectorized Sky-Crop Retention Audit on all 20,078 GT boxes\ndef _build_image_index(split_name: str, lbl_df: pd.DataFrame):\n    img_dir = KIT_LOCAL / f\'{split_name}_images\'\n    img_paths = sorted(img_dir.glob(\'*.jpg\'))\n    def _read_dim(p: Path):\n        with Image.open(p) as im:\n            return p.stem, im.size[0], im.size[1], str(p)\n    with ThreadPoolExecutor(max_workers=16) as pool:\n        rows = list(pool.map(_read_dim, img_paths, chunksize=64))\n    dims_df = pd.DataFrame(rows, columns=[\'image_id\', \'w\', \'h\', \'path\'])\n    grouped = {}\n    for iid, grp in lbl_df.groupby(\'image_id\'):\n        grouped[iid] = (\n            grp[[\'x_min\', \'y_min\', \'x_max\', \'y_max\']].to_numpy(dtype=np.float32).tolist(),\n            grp[\'class_id\'].to_numpy(dtype=np.int64).tolist(),\n        )\n    records = []\n    n_total_boxes, n_kept_boxes, n_clipped_boxes = 0, 0, 0\n    for row in rows:\n        iid, w, h, p_str = row\n        boxes, labels = grouped.get(iid, ([], []))\n        y_off = compute_sky_crop_offset(h, SKY_CROP_FRAC)\n        kb, _ = shift_boxes_for_sky_crop(boxes, labels, h, y_off)\n        n_total_boxes += len(boxes)\n        n_kept_boxes += len(kb)\n        for bx in boxes:\n            if float(bx[1]) < float(y_off) < float(bx[3]):\n                n_clipped_boxes += 1\n        records.append({\'image_id\': iid, \'w\': int(w), \'h\': int(h), \'path\': p_str, \'boxes\': boxes, \'labels\': labels})\n    pct_kept = 100.0 * n_kept_boxes / max(1, n_total_boxes)\n    print(f\'[{split_name}] Top-{int(SKY_CROP_FRAC*100)}% Sky-Crop Audit: {len(records)} imgs | GT kept {n_kept_boxes}/{n_total_boxes} ({pct_kept:.2f}%), upper-clipped {n_clipped_boxes}\')\n    return records, dims_df\n\ntrain_records, train_dims_df = _build_image_index(\'train\', train_df)\nval_records, val_dims_df = _build_image_index(\'val\', val_df)\n\n# 3. Road-Safe + EDA-Calibrated Dataset & DamagedGuaranteedBatchSampler\nROB_P = float(EXP_CFG.get(\'robustness_aug_p\', 0.30))\nCLAHE_TRAIN_P = float(EXP_CFG.get(\'clahe_train_p\', 0.35))\nCROP_JITTER_MIN = float(EXP_CFG.get(\'sky_crop_jitter_min\', 0.18))\nCROP_JITTER_MAX = float(EXP_CFG.get(\'sky_crop_jitter_max\', 0.24))\n\nclass RoadDamageDFineDataset(Dataset):\n    def __init__(self, records, imgsz: int, is_train: bool = True, override_dir: Path = None):\n        self.records = records\n        self.imgsz = int(imgsz)\n        self.is_train = bool(is_train)\n        self.override_dir = override_dir\n        if self.is_train:\n            self.geom_color_aug = A.Compose([\n                A.HorizontalFlip(p=0.5),\n                A.Affine(scale=(0.88, 1.12), translate_percent={\'x\': (-0.05, 0.05), \'y\': (-0.04, 0.04)}, rotate=0.0, border_mode=cv2.BORDER_CONSTANT, p=0.45),\n                A.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.25, hue=0.015, p=0.45),\n                A.OneOf([\n                    A.ImageCompression(compression_type=\'jpeg\', quality_range=(6, 22), p=1.0),\n                    A.Compose([\n                        A.RandomBrightnessContrast(brightness_limit=(-0.75, -0.45), contrast_limit=(-0.35, 0.0), p=1.0),\n                        A.RandomGamma(gamma_limit=(140, 220), p=0.5),\n                    ], p=1.0),\n                    A.OneOf([\n                        A.GaussianBlur(blur_limit=(7, 15), p=1.0),\n                        A.MotionBlur(blur_limit=(7, 15), p=1.0),\n                    ], p=1.0),\n                    A.GaussNoise(std_range=(0.05, 0.15), p=1.0),\n                ], p=ROB_P),\n            ], bbox_params=A.BboxParams(format=\'pascal_voc\', label_fields=[\'class_labels\'], clip=True, min_area=12.0))\n        else:\n            self.geom_color_aug = None\n\n    def __len__(self) -> int:\n        return len(self.records)\n\n    def __getitem__(self, idx: int):\n        rec = self.records[idx]\n        img_path = (self.override_dir / f"{rec[\'image_id\']}.jpg") if self.override_dir is not None else Path(rec[\'path\'])\n        with Image.open(img_path) as im:\n            img = np.asarray(im.convert(\'RGB\'))\n        orig_h, orig_w = img.shape[0], img.shape[1]\n\n        frac = random.uniform(CROP_JITTER_MIN, CROP_JITTER_MAX) if self.is_train else SKY_CROP_FRAC\n        y_off = compute_sky_crop_offset(orig_h, frac)\n        crop = img[y_off:, :, :]\n        crop_h, crop_w = crop.shape[0], crop.shape[1]\n\n        boxes_px, labels = shift_boxes_for_sky_crop(rec[\'boxes\'], rec[\'labels\'], orig_h, y_off)\n\n        if self.is_train:\n            if random.random() < CLAHE_TRAIN_P:\n                crop = apply_lab_clahe_rgb(crop, clip_limit=2.0, grid_size=8)\n            aug = self.geom_color_aug(image=crop, bboxes=boxes_px, class_labels=labels)\n            crop = aug[\'image\']\n            boxes_px = [list(b) for b in aug[\'bboxes\']]\n            labels = [int(c) for c in aug[\'class_labels\']]\n\n        resized = cv2.resize(crop, (self.imgsz, self.imgsz), interpolation=cv2.INTER_LINEAR)\n        px_tensor = torch.from_numpy(np.ascontiguousarray(resized)).permute(2, 0, 1).to(torch.float32).div_(255.0)\n\n        if boxes_px:\n            arr = np.asarray(boxes_px, dtype=np.float32)\n            # Normalized (cx, cy, w, h) relative to cropped dimensions (crop_w, crop_h)\n            cx = np.clip((arr[:, 0] + arr[:, 2]) * 0.5 / float(crop_w), 0.0, 1.0)\n            cy = np.clip((arr[:, 1] + arr[:, 3]) * 0.5 / float(crop_h), 0.0, 1.0)\n            bw = np.clip((arr[:, 2] - arr[:, 0]) / float(crop_w), 1e-4, 1.0)\n            bh = np.clip((arr[:, 3] - arr[:, 1]) / float(crop_h), 1e-4, 1.0)\n            norm_boxes = torch.from_numpy(np.stack([cx, cy, bw, bh], axis=1)).to(torch.float32)\n            cls_tensor = torch.tensor(labels, dtype=torch.long)\n        else:\n            norm_boxes = torch.zeros((0, 4), dtype=torch.float32)\n            cls_tensor = torch.zeros((0,), dtype=torch.long)\n\n        return {\n            \'pixel_values\': px_tensor,\n            \'target\': {\'class_labels\': cls_tensor, \'boxes\': norm_boxes},\n            \'image_id\': rec[\'image_id\'],\n            \'orig_size\': (orig_h, orig_w),\n            \'crop_size\': (crop_h, crop_w),\n            \'y_off\': y_off,\n        }\n\ndef dfine_collate_fn(batch):\n    return {\n        \'pixel_values\': torch.stack([b[\'pixel_values\'] for b in batch], dim=0),\n        \'labels\': [b[\'target\'] for b in batch],\n        \'image_ids\': [b[\'image_id\'] for b in batch],\n        \'orig_sizes\': [b[\'orig_size\'] for b in batch],\n        \'crop_sizes\': [b[\'crop_size\'] for b in batch],\n        \'y_offs\': [b[\'y_off\'] for b in batch],\n    }\n\nclass DamagedGuaranteedBatchSampler(Sampler):\n    """Visits 100% of train images once per epoch while guaranteeing >=1 damaged image in every batch."""\n    def __init__(self, damaged_indices: list, clean_indices: list, batch_size: int, seed: int = 42):\n        self.damaged = list(damaged_indices)\n        self.clean = list(clean_indices)\n        self.batch_size = int(batch_size)\n        self.seed = int(seed)\n        self.epoch = 0\n        self.total = len(self.damaged) + len(self.clean)\n        self.num_batches = max(1, math.ceil(self.total / self.batch_size))\n        if len(self.damaged) < self.num_batches:\n            raise ValueError(f\'Need damaged >= num_batches ({len(self.damaged)} < {self.num_batches})\')\n\n    def set_epoch(self, ep: int):\n        self.epoch = int(ep)\n\n    def __len__(self) -> int:\n        return self.num_batches\n\n    def __iter__(self):\n        rng = random.Random(self.seed + self.epoch * 9973)\n        dam = list(self.damaged)\n        cln = list(self.clean)\n        rng.shuffle(dam)\n        rng.shuffle(cln)\n        # Anchor 1 guaranteed damaged image into every single batch (including last batch)\n        anchors = dam[:self.num_batches]\n        rest = dam[self.num_batches:] + cln\n        rng.shuffle(rest)\n        batches = []\n        cursor = 0\n        for b_idx in range(self.num_batches):\n            cap = self.batch_size - 1\n            if b_idx == self.num_batches - 1:\n                take = len(rest) - cursor\n            else:\n                take = min(cap, max(0, len(rest) - cursor))\n            batch = [anchors[b_idx]] + rest[cursor:cursor + take]\n            cursor += take\n            rng.shuffle(batch)\n            batches.append(batch)\n        return iter(batches)\n\ntrain_ds = RoadDamageDFineDataset(train_records, IMGSZ, is_train=True)\nval_ds = RoadDamageDFineDataset(val_records, IMGSZ, is_train=False)\ndamaged_idx = [i for i, r in enumerate(train_records) if len(r[\'boxes\']) > 0]\nclean_idx = [i for i, r in enumerate(train_records) if len(r[\'boxes\']) == 0]\ntrain_sampler = DamagedGuaranteedBatchSampler(damaged_idx, clean_idx, BATCH_SIZE, seed=SEED)\ntrain_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=DATALOADER_WORKERS, pin_memory=True, persistent_workers=(DATALOADER_WORKERS > 0), collate_fn=dfine_collate_fn)\nval_loader = DataLoader(val_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=DATALOADER_WORKERS, pin_memory=True, collate_fn=dfine_collate_fn)\nprint(f\'DataLoaders ready -> train batches={len(train_loader)} (damaged={len(damaged_idx)}, clean={len(clean_idx)}), val batches={len(val_loader)}\')\n\n# 4. Build exact deterministic 4-bucket degraded val set matching v3\nVAL_DEGRADED_DIR = WORK_ROOT / \'val_images_degraded\'\nVAL_DEGRADED_DIR.mkdir(parents=True, exist_ok=True)\nval_jpgs = sorted((KIT_LOCAL / \'val_images\').glob(\'*.jpg\'))\n\n_STRESS_BUCKETS = [\n    lambda s: A.Compose([A.ImageCompression(compression_type=\'jpeg\', quality_range=(8, 18), p=1.0)], seed=s),\n    lambda s: A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.72, -0.48), contrast_limit=(-0.30, 0.0), p=1.0)], seed=s),\n    lambda s: A.Compose([A.OneOf([A.GaussianBlur(blur_limit=(9, 13), p=1.0), A.MotionBlur(blur_limit=(9, 13), p=1.0)], p=1.0)], seed=s),\n    lambda s: A.Compose([A.GaussNoise(std_range=(0.06, 0.13), p=1.0)], seed=s),\n]\n\ndef _degrade_one(idx_path):\n    idx, p = idx_path\n    out_p = VAL_DEGRADED_DIR / p.name\n    if out_p.is_file():\n        return 1\n    pipe = _STRESS_BUCKETS[idx % 4](SEED + idx)\n    arr = np.asarray(Image.open(p).convert(\'RGB\'))\n    deg = pipe(image=arr)[\'image\']\n    Image.fromarray(deg).save(out_p, format=\'JPEG\', quality=88)\n    return 1\n\nwith ThreadPoolExecutor(max_workers=8) as pool:\n    list(pool.map(_degrade_one, enumerate(val_jpgs), chunksize=32))\nprint(f\'Built 4-bucket degraded val set ({len(val_jpgs)} imgs) at {VAL_DEGRADED_DIR}\')\ncheck_budget()')


## 4. Train D-FINE-M (Differential LR, Warmup+Cosine, EMA, Official Val & Resume)
Fine-tune COCO D-FINE-M with backbone/head differential AdamW LRs, ModelEMA (0.9998), grad clipping (0.1), official score.py validation checkpoints, and full resumable state.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
class ModelEMA:
    def __init__(self, model: torch.nn.Module, decay: float = 0.9998):
        self.decay = float(decay)
        self.updates = 0
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.backup = None

    @torch.no_grad()
    def update(self, model: torch.nn.Module):
        self.updates += 1
        d = self.decay * (1.0 - math.exp(-self.updates / 2000.0))
        msd = model.state_dict()
        for k, v in msd.items():
            if not torch.is_floating_point(v):
                self.shadow[k].copy_(v)
            else:
                self.shadow[k].mul_(d).add_(v.detach(), alpha=1.0 - d)

    def apply_shadow(self, model: torch.nn.Module):
        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow, strict=True)

    def restore(self, model: torch.nn.Module):
        if self.backup is not None:
            model.load_state_dict(self.backup, strict=True)
            self.backup = None

@torch.no_grad()
def _evaluate_dfine_to_csv(eval_model, loader, out_csv: Path, conf_floor: float = 0.001, max_det: int = 100):
    eval_model.eval()
    rows = []
    for batch in loader:
        px = batch['pixel_values'].to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(AMP_DTYPE is not None)):
            outputs = eval_model(pixel_values=px)
        outputs.logits = outputs.logits.float()
        outputs.pred_boxes = outputs.pred_boxes.float()
        crop_sizes = torch.tensor(batch['crop_sizes'], dtype=torch.long)
        decoded = IMAGE_PROCESSOR.post_process_object_detection(outputs, target_sizes=crop_sizes, threshold=conf_floor)
        for iid, orig_sz, y_off, res in zip(batch['image_ids'], batch['orig_sizes'], batch['y_offs'], decoded):
            scores = res['scores'].detach().float().cpu().numpy()
            labels = res['labels'].detach().cpu().numpy().astype(int)
            boxes = res['boxes'].detach().float().cpu().numpy()
            if len(scores) > max_det:
                keep = np.argsort(-scores)[:max_det]
                scores, labels, boxes = scores[keep], labels[keep], boxes[keep]
            boxes = restore_boxes_from_sky_crop(boxes, orig_sz[0], orig_sz[1], y_off)
            for cls_id, sc, bx in zip(labels, scores, boxes):
                cid_i = int(cls_id)
                sc_f = max(0.0, min(1.0, float(sc)))
                x1, y1, x2, y2 = float(bx[0]), float(bx[1]), float(bx[2]), float(bx[3])
                if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (x2 - x1) >= 0.5 and (y2 - y1) >= 0.5:
                    rows.append([iid, cid_i, f'{sc_f:.6f}', f'{x1:.2f}', f'{y1:.2f}', f'{x2:.2f}', f'{y2:.2f}'])
    with out_csv.open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['image_id', 'class_id', 'score', 'x_min', 'y_min', 'x_max', 'y_max'])
        w.writerows(rows)

def _run_official_score_py(pred_csv: Path, score_json: Path, speed_json: Path = None) -> dict:
    cmd = [
        sys.executable, str(KIT_LOCAL / 'score.py'),
        '--pred', str(pred_csv),
        '--gt', str(KIT_LOCAL / 'val_labels.csv'),
        '--image-ids', str(KIT_LOCAL / 'val_image_ids.csv'),
        '--json-out', str(score_json),
    ]
    if speed_json is not None and speed_json.is_file():
        cmd.extend(['--speed-json', str(speed_json)])
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.returncode != 0:
        if res.stdout:
            print('score.py stdout:\n', res.stdout)
        if res.stderr:
            print('score.py stderr:\n', res.stderr)
        res.check_returncode()
    return json.loads(score_json.read_text(encoding='utf-8'))

model = DFineForObjectDetection.from_pretrained(
    MODEL_REPO,
    id2label=CLASS_MAP,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    cache_dir=str(HF_CACHE_DIR),
    **_TF_DTYPE_KW,
).to(DEVICE)

lr_head = float(EXP_CFG.get('lr_head', 2e-4))
lr_backbone = float(EXP_CFG.get('lr_backbone', 2e-5))
weight_decay = float(EXP_CFG.get('weight_decay', 1e-4))
max_grad_norm = float(EXP_CFG.get('max_grad_norm', 0.1))
ema_decay = float(EXP_CFG.get('ema_decay', 0.9998))
eval_every = int(EXP_CFG.get('eval_every_epochs', 3))
conf_floor = float(EXP_CFG.get('conf_floor', 0.001))
max_det = int(EXP_CFG.get('max_det', 100))

param_groups = [
    {'params': [p for n, p in model.named_parameters() if 'backbone' in n and p.requires_grad], 'lr': lr_backbone, 'weight_decay': weight_decay},
    {'params': [p for n, p in model.named_parameters() if 'backbone' not in n and p.requires_grad], 'lr': lr_head, 'weight_decay': weight_decay},
]
optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))
steps_per_epoch = max(1, math.ceil(len(train_loader) / GRAD_ACCUM))
total_opt_steps = max(1, TARGET_EPOCHS * steps_per_epoch)
warmup_steps = min(total_opt_steps // 2, int(round(float(EXP_CFG.get('warmup_epochs', 1.5)) * steps_per_epoch)))
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_opt_steps)
scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))
ema = ModelEMA(model, decay=ema_decay)

BEST_DIR_LOCAL = WORK_ROOT / 'dfine_m_best'
BEST_DIR_DRIVE = RUN_DRIVE_DIR / 'weights' / 'dfine_m_best'
LAST_CKPT_LOCAL = WORK_ROOT / 'last_resume_ckpt.pt'
LAST_CKPT_DRIVE = RUN_DRIVE_DIR / 'checkpoints' / 'last_resume_ckpt.pt'
RESULTS_CSV_LOCAL = WORK_ROOT / 'results.csv'
RESULTS_CSV_DRIVE = RUN_DRIVE_DIR / 'weights' / 'results.csv'

start_epoch = 1
best_acc = -1.0
history_rows = []
nonfinite_skips = 0

if LAST_CKPT_DRIVE.is_file():
    print('Found resumable checkpoint, restoring state from:', LAST_CKPT_DRIVE)
    ckpt = torch.load(LAST_CKPT_DRIVE, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'])
    ema.shadow = {k: v.to(DEVICE) for k, v in ckpt['ema_shadow'].items()}
    ema.updates = int(ckpt['ema_updates'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if 'scaler' in ckpt and ckpt['scaler'] is not None:
        scaler.load_state_dict(ckpt['scaler'])
    start_epoch = int(ckpt['epoch']) + 1
    best_acc = float(ckpt.get('best_acc', -1.0))
    history_rows = list(ckpt.get('history', []))

t_train_start = time.monotonic()
for epoch in range(start_epoch, TARGET_EPOCHS + 1):
    t_ep = time.monotonic()
    train_sampler.set_epoch(epoch)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    ep_loss_sum, ep_steps = 0.0, 0

    for step_idx, batch in enumerate(train_loader, start=1):
        px = batch['pixel_values'].to(DEVICE, non_blocking=True)
        lbls = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in batch['labels']]
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(AMP_DTYPE is not None)):
            out = model(pixel_values=px, labels=lbls)
            raw_loss = out.loss
            loss = raw_loss / float(GRAD_ACCUM)

        if not torch.isfinite(raw_loss):
            nonfinite_skips += 1
            optimizer.zero_grad(set_to_none=True)
            continue

        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        ep_loss_sum += float(raw_loss.detach().item())
        ep_steps += 1

        if (step_idx % GRAD_ACCUM == 0) or (step_idx == len(train_loader)):
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            if torch.isfinite(grad_norm):
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                ema.update(model)
            else:
                nonfinite_skips += 1
                if scaler.is_enabled():
                    scaler.update()
            optimizer.zero_grad(set_to_none=True)

    mean_loss = ep_loss_sum / max(1, ep_steps)
    cur_lr = float(optimizer.param_groups[1]['lr'])
    ep_row = {
        'epoch': epoch,
        'train_loss': round(mean_loss, 5),
        'lr_head': cur_lr,
        'lr_backbone': float(optimizer.param_groups[0]['lr']),
        'elapsed_s': round(time.monotonic() - t_train_start, 2),
        'epoch_s': round(time.monotonic() - t_ep, 2),
        'nonfinite_skips': int(nonfinite_skips),
        'val_accuracy_score': None,
        'val_mAP50': None,
        'val_mAP50_95': None,
        'val_P': None,
        'val_R': None,
    }

    do_eval = (epoch % eval_every == 0) or (epoch >= max(1, TARGET_EPOCHS - 2)) or (epoch == 1)
    if do_eval:
        ema.apply_shadow(model)
        val_csv_ep = WORK_ROOT / f'val_pred_ep{epoch}.csv'
        val_json_ep = WORK_ROOT / f'val_score_ep{epoch}.json'
        _evaluate_dfine_to_csv(model, val_loader, val_csv_ep, conf_floor=conf_floor, max_det=max_det)
        scr_ep = _run_official_score_py(val_csv_ep, val_json_ep)
        acc_ep = float(scr_ep['combined']['accuracy_score'])
        m_ep = scr_ep['main']
        ep_row.update({
            'val_accuracy_score': round(acc_ep, 4),
            'val_mAP50': round(float(m_ep['mAP50']), 5),
            'val_mAP50_95': round(float(m_ep['mAP50_95']), 5),
            'val_P': round(float(m_ep['precision']), 5),
            'val_R': round(float(m_ep['recall']), 5),
        })
        print(f'Epoch {epoch:02d}/{TARGET_EPOCHS} | loss={mean_loss:.4f} | lr={cur_lr:.2e} | Val Acc={acc_ep:.2f} | mAP50={m_ep["mAP50"]:.4f} | mAP50:95={m_ep["mAP50_95"]:.4f} | P={m_ep["precision"]:.4f} R={m_ep["recall"]:.4f} | {ep_row["epoch_s"]:.1f}s')

        if acc_ep > best_acc:
            best_acc = acc_ep
            if BEST_DIR_LOCAL.exists():
                shutil.rmtree(BEST_DIR_LOCAL)
            BEST_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(BEST_DIR_LOCAL))
            IMAGE_PROCESSOR.save_pretrained(str(BEST_DIR_LOCAL))
            if BEST_DIR_DRIVE.exists():
                shutil.rmtree(BEST_DIR_DRIVE)
            shutil.copytree(BEST_DIR_LOCAL, BEST_DIR_DRIVE)
            print(f'  -> Saved NEW BEST D-FINE-M EMA weights (Acc={best_acc:.2f}) to Drive.')
        ema.restore(model)
    else:
        print(f'Epoch {epoch:02d}/{TARGET_EPOCHS} | loss={mean_loss:.4f} | lr={cur_lr:.2e} | {ep_row["epoch_s"]:.1f}s (eval skipped)')

    history_rows.append(ep_row)
    pd.DataFrame(history_rows).to_csv(RESULTS_CSV_LOCAL, index=False)
    shutil.copy2(RESULTS_CSV_LOCAL, RESULTS_CSV_DRIVE)

    # Save full resumable checkpoint periodically + at final epoch
    if (epoch % 3 == 0) or (epoch == TARGET_EPOCHS):
        ckpt_payload = {
            'epoch': epoch,
            'best_acc': best_acc,
            'model': model.state_dict(),
            'ema_shadow': ema.shadow,
            'ema_updates': ema.updates,
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict() if scaler.is_enabled() else None,
            'history': history_rows,
            'config': EXP_CFG,
        }
        torch.save(ckpt_payload, LAST_CKPT_LOCAL)
        shutil.copy2(LAST_CKPT_LOCAL, LAST_CKPT_DRIVE)
    check_budget()

if not BEST_DIR_LOCAL.is_dir():
    raise FileNotFoundError(f'Missing best D-FINE-M directory at {BEST_DIR_LOCAL}')

REPO_DIR = WORK_ROOT / 'submission_repo'
(REPO_DIR / 'weights').mkdir(parents=True, exist_ok=True)
if (REPO_DIR / 'weights' / 'dfine_m_best').exists():
    shutil.rmtree(REPO_DIR / 'weights' / 'dfine_m_best')
shutil.copytree(BEST_DIR_LOCAL, REPO_DIR / 'weights' / 'dfine_m_best')

ckpt_rel_files = [
    f'03_results/runs/{RUN_ID}/weights/dfine_m_best/model.safetensors',
    f'03_results/runs/{RUN_ID}/weights/dfine_m_best/config.json',
    f'03_results/runs/{RUN_ID}/weights/results.csv',
]
manifest_best = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, 'best_model', CHECKPOINT_CONFIG, ckpt_rel_files)
resume_rel_files = ckpt_rel_files + ([f'03_results/runs/{RUN_ID}/checkpoints/last_resume_ckpt.pt'] if LAST_CKPT_DRIVE.is_file() else [])
manifest_resume = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, 'last_resume', CHECKPOINT_CONFIG, resume_rel_files)
TASK_ARTIFACTS['best_weights_dir'] = f'03_results/runs/{RUN_ID}/weights/dfine_m_best'
TASK_ARTIFACTS['checkpoint_manifest'] = str(manifest_best.relative_to(PROJECT_ROOT))
TASK_ARTIFACTS['resume_manifest'] = str(manifest_resume.relative_to(PROJECT_ROOT))
TASK_ARTIFACTS['gpu_tier'] = GPU_TIER
print('Saved best EMA checkpoint & full resume manifest.')
```


In [ ]:
_execute_stage(3, 'class ModelEMA:\n    def __init__(self, model: torch.nn.Module, decay: float = 0.9998):\n        self.decay = float(decay)\n        self.updates = 0\n        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        self.backup = None\n\n    @torch.no_grad()\n    def update(self, model: torch.nn.Module):\n        self.updates += 1\n        d = self.decay * (1.0 - math.exp(-self.updates / 2000.0))\n        msd = model.state_dict()\n        for k, v in msd.items():\n            if not torch.is_floating_point(v):\n                self.shadow[k].copy_(v)\n            else:\n                self.shadow[k].mul_(d).add_(v.detach(), alpha=1.0 - d)\n\n    def apply_shadow(self, model: torch.nn.Module):\n        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        model.load_state_dict(self.shadow, strict=True)\n\n    def restore(self, model: torch.nn.Module):\n        if self.backup is not None:\n            model.load_state_dict(self.backup, strict=True)\n            self.backup = None\n\n@torch.no_grad()\ndef _evaluate_dfine_to_csv(eval_model, loader, out_csv: Path, conf_floor: float = 0.001, max_det: int = 100):\n    eval_model.eval()\n    rows = []\n    for batch in loader:\n        px = batch[\'pixel_values\'].to(DEVICE, non_blocking=True)\n        with torch.amp.autocast(\'cuda\', dtype=AMP_DTYPE, enabled=(AMP_DTYPE is not None)):\n            outputs = eval_model(pixel_values=px)\n        outputs.logits = outputs.logits.float()\n        outputs.pred_boxes = outputs.pred_boxes.float()\n        crop_sizes = torch.tensor(batch[\'crop_sizes\'], dtype=torch.long)\n        decoded = IMAGE_PROCESSOR.post_process_object_detection(outputs, target_sizes=crop_sizes, threshold=conf_floor)\n        for iid, orig_sz, y_off, res in zip(batch[\'image_ids\'], batch[\'orig_sizes\'], batch[\'y_offs\'], decoded):\n            scores = res[\'scores\'].detach().float().cpu().numpy()\n            labels = res[\'labels\'].detach().cpu().numpy().astype(int)\n            boxes = res[\'boxes\'].detach().float().cpu().numpy()\n            if len(scores) > max_det:\n                keep = np.argsort(-scores)[:max_det]\n                scores, labels, boxes = scores[keep], labels[keep], boxes[keep]\n            boxes = restore_boxes_from_sky_crop(boxes, orig_sz[0], orig_sz[1], y_off)\n            for cls_id, sc, bx in zip(labels, scores, boxes):\n                cid_i = int(cls_id)\n                sc_f = max(0.0, min(1.0, float(sc)))\n                x1, y1, x2, y2 = float(bx[0]), float(bx[1]), float(bx[2]), float(bx[3])\n                if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (x2 - x1) >= 0.5 and (y2 - y1) >= 0.5:\n                    rows.append([iid, cid_i, f\'{sc_f:.6f}\', f\'{x1:.2f}\', f\'{y1:.2f}\', f\'{x2:.2f}\', f\'{y2:.2f}\'])\n    with out_csv.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        w = csv.writer(f)\n        w.writerow([\'image_id\', \'class_id\', \'score\', \'x_min\', \'y_min\', \'x_max\', \'y_max\'])\n        w.writerows(rows)\n\ndef _run_official_score_py(pred_csv: Path, score_json: Path, speed_json: Path = None) -> dict:\n    cmd = [\n        sys.executable, str(KIT_LOCAL / \'score.py\'),\n        \'--pred\', str(pred_csv),\n        \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'),\n        \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'),\n        \'--json-out\', str(score_json),\n    ]\n    if speed_json is not None and speed_json.is_file():\n        cmd.extend([\'--speed-json\', str(speed_json)])\n    res = subprocess.run(cmd, text=True, capture_output=True)\n    if res.returncode != 0:\n        if res.stdout:\n            print(\'score.py stdout:\\n\', res.stdout)\n        if res.stderr:\n            print(\'score.py stderr:\\n\', res.stderr)\n        res.check_returncode()\n    return json.loads(score_json.read_text(encoding=\'utf-8\'))\n\nmodel = DFineForObjectDetection.from_pretrained(\n    MODEL_REPO,\n    id2label=CLASS_MAP,\n    label2id=LABEL2ID,\n    ignore_mismatched_sizes=True,\n    cache_dir=str(HF_CACHE_DIR),\n    **_TF_DTYPE_KW,\n).to(DEVICE)\n\nlr_head = float(EXP_CFG.get(\'lr_head\', 2e-4))\nlr_backbone = float(EXP_CFG.get(\'lr_backbone\', 2e-5))\nweight_decay = float(EXP_CFG.get(\'weight_decay\', 1e-4))\nmax_grad_norm = float(EXP_CFG.get(\'max_grad_norm\', 0.1))\nema_decay = float(EXP_CFG.get(\'ema_decay\', 0.9998))\neval_every = int(EXP_CFG.get(\'eval_every_epochs\', 3))\nconf_floor = float(EXP_CFG.get(\'conf_floor\', 0.001))\nmax_det = int(EXP_CFG.get(\'max_det\', 100))\n\nparam_groups = [\n    {\'params\': [p for n, p in model.named_parameters() if \'backbone\' in n and p.requires_grad], \'lr\': lr_backbone, \'weight_decay\': weight_decay},\n    {\'params\': [p for n, p in model.named_parameters() if \'backbone\' not in n and p.requires_grad], \'lr\': lr_head, \'weight_decay\': weight_decay},\n]\noptimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))\nsteps_per_epoch = max(1, math.ceil(len(train_loader) / GRAD_ACCUM))\ntotal_opt_steps = max(1, TARGET_EPOCHS * steps_per_epoch)\nwarmup_steps = min(total_opt_steps // 2, int(round(float(EXP_CFG.get(\'warmup_epochs\', 1.5)) * steps_per_epoch)))\nscheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_opt_steps)\nscaler = torch.amp.GradScaler(\'cuda\', enabled=(AMP_DTYPE == torch.float16))\nema = ModelEMA(model, decay=ema_decay)\n\nBEST_DIR_LOCAL = WORK_ROOT / \'dfine_m_best\'\nBEST_DIR_DRIVE = RUN_DRIVE_DIR / \'weights\' / \'dfine_m_best\'\nLAST_CKPT_LOCAL = WORK_ROOT / \'last_resume_ckpt.pt\'\nLAST_CKPT_DRIVE = RUN_DRIVE_DIR / \'checkpoints\' / \'last_resume_ckpt.pt\'\nRESULTS_CSV_LOCAL = WORK_ROOT / \'results.csv\'\nRESULTS_CSV_DRIVE = RUN_DRIVE_DIR / \'weights\' / \'results.csv\'\n\nstart_epoch = 1\nbest_acc = -1.0\nhistory_rows = []\nnonfinite_skips = 0\n\nif LAST_CKPT_DRIVE.is_file():\n    print(\'Found resumable checkpoint, restoring state from:\', LAST_CKPT_DRIVE)\n    ckpt = torch.load(LAST_CKPT_DRIVE, map_location=DEVICE, weights_only=False)\n    model.load_state_dict(ckpt[\'model\'])\n    ema.shadow = {k: v.to(DEVICE) for k, v in ckpt[\'ema_shadow\'].items()}\n    ema.updates = int(ckpt[\'ema_updates\'])\n    optimizer.load_state_dict(ckpt[\'optimizer\'])\n    scheduler.load_state_dict(ckpt[\'scheduler\'])\n    if \'scaler\' in ckpt and ckpt[\'scaler\'] is not None:\n        scaler.load_state_dict(ckpt[\'scaler\'])\n    start_epoch = int(ckpt[\'epoch\']) + 1\n    best_acc = float(ckpt.get(\'best_acc\', -1.0))\n    history_rows = list(ckpt.get(\'history\', []))\n\nt_train_start = time.monotonic()\nfor epoch in range(start_epoch, TARGET_EPOCHS + 1):\n    t_ep = time.monotonic()\n    train_sampler.set_epoch(epoch)\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    ep_loss_sum, ep_steps = 0.0, 0\n\n    for step_idx, batch in enumerate(train_loader, start=1):\n        px = batch[\'pixel_values\'].to(DEVICE, non_blocking=True)\n        lbls = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in batch[\'labels\']]\n        with torch.amp.autocast(\'cuda\', dtype=AMP_DTYPE, enabled=(AMP_DTYPE is not None)):\n            out = model(pixel_values=px, labels=lbls)\n            raw_loss = out.loss\n            loss = raw_loss / float(GRAD_ACCUM)\n\n        if not torch.isfinite(raw_loss):\n            nonfinite_skips += 1\n            optimizer.zero_grad(set_to_none=True)\n            continue\n\n        if scaler.is_enabled():\n            scaler.scale(loss).backward()\n        else:\n            loss.backward()\n\n        ep_loss_sum += float(raw_loss.detach().item())\n        ep_steps += 1\n\n        if (step_idx % GRAD_ACCUM == 0) or (step_idx == len(train_loader)):\n            if scaler.is_enabled():\n                scaler.unscale_(optimizer)\n            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)\n            if torch.isfinite(grad_norm):\n                if scaler.is_enabled():\n                    scaler.step(optimizer)\n                    scaler.update()\n                else:\n                    optimizer.step()\n                scheduler.step()\n                ema.update(model)\n            else:\n                nonfinite_skips += 1\n                if scaler.is_enabled():\n                    scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n\n    mean_loss = ep_loss_sum / max(1, ep_steps)\n    cur_lr = float(optimizer.param_groups[1][\'lr\'])\n    ep_row = {\n        \'epoch\': epoch,\n        \'train_loss\': round(mean_loss, 5),\n        \'lr_head\': cur_lr,\n        \'lr_backbone\': float(optimizer.param_groups[0][\'lr\']),\n        \'elapsed_s\': round(time.monotonic() - t_train_start, 2),\n        \'epoch_s\': round(time.monotonic() - t_ep, 2),\n        \'nonfinite_skips\': int(nonfinite_skips),\n        \'val_accuracy_score\': None,\n        \'val_mAP50\': None,\n        \'val_mAP50_95\': None,\n        \'val_P\': None,\n        \'val_R\': None,\n    }\n\n    do_eval = (epoch % eval_every == 0) or (epoch >= max(1, TARGET_EPOCHS - 2)) or (epoch == 1)\n    if do_eval:\n        ema.apply_shadow(model)\n        val_csv_ep = WORK_ROOT / f\'val_pred_ep{epoch}.csv\'\n        val_json_ep = WORK_ROOT / f\'val_score_ep{epoch}.json\'\n        _evaluate_dfine_to_csv(model, val_loader, val_csv_ep, conf_floor=conf_floor, max_det=max_det)\n        scr_ep = _run_official_score_py(val_csv_ep, val_json_ep)\n        acc_ep = float(scr_ep[\'combined\'][\'accuracy_score\'])\n        m_ep = scr_ep[\'main\']\n        ep_row.update({\n            \'val_accuracy_score\': round(acc_ep, 4),\n            \'val_mAP50\': round(float(m_ep[\'mAP50\']), 5),\n            \'val_mAP50_95\': round(float(m_ep[\'mAP50_95\']), 5),\n            \'val_P\': round(float(m_ep[\'precision\']), 5),\n            \'val_R\': round(float(m_ep[\'recall\']), 5),\n        })\n        print(f\'Epoch {epoch:02d}/{TARGET_EPOCHS} | loss={mean_loss:.4f} | lr={cur_lr:.2e} | Val Acc={acc_ep:.2f} | mAP50={m_ep["mAP50"]:.4f} | mAP50:95={m_ep["mAP50_95"]:.4f} | P={m_ep["precision"]:.4f} R={m_ep["recall"]:.4f} | {ep_row["epoch_s"]:.1f}s\')\n\n        if acc_ep > best_acc:\n            best_acc = acc_ep\n            if BEST_DIR_LOCAL.exists():\n                shutil.rmtree(BEST_DIR_LOCAL)\n            BEST_DIR_LOCAL.mkdir(parents=True, exist_ok=True)\n            model.save_pretrained(str(BEST_DIR_LOCAL))\n            IMAGE_PROCESSOR.save_pretrained(str(BEST_DIR_LOCAL))\n            if BEST_DIR_DRIVE.exists():\n                shutil.rmtree(BEST_DIR_DRIVE)\n            shutil.copytree(BEST_DIR_LOCAL, BEST_DIR_DRIVE)\n            print(f\'  -> Saved NEW BEST D-FINE-M EMA weights (Acc={best_acc:.2f}) to Drive.\')\n        ema.restore(model)\n    else:\n        print(f\'Epoch {epoch:02d}/{TARGET_EPOCHS} | loss={mean_loss:.4f} | lr={cur_lr:.2e} | {ep_row["epoch_s"]:.1f}s (eval skipped)\')\n\n    history_rows.append(ep_row)\n    pd.DataFrame(history_rows).to_csv(RESULTS_CSV_LOCAL, index=False)\n    shutil.copy2(RESULTS_CSV_LOCAL, RESULTS_CSV_DRIVE)\n\n    # Save full resumable checkpoint periodically + at final epoch\n    if (epoch % 3 == 0) or (epoch == TARGET_EPOCHS):\n        ckpt_payload = {\n            \'epoch\': epoch,\n            \'best_acc\': best_acc,\n            \'model\': model.state_dict(),\n            \'ema_shadow\': ema.shadow,\n            \'ema_updates\': ema.updates,\n            \'optimizer\': optimizer.state_dict(),\n            \'scheduler\': scheduler.state_dict(),\n            \'scaler\': scaler.state_dict() if scaler.is_enabled() else None,\n            \'history\': history_rows,\n            \'config\': EXP_CFG,\n        }\n        torch.save(ckpt_payload, LAST_CKPT_LOCAL)\n        shutil.copy2(LAST_CKPT_LOCAL, LAST_CKPT_DRIVE)\n    check_budget()\n\nif not BEST_DIR_LOCAL.is_dir():\n    raise FileNotFoundError(f\'Missing best D-FINE-M directory at {BEST_DIR_LOCAL}\')\n\nREPO_DIR = WORK_ROOT / \'submission_repo\'\n(REPO_DIR / \'weights\').mkdir(parents=True, exist_ok=True)\nif (REPO_DIR / \'weights\' / \'dfine_m_best\').exists():\n    shutil.rmtree(REPO_DIR / \'weights\' / \'dfine_m_best\')\nshutil.copytree(BEST_DIR_LOCAL, REPO_DIR / \'weights\' / \'dfine_m_best\')\n\nckpt_rel_files = [\n    f\'03_results/runs/{RUN_ID}/weights/dfine_m_best/model.safetensors\',\n    f\'03_results/runs/{RUN_ID}/weights/dfine_m_best/config.json\',\n    f\'03_results/runs/{RUN_ID}/weights/results.csv\',\n]\nmanifest_best = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, \'best_model\', CHECKPOINT_CONFIG, ckpt_rel_files)\nresume_rel_files = ckpt_rel_files + ([f\'03_results/runs/{RUN_ID}/checkpoints/last_resume_ckpt.pt\'] if LAST_CKPT_DRIVE.is_file() else [])\nmanifest_resume = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, \'last_resume\', CHECKPOINT_CONFIG, resume_rel_files)\nTASK_ARTIFACTS[\'best_weights_dir\'] = f\'03_results/runs/{RUN_ID}/weights/dfine_m_best\'\nTASK_ARTIFACTS[\'checkpoint_manifest\'] = str(manifest_best.relative_to(PROJECT_ROOT))\nTASK_ARTIFACTS[\'resume_manifest\'] = str(manifest_resume.relative_to(PROJECT_ROOT))\nTASK_ARTIFACTS[\'gpu_tier\'] = GPU_TIER\nprint(\'Saved best EMA checkpoint & full resume manifest.\')')


## 5. 4-Way Inference A/B Test (LUT / Bright-CLAHE / Flip-TTA+WBF) & Speed Benchmark
Benchmark raw Sky-Crop vs Gamma LUT (<60) vs LUT + Bright-Concrete CLAHE (>155) vs Batched Horizontal Flip-TTA + Class-Aware WBF on official benchmark_speed.py + score.py and export ensemble val_pred_dfine_m.csv.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
luma_thresh = float(EXP_CFG.get('lowlight_luma_thresh', 60.0))
target_luma = float(EXP_CFG.get('lowlight_target_luma', 115.0))
bright_clahe_thresh = float(EXP_CFG.get('bright_clahe_luma_thresh', 155.0))
wbf_iou_thr = float(EXP_CFG.get('wbf_iou_thr', 0.55))

def _make_dfine_inference_py(enable_lut: bool, enable_bright_clahe: bool, enable_flip_tta: bool) -> str:
    return f'''import math
import os
import cv2
import numpy as np
import torch
import transformers
from transformers import AutoImageProcessor, DFineForObjectDetection

_TF_MAJOR = int(transformers.__version__.split(".")[0])
_TF_DTYPE_KW = {{"dtype": torch.float32}} if _TF_MAJOR >= 5 else {{"torch_dtype": torch.float32}}

_HERE = os.path.dirname(os.path.abspath(__file__))
WEIGHTS_DIR = os.path.join(_HERE, "weights", "dfine_m_best")
IMGSZ = {IMGSZ}
SKY_CROP_FRAC = {SKY_CROP_FRAC}
CONF_FLOOR = {conf_floor}
MAX_DET = {max_det}
ENABLE_LUT = {enable_lut!r}
ENABLE_BRIGHT_CLAHE = {enable_bright_clahe!r}
ENABLE_FLIP_TTA = {enable_flip_tta!r}
LUMA_THRESH = {luma_thresh}
TARGET_LUMA = {target_luma}
BRIGHT_CLAHE_THRESH = {bright_clahe_thresh}
WBF_IOU_THR = {wbf_iou_thr}

def _build_lut():
    x = np.arange(256, dtype=np.float32) / 255.0
    tgt = math.log(TARGET_LUMA / 255.0)
    luts = {{}}
    for lum in range(int(LUMA_THRESH)):
        gamma = max(0.35, min(0.85, tgt / math.log(max(float(lum), 8.0) / 255.0)))
        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)
    return luts

def _apply_clahe_rgb(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    l_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l_ch)
    return cv2.cvtColor(cv2.merge((l_eq, a_ch, b_ch)), cv2.COLOR_LAB2RGB)

def _wbf_two_views(ba, sa, la, bb, sb, lb, iou_thr, max_det):
    out = []
    classes = sorted(set([int(c) for c in la] + [int(c) for c in lb]))
    for cid in classes:
        cands = []
        for b, s, c in zip(ba, sa, la):
            if int(c) == cid:
                cands.append((float(s), np.asarray(b, dtype=np.float64)))
        for b, s, c in zip(bb, sb, lb):
            if int(c) == cid:
                cands.append((float(s), np.asarray(b, dtype=np.float64)))
        cands.sort(key=lambda t: t[0], reverse=True)
        clusters = []
        for sc, box in cands:
            best_i, best_iou = -1, iou_thr
            bx1, by1, bx2, by2 = box
            barea = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
            for ci, cl in enumerate(clusters):
                cx1, cy1, cx2, cy2 = cl["fused"]
                iw = max(0.0, min(bx2, cx2) - max(bx1, cx1))
                ih = max(0.0, min(by2, cy2) - max(by1, cy1))
                inter = iw * ih
                if inter <= 0.0:
                    continue
                carea = max(0.0, cx2 - cx1) * max(0.0, cy2 - cy1)
                iou = inter / max(1e-9, barea + carea - inter)
                if iou >= best_iou:
                    best_iou, best_i = iou, ci
            if best_i >= 0:
                cl = clusters[best_i]
                ws = cl["w_sum"] + sc
                cl["fused"] = (cl["fused"] * cl["w_sum"] + box * sc) / max(1e-9, ws)
                cl["w_sum"] = ws
                cl["scores"].append(sc)
            else:
                clusters.append({{"fused": box.copy(), "w_sum": sc, "scores": [sc]}})
        for cl in clusters:
            fb = cl["fused"]
            out.append((cid, float(sum(cl["scores"]) / 2.0), float(fb[0]), float(fb[1]), float(fb[2]), float(fb[3])))
    out.sort(key=lambda t: t[1], reverse=True)
    return out[:max_det]

class _DFineHolder:
    def __init__(self, model, processor, device, amp_dtype, luts):
        self.model = model
        self.processor = processor
        self.device = device
        self.amp_dtype = amp_dtype
        self.luts = luts

def load(device="cuda"):
    dev = torch.device(device if (torch.cuda.is_available() and str(device).startswith("cuda")) else "cpu")
    proc = AutoImageProcessor.from_pretrained(WEIGHTS_DIR, local_files_only=True)
    net = DFineForObjectDetection.from_pretrained(WEIGHTS_DIR, local_files_only=True, **_TF_DTYPE_KW).to(dev).eval()
    amp_dt = torch.bfloat16 if (dev.type == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if dev.type == "cuda" else None)
    luts = _build_lut() if ENABLE_LUT else None
    return _DFineHolder(net, proc, dev, amp_dt, luts)

@torch.no_grad()
def predict(model, image):
    orig_h, orig_w = int(image.shape[0]), int(image.shape[1])
    y_off = max(0, min(orig_h - 32, int(round(orig_h * SKY_CROP_FRAC))))
    crop = image[y_off:, :, :]
    if model.luts is not None or ENABLE_BRIGHT_CLAHE:
        mean_lum = float(crop[::4, ::4].mean())
        if model.luts is not None and mean_lum < LUMA_THRESH:
            crop = model.luts[max(0, min(int(LUMA_THRESH) - 1, int(mean_lum)))][crop]
        elif ENABLE_BRIGHT_CLAHE and mean_lum > BRIGHT_CLAHE_THRESH:
            crop = _apply_clahe_rgb(crop)

    crop_h, crop_w = int(crop.shape[0]), int(crop.shape[1])
    rz = cv2.resize(crop, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)
    t1 = torch.from_numpy(np.ascontiguousarray(rz)).permute(2, 0, 1).to(torch.float32).div_(255.0)
    if ENABLE_FLIP_TTA:
        t2 = torch.flip(t1, dims=[2])
        px = torch.stack([t1, t2], dim=0).to(model.device)
        t_sizes = torch.tensor([[crop_h, crop_w], [crop_h, crop_w]], dtype=torch.long)
    else:
        px = t1.unsqueeze(0).to(model.device)
        t_sizes = torch.tensor([[crop_h, crop_w]], dtype=torch.long)

    with torch.amp.autocast("cuda", dtype=model.amp_dtype, enabled=(model.device.type == "cuda" and model.amp_dtype is not None)):
        outs = model.model(pixel_values=px)
    outs.logits = outs.logits.float()
    outs.pred_boxes = outs.pred_boxes.float()
    decoded = model.processor.post_process_object_detection(outs, target_sizes=t_sizes, threshold=CONF_FLOOR)

    r0 = decoded[0]
    b0 = r0["boxes"].detach().float().cpu().numpy()
    s0 = r0["scores"].detach().float().cpu().numpy()
    l0 = r0["labels"].detach().cpu().numpy().astype(int)

    if ENABLE_FLIP_TTA and len(decoded) > 1:
        r1 = decoded[1]
        b1 = r1["boxes"].detach().float().cpu().numpy()
        s1 = r1["scores"].detach().float().cpu().numpy()
        l1 = r1["labels"].detach().cpu().numpy().astype(int)
        if len(b1) > 0:
            b1_u = b1.copy()
            b1_u[:, 0] = float(crop_w) - b1[:, 2]
            b1_u[:, 2] = float(crop_w) - b1[:, 0]
        else:
            b1_u = b1
        preds = _wbf_two_views(b0, s0, l0, b1_u, s1, l1, WBF_IOU_THR, MAX_DET)
        out_list = []
        for cid, sc, x1, y1, x2, y2 in preds:
            cid_i = int(cid)
            sc_f = max(0.0, min(1.0, float(sc)))
            fx1 = max(0.0, min(float(orig_w), float(x1)))
            fy1 = max(0.0, min(float(orig_h), float(y1) + float(y_off)))
            fx2 = max(0.0, min(float(orig_w), float(x2)))
            fy2 = max(0.0, min(float(orig_h), float(y2) + float(y_off)))
            if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (fx2 - fx1) >= 0.5 and (fy2 - fy1) >= 0.5:
                out_list.append((cid_i, sc_f, fx1, fy1, fx2, fy2))
        return out_list

    if len(s0) > MAX_DET:
        keep = np.argsort(-s0)[:MAX_DET]
        s0, l0, b0 = s0[keep], l0[keep], b0[keep]
    out_list = []
    for cid, sc, bx in zip(l0, s0, b0):
        cid_i = int(cid)
        sc_f = max(0.0, min(1.0, float(sc)))
        fx1 = max(0.0, min(float(orig_w), float(bx[0])))
        fy1 = max(0.0, min(float(orig_h), float(bx[1]) + float(y_off)))
        fx2 = max(0.0, min(float(orig_w), float(bx[2])))
        fy2 = max(0.0, min(float(orig_h), float(bx[3]) + float(y_off)))
        if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (fx2 - fx1) >= 0.5 and (fy2 - fy1) >= 0.5:
            out_list.append((cid_i, sc_f, fx1, fy1, fx2, fy2))
    return out_list
'''

def _eval_dfine_variant(tag: str, enable_lut: bool, enable_bright_clahe: bool, enable_flip_tta: bool) -> dict:
    code_str = _make_dfine_inference_py(enable_lut, enable_bright_clahe, enable_flip_tta)
    (REPO_DIR / 'inference.py').write_text(code_str, encoding='utf-8')
    v_pred = WORK_ROOT / f'{tag}_val_pred.csv'
    v_spd = WORK_ROOT / f'{tag}_val_speed.json'
    v_scr = WORK_ROOT / f'{tag}_val_score.json'
    s_pred = WORK_ROOT / f'{tag}_stress_pred.csv'
    s_spd = WORK_ROOT / f'{tag}_stress_speed.json'
    s_scr = WORK_ROOT / f'{tag}_stress_score.json'
    subprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(KIT_LOCAL / 'val_images'), '--out-csv', str(v_pred), '--out-json', str(v_spd), '--runs', '2', '--repeat-images', '300'], check=True)
    subprocess.run([sys.executable, str(KIT_LOCAL / 'score.py'), '--pred', str(v_pred), '--gt', str(KIT_LOCAL / 'val_labels.csv'), '--image-ids', str(KIT_LOCAL / 'val_image_ids.csv'), '--speed-json', str(v_spd), '--json-out', str(v_scr)], check=True)
    subprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(VAL_DEGRADED_DIR), '--out-csv', str(s_pred), '--out-json', str(s_spd), '--runs', '1', '--warmup', '5'], check=True)
    subprocess.run([sys.executable, str(KIT_LOCAL / 'score.py'), '--pred', str(s_pred), '--gt', str(KIT_LOCAL / 'val_labels.csv'), '--image-ids', str(KIT_LOCAL / 'val_image_ids.csv'), '--json-out', str(s_scr)], check=True)
    spd = json.loads(v_spd.read_text(encoding='utf-8'))
    scr = json.loads(v_scr.read_text(encoding='utf-8'))
    st_scr = json.loads(s_scr.read_text(encoding='utf-8'))
    m = scr['main']
    rob = float(st_scr['main']['mAP50'])
    spd_sc = float(spd['speed_score'])
    final_sc = 100.0 * (0.30 * float(m['mAP50']) + 0.25 * float(m['mAP50_95']) + 0.15 * float(m['precision']) + 0.15 * float(m['recall']) + 0.10 * spd_sc + 0.05 * rob)
    return {
        'tag': tag,
        'enable_lut': enable_lut,
        'enable_bright_clahe': enable_bright_clahe,
        'enable_flip_tta': enable_flip_tta,
        'code': code_str,
        'val_pred_path': v_pred,
        'val_speed': spd,
        'val_score': scr,
        'stress_score': st_scr,
        'final_score_proxy_100': final_sc,
        'robustness_mAP50_synth': rob,
        'v_spd_path': v_spd,
        'v_scr_path': v_scr,
        's_scr_path': s_scr,
    }

res_a = _eval_dfine_variant('A_sky_raw', False, False, False)
res_b = _eval_dfine_variant('B_sky_lut', True, False, False)
res_c = _eval_dfine_variant('C_sky_lut_clahe', True, True, False)
best_pre = max([res_a, res_b, res_c], key=lambda r: r['final_score_proxy_100'])
candidates = [res_a, res_b, res_c]
if bool(EXP_CFG.get('test_flip_tta', True)):
    res_d = _eval_dfine_variant('D_sky_bestpre_flip_tta_wbf', best_pre['enable_lut'], best_pre['enable_bright_clahe'], True)
    candidates.append(res_d)

winner = max(candidates, key=lambda r: r['final_score_proxy_100'])
ab_delta = float(winner['final_score_proxy_100'] - res_a['final_score_proxy_100'])

(REPO_DIR / 'inference.py').write_text(winner['code'], encoding='utf-8')
shutil.copy2(REPO_DIR / 'inference.py', RUN_DRIVE_DIR / 'inference.py')
shutil.copy2(winner['v_spd_path'], RUN_DRIVE_DIR / 'val_speed.json')
shutil.copy2(winner['v_scr_path'], RUN_DRIVE_DIR / 'val_score.json')
shutil.copy2(winner['s_scr_path'], RUN_DRIVE_DIR / 'stress_score.json')
shutil.copy2(winner['val_pred_path'], RUN_DRIVE_DIR / 'val_pred_dfine_m.csv')

ab_summary = {
    c['tag']: {
        'accuracy_score_100': c['val_score']['combined']['accuracy_score'],
        'mAP50': c['val_score']['main']['mAP50'],
        'mAP50_95': c['val_score']['main']['mAP50_95'],
        'robustness_mAP50': c['robustness_mAP50_synth'],
        'median_ms': c['val_speed']['median_ms'],
        'final_score_proxy_100': round(c['final_score_proxy_100'], 4),
    }
    for c in candidates
}
ab_summary['winner_tag'] = winner['tag']
ab_summary['ab_winner_delta_final_pts'] = round(ab_delta, 4)
(RUN_DRIVE_DIR / 'ab_inference_comparison.json').write_text(json.dumps(ab_summary, indent=2), encoding='utf-8')
print('4-Way D-FINE-M Inference A/B Comparison:', json.dumps(ab_summary, indent=2))

m = winner['val_score']['main']
pc = m['per_class']
telemetry = autopilot.collect_colab_telemetry(time.monotonic() - STARTED, burn_rate_cu_per_hour=CONFIG.get('budget', {}).get('burn_rate_cu_per_hour'))
metrics.update({
    'accuracy_score_100': float(winner['val_score']['combined']['accuracy_score']),
    'mAP50': float(m['mAP50']),
    'mAP50_95': float(m['mAP50_95']),
    'precision': float(m['precision']),
    'recall': float(m['recall']),
    'ap50_D00': float(pc['D00_longitudinal_crack']['AP50']),
    'ap50_D10': float(pc['D10_transverse_crack']['AP50']),
    'ap50_D20': float(pc['D20_alligator_crack']['AP50']),
    'ap50_D40': float(pc['D40_pothole']['AP50']),
    'speed_score_t4': float(winner['val_speed']['speed_score']),
    'median_ms_t4': float(winner['val_speed']['median_ms']),
    'robustness_mAP50_synth': float(winner['robustness_mAP50_synth']),
    'final_score_proxy_100': float(winner['final_score_proxy_100']),
    'ab_winner_delta_final_pts': float(ab_delta),
    'train_images_count': int(len(train_records)),
    'val_images_count': int(len(val_records)),
    'effective_batch_size': int(BATCH_SIZE * GRAD_ACCUM),
})
if 'peak_vram_gib' in telemetry:
    metrics['peak_vram_gib'] = float(telemetry['peak_vram_gib'])
if 'estimated_compute_units' in telemetry:
    metrics['estimated_compute_units'] = float(telemetry['estimated_compute_units'])

TASK_ARTIFACTS.update({
    'val_score_json': f'03_results/runs/{RUN_ID}/val_score.json',
    'val_speed_json': f'03_results/runs/{RUN_ID}/val_speed.json',
    'stress_score_json': f'03_results/runs/{RUN_ID}/stress_score.json',
    'val_pred_dfine_m_csv': f'03_results/runs/{RUN_ID}/val_pred_dfine_m.csv',
    'ab_inference_comparison': f'03_results/runs/{RUN_ID}/ab_inference_comparison.json',
    'inference_py': f'03_results/runs/{RUN_ID}/inference.py',
    'winner_inference_variant': winner['tag'],
})
check_budget()
```


In [ ]:
_execute_stage(4, 'luma_thresh = float(EXP_CFG.get(\'lowlight_luma_thresh\', 60.0))\ntarget_luma = float(EXP_CFG.get(\'lowlight_target_luma\', 115.0))\nbright_clahe_thresh = float(EXP_CFG.get(\'bright_clahe_luma_thresh\', 155.0))\nwbf_iou_thr = float(EXP_CFG.get(\'wbf_iou_thr\', 0.55))\n\ndef _make_dfine_inference_py(enable_lut: bool, enable_bright_clahe: bool, enable_flip_tta: bool) -> str:\n    return f\'\'\'import math\nimport os\nimport cv2\nimport numpy as np\nimport torch\nimport transformers\nfrom transformers import AutoImageProcessor, DFineForObjectDetection\n\n_TF_MAJOR = int(transformers.__version__.split(".")[0])\n_TF_DTYPE_KW = {{"dtype": torch.float32}} if _TF_MAJOR >= 5 else {{"torch_dtype": torch.float32}}\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nWEIGHTS_DIR = os.path.join(_HERE, "weights", "dfine_m_best")\nIMGSZ = {IMGSZ}\nSKY_CROP_FRAC = {SKY_CROP_FRAC}\nCONF_FLOOR = {conf_floor}\nMAX_DET = {max_det}\nENABLE_LUT = {enable_lut!r}\nENABLE_BRIGHT_CLAHE = {enable_bright_clahe!r}\nENABLE_FLIP_TTA = {enable_flip_tta!r}\nLUMA_THRESH = {luma_thresh}\nTARGET_LUMA = {target_luma}\nBRIGHT_CLAHE_THRESH = {bright_clahe_thresh}\nWBF_IOU_THR = {wbf_iou_thr}\n\ndef _build_lut():\n    x = np.arange(256, dtype=np.float32) / 255.0\n    tgt = math.log(TARGET_LUMA / 255.0)\n    luts = {{}}\n    for lum in range(int(LUMA_THRESH)):\n        gamma = max(0.35, min(0.85, tgt / math.log(max(float(lum), 8.0) / 255.0)))\n        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)\n    return luts\n\ndef _apply_clahe_rgb(img_rgb):\n    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)\n    l_ch, a_ch, b_ch = cv2.split(lab)\n    l_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l_ch)\n    return cv2.cvtColor(cv2.merge((l_eq, a_ch, b_ch)), cv2.COLOR_LAB2RGB)\n\ndef _wbf_two_views(ba, sa, la, bb, sb, lb, iou_thr, max_det):\n    out = []\n    classes = sorted(set([int(c) for c in la] + [int(c) for c in lb]))\n    for cid in classes:\n        cands = []\n        for b, s, c in zip(ba, sa, la):\n            if int(c) == cid:\n                cands.append((float(s), np.asarray(b, dtype=np.float64)))\n        for b, s, c in zip(bb, sb, lb):\n            if int(c) == cid:\n                cands.append((float(s), np.asarray(b, dtype=np.float64)))\n        cands.sort(key=lambda t: t[0], reverse=True)\n        clusters = []\n        for sc, box in cands:\n            best_i, best_iou = -1, iou_thr\n            bx1, by1, bx2, by2 = box\n            barea = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)\n            for ci, cl in enumerate(clusters):\n                cx1, cy1, cx2, cy2 = cl["fused"]\n                iw = max(0.0, min(bx2, cx2) - max(bx1, cx1))\n                ih = max(0.0, min(by2, cy2) - max(by1, cy1))\n                inter = iw * ih\n                if inter <= 0.0:\n                    continue\n                carea = max(0.0, cx2 - cx1) * max(0.0, cy2 - cy1)\n                iou = inter / max(1e-9, barea + carea - inter)\n                if iou >= best_iou:\n                    best_iou, best_i = iou, ci\n            if best_i >= 0:\n                cl = clusters[best_i]\n                ws = cl["w_sum"] + sc\n                cl["fused"] = (cl["fused"] * cl["w_sum"] + box * sc) / max(1e-9, ws)\n                cl["w_sum"] = ws\n                cl["scores"].append(sc)\n            else:\n                clusters.append({{"fused": box.copy(), "w_sum": sc, "scores": [sc]}})\n        for cl in clusters:\n            fb = cl["fused"]\n            out.append((cid, float(sum(cl["scores"]) / 2.0), float(fb[0]), float(fb[1]), float(fb[2]), float(fb[3])))\n    out.sort(key=lambda t: t[1], reverse=True)\n    return out[:max_det]\n\nclass _DFineHolder:\n    def __init__(self, model, processor, device, amp_dtype, luts):\n        self.model = model\n        self.processor = processor\n        self.device = device\n        self.amp_dtype = amp_dtype\n        self.luts = luts\n\ndef load(device="cuda"):\n    dev = torch.device(device if (torch.cuda.is_available() and str(device).startswith("cuda")) else "cpu")\n    proc = AutoImageProcessor.from_pretrained(WEIGHTS_DIR, local_files_only=True)\n    net = DFineForObjectDetection.from_pretrained(WEIGHTS_DIR, local_files_only=True, **_TF_DTYPE_KW).to(dev).eval()\n    amp_dt = torch.bfloat16 if (dev.type == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if dev.type == "cuda" else None)\n    luts = _build_lut() if ENABLE_LUT else None\n    return _DFineHolder(net, proc, dev, amp_dt, luts)\n\n@torch.no_grad()\ndef predict(model, image):\n    orig_h, orig_w = int(image.shape[0]), int(image.shape[1])\n    y_off = max(0, min(orig_h - 32, int(round(orig_h * SKY_CROP_FRAC))))\n    crop = image[y_off:, :, :]\n    if model.luts is not None or ENABLE_BRIGHT_CLAHE:\n        mean_lum = float(crop[::4, ::4].mean())\n        if model.luts is not None and mean_lum < LUMA_THRESH:\n            crop = model.luts[max(0, min(int(LUMA_THRESH) - 1, int(mean_lum)))][crop]\n        elif ENABLE_BRIGHT_CLAHE and mean_lum > BRIGHT_CLAHE_THRESH:\n            crop = _apply_clahe_rgb(crop)\n\n    crop_h, crop_w = int(crop.shape[0]), int(crop.shape[1])\n    rz = cv2.resize(crop, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)\n    t1 = torch.from_numpy(np.ascontiguousarray(rz)).permute(2, 0, 1).to(torch.float32).div_(255.0)\n    if ENABLE_FLIP_TTA:\n        t2 = torch.flip(t1, dims=[2])\n        px = torch.stack([t1, t2], dim=0).to(model.device)\n        t_sizes = torch.tensor([[crop_h, crop_w], [crop_h, crop_w]], dtype=torch.long)\n    else:\n        px = t1.unsqueeze(0).to(model.device)\n        t_sizes = torch.tensor([[crop_h, crop_w]], dtype=torch.long)\n\n    with torch.amp.autocast("cuda", dtype=model.amp_dtype, enabled=(model.device.type == "cuda" and model.amp_dtype is not None)):\n        outs = model.model(pixel_values=px)\n    outs.logits = outs.logits.float()\n    outs.pred_boxes = outs.pred_boxes.float()\n    decoded = model.processor.post_process_object_detection(outs, target_sizes=t_sizes, threshold=CONF_FLOOR)\n\n    r0 = decoded[0]\n    b0 = r0["boxes"].detach().float().cpu().numpy()\n    s0 = r0["scores"].detach().float().cpu().numpy()\n    l0 = r0["labels"].detach().cpu().numpy().astype(int)\n\n    if ENABLE_FLIP_TTA and len(decoded) > 1:\n        r1 = decoded[1]\n        b1 = r1["boxes"].detach().float().cpu().numpy()\n        s1 = r1["scores"].detach().float().cpu().numpy()\n        l1 = r1["labels"].detach().cpu().numpy().astype(int)\n        if len(b1) > 0:\n            b1_u = b1.copy()\n            b1_u[:, 0] = float(crop_w) - b1[:, 2]\n            b1_u[:, 2] = float(crop_w) - b1[:, 0]\n        else:\n            b1_u = b1\n        preds = _wbf_two_views(b0, s0, l0, b1_u, s1, l1, WBF_IOU_THR, MAX_DET)\n        out_list = []\n        for cid, sc, x1, y1, x2, y2 in preds:\n            cid_i = int(cid)\n            sc_f = max(0.0, min(1.0, float(sc)))\n            fx1 = max(0.0, min(float(orig_w), float(x1)))\n            fy1 = max(0.0, min(float(orig_h), float(y1) + float(y_off)))\n            fx2 = max(0.0, min(float(orig_w), float(x2)))\n            fy2 = max(0.0, min(float(orig_h), float(y2) + float(y_off)))\n            if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (fx2 - fx1) >= 0.5 and (fy2 - fy1) >= 0.5:\n                out_list.append((cid_i, sc_f, fx1, fy1, fx2, fy2))\n        return out_list\n\n    if len(s0) > MAX_DET:\n        keep = np.argsort(-s0)[:MAX_DET]\n        s0, l0, b0 = s0[keep], l0[keep], b0[keep]\n    out_list = []\n    for cid, sc, bx in zip(l0, s0, b0):\n        cid_i = int(cid)\n        sc_f = max(0.0, min(1.0, float(sc)))\n        fx1 = max(0.0, min(float(orig_w), float(bx[0])))\n        fy1 = max(0.0, min(float(orig_h), float(bx[1]) + float(y_off)))\n        fx2 = max(0.0, min(float(orig_w), float(bx[2])))\n        fy2 = max(0.0, min(float(orig_h), float(bx[3]) + float(y_off)))\n        if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (fx2 - fx1) >= 0.5 and (fy2 - fy1) >= 0.5:\n            out_list.append((cid_i, sc_f, fx1, fy1, fx2, fy2))\n    return out_list\n\'\'\'\n\ndef _eval_dfine_variant(tag: str, enable_lut: bool, enable_bright_clahe: bool, enable_flip_tta: bool) -> dict:\n    code_str = _make_dfine_inference_py(enable_lut, enable_bright_clahe, enable_flip_tta)\n    (REPO_DIR / \'inference.py\').write_text(code_str, encoding=\'utf-8\')\n    v_pred = WORK_ROOT / f\'{tag}_val_pred.csv\'\n    v_spd = WORK_ROOT / f\'{tag}_val_speed.json\'\n    v_scr = WORK_ROOT / f\'{tag}_val_score.json\'\n    s_pred = WORK_ROOT / f\'{tag}_stress_pred.csv\'\n    s_spd = WORK_ROOT / f\'{tag}_stress_speed.json\'\n    s_scr = WORK_ROOT / f\'{tag}_stress_score.json\'\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'benchmark_speed.py\'), \'--repo\', str(REPO_DIR), \'--images\', str(KIT_LOCAL / \'val_images\'), \'--out-csv\', str(v_pred), \'--out-json\', str(v_spd), \'--runs\', \'2\', \'--repeat-images\', \'300\'], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'score.py\'), \'--pred\', str(v_pred), \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'), \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'), \'--speed-json\', str(v_spd), \'--json-out\', str(v_scr)], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'benchmark_speed.py\'), \'--repo\', str(REPO_DIR), \'--images\', str(VAL_DEGRADED_DIR), \'--out-csv\', str(s_pred), \'--out-json\', str(s_spd), \'--runs\', \'1\', \'--warmup\', \'5\'], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'score.py\'), \'--pred\', str(s_pred), \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'), \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'), \'--json-out\', str(s_scr)], check=True)\n    spd = json.loads(v_spd.read_text(encoding=\'utf-8\'))\n    scr = json.loads(v_scr.read_text(encoding=\'utf-8\'))\n    st_scr = json.loads(s_scr.read_text(encoding=\'utf-8\'))\n    m = scr[\'main\']\n    rob = float(st_scr[\'main\'][\'mAP50\'])\n    spd_sc = float(spd[\'speed_score\'])\n    final_sc = 100.0 * (0.30 * float(m[\'mAP50\']) + 0.25 * float(m[\'mAP50_95\']) + 0.15 * float(m[\'precision\']) + 0.15 * float(m[\'recall\']) + 0.10 * spd_sc + 0.05 * rob)\n    return {\n        \'tag\': tag,\n        \'enable_lut\': enable_lut,\n        \'enable_bright_clahe\': enable_bright_clahe,\n        \'enable_flip_tta\': enable_flip_tta,\n        \'code\': code_str,\n        \'val_pred_path\': v_pred,\n        \'val_speed\': spd,\n        \'val_score\': scr,\n        \'stress_score\': st_scr,\n        \'final_score_proxy_100\': final_sc,\n        \'robustness_mAP50_synth\': rob,\n        \'v_spd_path\': v_spd,\n        \'v_scr_path\': v_scr,\n        \'s_scr_path\': s_scr,\n    }\n\nres_a = _eval_dfine_variant(\'A_sky_raw\', False, False, False)\nres_b = _eval_dfine_variant(\'B_sky_lut\', True, False, False)\nres_c = _eval_dfine_variant(\'C_sky_lut_clahe\', True, True, False)\nbest_pre = max([res_a, res_b, res_c], key=lambda r: r[\'final_score_proxy_100\'])\ncandidates = [res_a, res_b, res_c]\nif bool(EXP_CFG.get(\'test_flip_tta\', True)):\n    res_d = _eval_dfine_variant(\'D_sky_bestpre_flip_tta_wbf\', best_pre[\'enable_lut\'], best_pre[\'enable_bright_clahe\'], True)\n    candidates.append(res_d)\n\nwinner = max(candidates, key=lambda r: r[\'final_score_proxy_100\'])\nab_delta = float(winner[\'final_score_proxy_100\'] - res_a[\'final_score_proxy_100\'])\n\n(REPO_DIR / \'inference.py\').write_text(winner[\'code\'], encoding=\'utf-8\')\nshutil.copy2(REPO_DIR / \'inference.py\', RUN_DRIVE_DIR / \'inference.py\')\nshutil.copy2(winner[\'v_spd_path\'], RUN_DRIVE_DIR / \'val_speed.json\')\nshutil.copy2(winner[\'v_scr_path\'], RUN_DRIVE_DIR / \'val_score.json\')\nshutil.copy2(winner[\'s_scr_path\'], RUN_DRIVE_DIR / \'stress_score.json\')\nshutil.copy2(winner[\'val_pred_path\'], RUN_DRIVE_DIR / \'val_pred_dfine_m.csv\')\n\nab_summary = {\n    c[\'tag\']: {\n        \'accuracy_score_100\': c[\'val_score\'][\'combined\'][\'accuracy_score\'],\n        \'mAP50\': c[\'val_score\'][\'main\'][\'mAP50\'],\n        \'mAP50_95\': c[\'val_score\'][\'main\'][\'mAP50_95\'],\n        \'robustness_mAP50\': c[\'robustness_mAP50_synth\'],\n        \'median_ms\': c[\'val_speed\'][\'median_ms\'],\n        \'final_score_proxy_100\': round(c[\'final_score_proxy_100\'], 4),\n    }\n    for c in candidates\n}\nab_summary[\'winner_tag\'] = winner[\'tag\']\nab_summary[\'ab_winner_delta_final_pts\'] = round(ab_delta, 4)\n(RUN_DRIVE_DIR / \'ab_inference_comparison.json\').write_text(json.dumps(ab_summary, indent=2), encoding=\'utf-8\')\nprint(\'4-Way D-FINE-M Inference A/B Comparison:\', json.dumps(ab_summary, indent=2))\n\nm = winner[\'val_score\'][\'main\']\npc = m[\'per_class\']\ntelemetry = autopilot.collect_colab_telemetry(time.monotonic() - STARTED, burn_rate_cu_per_hour=CONFIG.get(\'budget\', {}).get(\'burn_rate_cu_per_hour\'))\nmetrics.update({\n    \'accuracy_score_100\': float(winner[\'val_score\'][\'combined\'][\'accuracy_score\']),\n    \'mAP50\': float(m[\'mAP50\']),\n    \'mAP50_95\': float(m[\'mAP50_95\']),\n    \'precision\': float(m[\'precision\']),\n    \'recall\': float(m[\'recall\']),\n    \'ap50_D00\': float(pc[\'D00_longitudinal_crack\'][\'AP50\']),\n    \'ap50_D10\': float(pc[\'D10_transverse_crack\'][\'AP50\']),\n    \'ap50_D20\': float(pc[\'D20_alligator_crack\'][\'AP50\']),\n    \'ap50_D40\': float(pc[\'D40_pothole\'][\'AP50\']),\n    \'speed_score_t4\': float(winner[\'val_speed\'][\'speed_score\']),\n    \'median_ms_t4\': float(winner[\'val_speed\'][\'median_ms\']),\n    \'robustness_mAP50_synth\': float(winner[\'robustness_mAP50_synth\']),\n    \'final_score_proxy_100\': float(winner[\'final_score_proxy_100\']),\n    \'ab_winner_delta_final_pts\': float(ab_delta),\n    \'train_images_count\': int(len(train_records)),\n    \'val_images_count\': int(len(val_records)),\n    \'effective_batch_size\': int(BATCH_SIZE * GRAD_ACCUM),\n})\nif \'peak_vram_gib\' in telemetry:\n    metrics[\'peak_vram_gib\'] = float(telemetry[\'peak_vram_gib\'])\nif \'estimated_compute_units\' in telemetry:\n    metrics[\'estimated_compute_units\'] = float(telemetry[\'estimated_compute_units\'])\n\nTASK_ARTIFACTS.update({\n    \'val_score_json\': f\'03_results/runs/{RUN_ID}/val_score.json\',\n    \'val_speed_json\': f\'03_results/runs/{RUN_ID}/val_speed.json\',\n    \'stress_score_json\': f\'03_results/runs/{RUN_ID}/stress_score.json\',\n    \'val_pred_dfine_m_csv\': f\'03_results/runs/{RUN_ID}/val_pred_dfine_m.csv\',\n    \'ab_inference_comparison\': f\'03_results/runs/{RUN_ID}/ab_inference_comparison.json\',\n    \'inference_py\': f\'03_results/runs/{RUN_ID}/inference.py\',\n    \'winner_inference_variant\': winner[\'tag\'],\n})\ncheck_budget()')


## 6. Full 4,200 Test Inference, Ensemble CSV Export & Submission Validation
Run winning inference.py over all 4,200 test images, export ensemble-ready test_pred_dfine_m.csv, generate and validate submission CSV with check_submission.py.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
expected_test_ids = len(test_ids_df)
actual_test_imgs = len(list((KIT_LOCAL / 'test_images').glob('*.jpg')))
if actual_test_imgs < expected_test_ids:
    raise RuntimeError(f'Expected {expected_test_ids} test images, got {actual_test_imgs}')

test_pred_csv = WORK_ROOT / 'test_pred_dfine_m.csv'
test_speed_json = WORK_ROOT / 'test_speed.json'
sub_csv = WORK_ROOT / f'submission_{RUN_ID}.csv'

subprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(KIT_LOCAL / 'test_images'), '--out-csv', str(test_pred_csv), '--out-json', str(test_speed_json), '--runs', '1', '--warmup', '10'], check=True)
subprocess.run([sys.executable, str(KIT_LOCAL / 'make_kaggle_submission.py'), '--pred', str(test_pred_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv'), '--out', str(sub_csv)], check=True)
subprocess.run([sys.executable, str(KIT_LOCAL / 'check_submission.py'), '--pred', str(sub_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv')], check=True)

drive_sub_dir = PROJECT_ROOT / '04_submissions'
drive_sub_dir.mkdir(parents=True, exist_ok=True)
drive_sub = drive_sub_dir / f'submission_{RUN_ID}.csv'
shutil.copy2(sub_csv, drive_sub)
shutil.copy2(test_pred_csv, RUN_DRIVE_DIR / 'test_pred_dfine_m.csv')
shutil.copy2(test_speed_json, RUN_DRIVE_DIR / 'test_speed.json')

TASK_ARTIFACTS['test_pred_dfine_m_csv'] = f'03_results/runs/{RUN_ID}/test_pred_dfine_m.csv'
TASK_ARTIFACTS['test_speed_json'] = f'03_results/runs/{RUN_ID}/test_speed.json'
TASK_ARTIFACTS['submission_csv'] = f'04_submissions/submission_{RUN_ID}.csv'
TASK_ARTIFACTS['submission_sha256'] = autopilot.file_digest(drive_sub)
TASK_ARTIFACTS['submission_status'] = 'validated_by_check_submission_py'
print('Validated D-FINE-M submission saved to:', drive_sub)
print('Ensemble-ready predictions saved to:', RUN_DRIVE_DIR / 'val_pred_dfine_m.csv', 'and', RUN_DRIVE_DIR / 'test_pred_dfine_m.csv')
```


In [ ]:
_execute_stage(5, "expected_test_ids = len(test_ids_df)\nactual_test_imgs = len(list((KIT_LOCAL / 'test_images').glob('*.jpg')))\nif actual_test_imgs < expected_test_ids:\n    raise RuntimeError(f'Expected {expected_test_ids} test images, got {actual_test_imgs}')\n\ntest_pred_csv = WORK_ROOT / 'test_pred_dfine_m.csv'\ntest_speed_json = WORK_ROOT / 'test_speed.json'\nsub_csv = WORK_ROOT / f'submission_{RUN_ID}.csv'\n\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(KIT_LOCAL / 'test_images'), '--out-csv', str(test_pred_csv), '--out-json', str(test_speed_json), '--runs', '1', '--warmup', '10'], check=True)\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'make_kaggle_submission.py'), '--pred', str(test_pred_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv'), '--out', str(sub_csv)], check=True)\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'check_submission.py'), '--pred', str(sub_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv')], check=True)\n\ndrive_sub_dir = PROJECT_ROOT / '04_submissions'\ndrive_sub_dir.mkdir(parents=True, exist_ok=True)\ndrive_sub = drive_sub_dir / f'submission_{RUN_ID}.csv'\nshutil.copy2(sub_csv, drive_sub)\nshutil.copy2(test_pred_csv, RUN_DRIVE_DIR / 'test_pred_dfine_m.csv')\nshutil.copy2(test_speed_json, RUN_DRIVE_DIR / 'test_speed.json')\n\nTASK_ARTIFACTS['test_pred_dfine_m_csv'] = f'03_results/runs/{RUN_ID}/test_pred_dfine_m.csv'\nTASK_ARTIFACTS['test_speed_json'] = f'03_results/runs/{RUN_ID}/test_speed.json'\nTASK_ARTIFACTS['submission_csv'] = f'04_submissions/submission_{RUN_ID}.csv'\nTASK_ARTIFACTS['submission_sha256'] = autopilot.file_digest(drive_sub)\nTASK_ARTIFACTS['submission_status'] = 'validated_by_check_submission_py'\nprint('Validated D-FINE-M submission saved to:', drive_sub)\nprint('Ensemble-ready predictions saved to:', RUN_DRIVE_DIR / 'val_pred_dfine_m.csv', 'and', RUN_DRIVE_DIR / 'test_pred_dfine_m.csv')")


## Verify and persist results
This cell records completion only after every section succeeds. If delivery fails, recover the spool rather than repeat the experiment.


In [ ]:
if _STAGE_STATE['failed'] or _STAGE_STATE['finalized'] or _STAGE_STATE['next'] != 6:
    raise RuntimeError('All task sections must succeed before finalization')
try:
    autopilot.validate_metrics(metrics)
    elapsed_seconds = time.monotonic() - STARTED
    limit = CONFIG.get('budget', {}).get('max_hours')
    if limit is not None and elapsed_seconds > limit * 3600:
        raise RuntimeError('Observed runtime exceeded configured limit')
    log_path = autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), metrics, run_id=RUN_ID, spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}, context=CONFIG.get('context', {}), metric_context=CONFIG.get('metric_context', {}), artifacts={**TASK_ARTIFACTS, 'elapsed_seconds': elapsed_seconds})
    print('Completed run:', RUN_ID, 'Log:', log_path)
except autopilot.LogDeliveryError as error:
    _STAGE_STATE['failed'] = True
    print('Result delivery is pending. Preserve the local spool and use RECOVER_RUN_ID:', RUN_ID)
    raise
except BaseException as error:
    _STAGE_STATE['failed'] = True
    status = 'interrupted' if isinstance(error, KeyboardInterrupt) else 'failed'
    try:
        autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), {}, run_id=RUN_ID, event_id='failure-' + str(uuid.uuid4()), status=status, spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}, errors=[{'type': type(error).__name__}])
    except Exception:
        print('Could not persist failure event; inspect runtime storage before disconnecting.')
    raise
_STAGE_STATE['finalized'] = True
